# Data Crunch Notebook
Goals of this notebook:
- Sandbox to discover your study case country's profile
- Discover key indicators that will help shape the system
- Discover new concepts & methods: load duration curves, controllable/fatal generation, baseload/peakload, Raw/Net load, Boiteux stacking
- Understand main power generation technologies' characteristics (technical, environmental & economic)
- Simulate various power mix

Expected Output:
- Choice of power mix for your study case country -> Input for PyPSA and the rest of the week

# 1 - Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
import plotly.express as px
import json
from itertools import combinations



# 2 - Choice of your study case
Choose the country (or group zone) that was assigned to your group

*Choose between (**in lowercase**): benelux, france, germany, iberian-peninsula, italy, poland, scandinavia*

In [ ]:
country_choice = "iberian-peninsula"

#List of years considered for this practical work
ERAA_years = [2025,2033]



# 3 - Power Demand 

## 3.1 - Download of gross demand data

In [ ]:
demands = {}
for year in ERAA_years:
    df = pd.read_csv(f"data/ERAA_2023-2/demand/demand_{year}_{country_choice}.csv", sep=';', header=0)
    pivoted = df.pivot(index='date', columns='climatic_year', values='value')
    demands[year] = pivoted

Renaming the climatic years for clarity

In [ ]:
# List of all unique climatic years considered for this practical work
for year in ERAA_years:
    ERAA_climatic_years = list(demands[year].columns)

    # Renaming them WS1, WS2, ... for clarity
    for cy in (ERAA_climatic_years):
        demands[year].rename(columns={cy: f'WS{cy}'}, inplace=True)

climatic_year_ws = [f'WS{cy}' for cy in ERAA_climatic_years]

Creating an average weather scenario


***WARNING***: the weather scenarios are not equally probable! The "average whether scenario" is therefore not really the average observed climate

In [ ]:
for year in ERAA_years:
    demands[year]['Avg_WS'] = demands[year].iloc[:, 1:].mean(axis=1)

## 3.2 - Data Analysis (Average weather scenario)

***WARNING***: the weather scenarios are not equally probable! The "average whether scenario" is therefore not really the average observed climate

### 3.2.1 - Raw demand plots  
*(Run this command in terminal if needed : pip install --upgrade nbformat)*

In [ ]:
fig = go.Figure()

for year in ERAA_years:
    df = demands[year]

    # Build a plotting x-axis where the year is replaced with a common dummy year (e.g., 2000)
    idx = df.index
    #x_superposed = [ts.replace(year=2018) for ts in idx]

    fig.add_trace(
        go.Scatter(
            x=idx,
            y=df["Avg_WS"],
            mode="lines",
            name=f"Year {year}"
        )
    )

fig.update_layout(
    title="Hourly Electricity Demand (Average Weather Scenario, Superposed Across Years)",
    xaxis_title="Date (Year removed)",
    yaxis_title="Demand (MW)",
    hovermode="x unified",
    template="plotly_white",
    legend_title="Year",
    width=1100,
    height=600
)

# Set labels
fig.update_xaxes(
    dtick=None,
    tickformat="%d %b\n%H:%M"
)

fig.show()

Questions:
- What main patterns can you observe?
- How could you explain these?
- Are there any evolutions between 2025 and 2033?
- How could you explain these?
- What indicators could you use to quantify and compare your country's demand profile with your neighbors?
- How could these patterns impact the power mix design?

### 3.2.2 - Usefull indicators for profile analysis

Let's compute basic statistics from load profile:
- Annual Gross Demand
- Peak Load
- Seasonal Variation (difference between months with resp. highest and lowest average demand)
- Daily Variation (maximal load spread in a single day)

Computing the indicators

In [ ]:
results = []

for year in ERAA_years:
    df = demands[year].copy()

    # Ensure index is datetime
    df.index = pd.to_datetime(df.index)

    # 1) Total annual demand (MWh)
    # Avg_WS is in MW sampled hourly ⇒ sum gives MWh
    total_demand_mwh = df["Avg_WS"].sum()

    # 2) Peak demand (MW)
    peak_load = df["Avg_WS"].max()

    # 3) Seasonal variation:
    #   Compute average demand per month
    monthly_avg = df["Avg_WS"].resample("ME").mean()
    seasonal_variation = monthly_avg.max() - monthly_avg.min()

    # 4) Daily variation:
    #   Compute daily max-min spread
    daily_spread = df["Avg_WS"].resample("D").apply(lambda x: x.max() - x.min())
    daily_variation = daily_spread.max()

    results.append({
        "Year": year,
        "Total_Demand_TWh": total_demand_mwh,
        "Peak_Load_GW": peak_load,
        "Seasonal_Variation_GW": seasonal_variation,
        "Max_Daily_Variation_GW": daily_variation,
    })

results_df = pd.DataFrame(results).set_index("Year")

Plotting the indicators

In [ ]:
fig = make_subplots(
rows=2, cols=2,
subplot_titles=[
"Total Annual Demand (TWh)",
"Peak Demand (GW)",
"Seasonal Variation (GW)",
"Max Daily Variation (GW)"
]
)

fig.add_trace(
go.Bar(
x=results_df.index,
y=results_df["Total_Demand_TWh"]/1e6,
text=(results_df["Total_Demand_TWh"]/1e6).round(0),
textposition="auto",
name="Total Demand (TWh)"
),
row=1, col=1
)

fig.add_trace(
go.Bar(
x=results_df.index,
y=results_df["Peak_Load_GW"]/1e3,
text=(results_df["Peak_Load_GW"]/1e3).round(0),
textposition="auto",
name="Peak Load (GW)"
),
row=1, col=2
)

fig.add_trace(
go.Bar(
x=results_df.index,
y=results_df["Seasonal_Variation_GW"]/1e3,
text=(results_df["Seasonal_Variation_GW"]/1e3).round(1),
textposition="auto",
name="Seasonal Variation (GW)"
),
row=2, col=1
)

fig.add_trace(
go.Bar(
x=results_df.index,
y=results_df["Max_Daily_Variation_GW"]/1e3,
text=(results_df["Max_Daily_Variation_GW"]/1e3).round(1),
textposition="auto",
name="Max Daily Variation (GW)"
),
row=2, col=2
)

fig.update_layout(
title=f"Electricity Demand in {country_choice.capitalize()} by Year (Avg. Weather Scenario)",
template="plotly_white",
height=800,
width=1100,
showlegend=False
)

fig.update_xaxes(title_text="Year")
#fig.update_yaxes(title_text="MW / MWh")

fig.show()

*Optional: what other relevant indicators could you compute?*

#### Optional indicators

Here are a few suggestions, you can add your own ideas below if you want!
- Distribution of daily variation
- Distribuiton of weekly variation
- *Frequency analysis -> (ask if you want to know more about this)*

In [ ]:
# Compute daily variation
daily_variation = df["Avg_WS"].resample("D").apply(lambda x: x.max() - x.min())

# Compute weekly variation: mean of weekends vs mean of weekdays for each week
df["weekday"] = df.index.weekday  # Monday=0, Sunday=6

# For each week, compute mean on weekends, mean on weekdays
def weekly_variation(group):
    weekend = group[group["weekday"] >= 5]["Avg_WS"]   # Sat (5) + Sun (6)
    weekday = group[group["weekday"] < 5]["Avg_WS"]    # Mon-Fri (0-4)
    if len(weekend) > 0:
        mean_weekend = weekend.mean()
    else:
        mean_weekend = None
    if len(weekday) > 0:
        mean_weekday = weekday.mean()
    else:
        mean_weekday = None
    return pd.Series({"mean_weekend": mean_weekend, "mean_weekday": mean_weekday})

weekly_means = df.groupby(pd.Grouper(freq='W')).apply(weekly_variation).dropna()

# Weekly variation is the absolute difference
weekly_variation_values = (weekly_means["mean_weekend"] - weekly_means["mean_weekday"]).abs().dropna()

# Create boxplots
fig = go.Figure()

fig.add_trace(go.Box(
    y=daily_variation,
    name="Daily Variation",
    boxmean='sd',
    marker_color='royalblue'
))

fig.add_trace(go.Box(
    y=weekly_variation_values,
    name="Weekly Variation\n(|mean weekend - mean weekday|)",
    boxmean='sd',
    marker_color='darkorange'
))

fig.update_layout(
    title="Distribution of Daily and Weekly Demand Variation",
    yaxis_title="Demand Variation (MW)",
    template="plotly_white"
)

fig.show()




Questions:
- Why are these indicators relevant?
- What does an increased variation implicate in your opinion?

### 3.2.3 - Load Duration Curve

The load duration curve is a commonly used representation of the distribution of power demand across one or multiple climatic years or periods.


Typically, it is often used to plot the loads across one year. It helps identify the peak load and baseload.

Computation of load duration curves


In [ ]:
ldc = {}

for year in ERAA_years:
    df = demands[year].copy()
    ldc_df = pd.DataFrame()
    
    for col in df.columns:
        # Sort descending to create load duration curve
        ldc_df[col] = df[col].sort_values(ascending=False).reset_index(drop=True)
    
    ldc[year] = ldc_df

Plot of the load duration curves for the average weather scenario


In [ ]:
fig = go.Figure()

for year in ERAA_years:
    # Add the curve for this year
    fig.add_trace(
        go.Scatter(
            x=ldc[year].index,          # Hour rank
            y=ldc[year]["Avg_WS"]/1e3,      # Sorted load values
            mode="lines",
            name=str(year)
        )
    )

fig.update_layout(
    title=f"Load Duration Curves – {country_choice.capitalize()} - Average Weather Scenario",
    xaxis_title="Hours",
    yaxis_title="Demand (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600
)

fig.show()

Questions:
- Can you interpret this graph?
- Can you spot the peak load?
- Can you spot the baseload?
- Does the shape of these lines change in ERAA projections?
- Could you guess the underlying hypotheses that explain that?

## 3.3 - Data Analysis (incl. climatic years)

***Notebook Break**: In-session presentation of the concept of climatic years, various methods existing, and the one considered for this week*

Questions:
1) Could you summarize the concept of climatic years, and explain breifly why they are mandatory for power mix design?
2) Could you describe the various climatic year considered in this practical session?
3) Which one(s) do you want to consider for further analysis? Why?
4) In your opinion, what are the benefits and drawbacks of your approach?

----------------------------------------------------------------------------------------------------------------
Your answers here:
1) ...
2) ...
3) ...
4) ...

### 3.3.1 - Raw demand plots

You can first browse through the raw data to see manually how the load can evolve based on each climatic year

For better readability, choose a single year to plot here:

In [ ]:
plot_year = 2033

In [ ]:

fig = go.Figure()

for ws in climatic_year_ws:
    fig.add_trace(go.Scatter(
        x=demands[plot_year].index,
        y=demands[plot_year][ws],
        mode='lines',
        name=ws,
        line=dict(width=2)
    ))

fig.update_layout(
    title=f"Hourly Electricity Demand in {country_choice.capitalize()} for {plot_year} (All Climatic Years)",
    xaxis_title="Date",
    yaxis_title="Demand (MW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600
)


### 3.3.2 Usefull indicators for profile analysis

Again, choose a single year to plot here:

In [ ]:
plot_year = 2025

In [ ]:
# Compute statistics on profiles
results_WS = []
df = demands[plot_year].copy()

for ws in climatic_year_ws:
    # Ensure index is datetime
    df.index = pd.to_datetime(df.index)

    # 1) Total annual demand (MWh)
    # Avg_WS is in MW sampled hourly ⇒ sum gives MWh
    total_demand_mwh = df[ws].sum()

    # 2) Peak demand (MW)
    peak_load = df[ws].max()

    # 3) Seasonal variation:
    #   Compute average demand per month
    monthly_avg = df[ws].resample("ME").mean()
    seasonal_variation = monthly_avg.max() - monthly_avg.min()

    # 4) Daily variation:
    #   Compute daily max-min spread
    daily_spread = df[ws].resample("D").apply(lambda x: x.max() - x.min())
    daily_variation = daily_spread.max()

    results_WS.append({
        "WS": ws,
        "Total_Demand_TWh": total_demand_mwh,
        "Peak_Load_GW": peak_load,
        "Seasonal_Variation_GW": seasonal_variation,
        "Max_Daily_Variation_GW": daily_variation,
    })

results_df_WS = pd.DataFrame(results_WS).set_index("WS")

fig = go.Figure()

In [ ]:
# Plot dashboard
fig = make_subplots(
rows=2, cols=2,
subplot_titles=[
"Total Annual Demand (TWh)",
"Peak Demand (GW)",
"Seasonal Variation (GW)",
"Max Daily Variation (GW)"
]
)

fig.add_trace(
go.Bar(
x=results_df_WS.index,
y=results_df_WS["Total_Demand_TWh"]/1e6,
text=(results_df_WS["Total_Demand_TWh"]/1e6).round(0),
textposition="auto",
name="Total Demand (TWh)"
),
row=1, col=1
)

fig.add_trace(
go.Bar(
x=results_df_WS.index,
y=results_df_WS["Peak_Load_GW"]/1e3,
text=(results_df_WS["Peak_Load_GW"]/1e3).round(0),
textposition="auto",
name="Peak Load (GW)"
),
row=1, col=2
)

fig.add_trace(
go.Bar(
x=results_df_WS.index,
y=results_df_WS["Seasonal_Variation_GW"]/1e3,
text=(results_df_WS["Seasonal_Variation_GW"]/1e3).round(1),
textposition="auto",
name="Seasonal Variation (GW)"
),
row=2, col=1
)

fig.add_trace(
go.Bar(
x=results_df_WS.index,
y=results_df_WS["Max_Daily_Variation_GW"]/1e3,
text=(results_df_WS["Max_Daily_Variation_GW"]/1e3).round(1),
textposition="auto",
name="Max Daily Variation (GW)"
),
row=2, col=2
)

fig.update_layout(
title=f"Electricity Demand in {country_choice.capitalize()} for {plot_year} (all climatic years)",
template="plotly_white",
height=800,
width=1100,
showlegend=False
)

fig.update_xaxes(title_text="Weather Scenario")
#fig.update_yaxes(title_text="MW / MWh")

fig.show()

#### Optional indicators

In [ ]:
# Compute daily and weekly variations for all climatic years and plot their distributions as separate box plots (one per climatic year) for the selected year (plot_year)

fig = go.Figure()

# Box plots for daily variations, one per climatic year
for ws in climatic_year_ws:
    df = demands[plot_year].copy()
    df.index = pd.to_datetime(df.index)
    if ws in df.columns:
        daily_var = df[ws].resample("D").apply(lambda x: x.max() - x.min())
        fig.add_trace(
            go.Box(
                y=daily_var.values,
                name=f"Daily: WS{ws}",
                boxmean='sd',
                marker_color='royalblue',
                boxpoints=False  # no individual points
            )
        )

# Box plots for weekly variations, one per climatic year
for ws in climatic_year_ws:
    df = demands[plot_year].copy()
    df.index = pd.to_datetime(df.index)
    if ws in df.columns:
        weekly_var = df[ws].resample("W").apply(lambda x: x.max() - x.min())
        fig.add_trace(
            go.Box(
                y=weekly_var.values,
                name=f"Weekly: {ws}",
                boxmean='sd',
                marker_color='firebrick',
                boxpoints=False  # no individual points
            )
        )

fig.update_layout(
    title=f"Daily and Weekly variations per climatic year in {country_choice.capitalize()} for {plot_year}",
    yaxis_title="Variation (MW)",
    template="plotly_white",
    width=1300,
    height=650,
    boxmode='group'
)

fig.show()

Add here below any additional indicators if you want to

### 3.3.3 Load duration curve

Again, choose a single year to plot here:

In [ ]:
plot_year = 2033

In [ ]:
# Plot LDC for target year and WS list
fig = go.Figure()

for ws in climatic_year_ws:
    fig.add_trace(
        go.Scatter(
            x=ldc[plot_year].index,
            y=ldc[plot_year][ws]/1e3,
            mode="lines",
            name=ws
        )
    )

fig.update_layout(
    title=f"Load duration curves in {country_choice.capitalize()} for {plot_year}",
    xaxis_title="Hours",
    yaxis_title="Demand (GW)",
    template="plotly_white"
)

fig.show()

Questions:
- How if each load duraction curve affected by each scenario?
- Do it echoes the description of the climatic years presented at the beginning of the section?
- Which one to consider for further studies? Why?


## 3.4 Risk management and unsupplied energy

**Notebook Break**: in class discussion on risk management (LOLE), value of loss load (VOLL), cost of risk hedging...

The VOLL considerd by ERAA (in technology.json) is set at **20000€/MWh**

# 4 - Generation

## 4.1 - Notion of controllable and variable generation

**Notebook Break**: in class discussion on the concepts of controllable and variable generation, gross and net load.

*You can write down here what you understood from these notions* + a few questions as **food for thoughts** (no need to answer yet)

Questions:
- What is the difference between gross and net load?
- Can we expect variable generation to participate to net peakload reduction? Why?
- Is variable generation totally random?
- Do you think we could run a power based on 100% renewables? What would it require?

## 4.2 - Optimal power mix with controllable generation

As the power mix was not built including vRES, the founding theories on power mix design are based on a fully controllable generation mix. This is why, in this section, we will only consider controllable technologies.

Nonetheless, the methods developed here will still be usefull when including renewables to the power mix!

## 4.2.1 - Controllable generation technologies

There are two main categories of controllable generation technologies:
- **Baseload power plants**: Capital intensive, with inertia, but cheap to run. They are used to cover the baseload, as they need to run long enough to be economically efficient.
- **Peak load power plants**: OPEX intensive, but low CAPEX and modular. They are used to cover the peakloads, as they can be started quickly, and should be running to long as they are expensive.


Among all existing technologies, based on ERAA data, we consider here:
- **Nuclear PP**: typical baseload (technical & economic)
- **Biomass PP**: usually peakload (economic)
- **Oil PP**: typical peakload (economic)
- **CCGT**: usually baseload (technical)
- **OCGT**: usually peakload (economic)
- **Hard Coal PP**: usually baseload (technical)
- **Lignite**: usually baseload (technical)


**Remark**: Storage assets, such as hydro reservoirs and pumped hydro or batteries, can also be considered as controllable technologies, but are not considered in this data crunch session. Storage require more advanded modelling. ***We will discuss storage at the end of this session***

### 4.2.2 - Economic and environmental characteristics

In this subsection we are going to investigate the main characteristics of the considered technologies:

- **Environmental**: CO2eq emissions
- **Economic**:
    - <u>**Fixed cost**, annualized, include:</u>
        - Investment costs (CAPEX)
        - Cost of capital (WACC)
        - Fixed O&M costs (FOM)

    - <u>**Variable costs** include:</u>
        - Fuel Cost
        - CO2 taxes
        - Efficiency
        - Variable O&M costs

#### Fixed costs

To be able to analyze FOM & CAPEX together, include cost of capital, and compare technologies with different operational lifespan, we need to annualize the costs first 

In [ ]:
# Cost of Capital + CAPEX computation function
def vpm(taux, nper, va, vc=0, type=0):
    """
    Reproduit la fonction Excel VPM(taux; nper; va; [vc]; [type])

    taux : taux par période
    nper : nombre total de périodes
    va   : valeur actuelle (ex: montant emprunté)
    vc   : valeur future (ex: capital accumulé) [optionnel]
    type : 0 = paiement fin de période, 1 = début de période
    """
    
    if taux == 0:
        return (va + vc) / nper
    
    return (taux * (va * (1 + taux)**nper + vc)/((1 + taux * type) * ((1 + taux)**nper - 1)))

In [ ]:
# ---- Load economic data JSON files ----
with open("data/fuel_sources/technology.json", "r") as f:
    tech_data = json.load(f)
with open("data/fuel_sources/fuels.json", "r") as f:
    fuel_data = json.load(f)

In [ ]:
# ---- Compute fixed annual costs ----
fixed_costs = {}
fixed_costs_repartition = {}

for tech, vals in tech_data.items():
    capex = vals.get("CAPEX")
    lifetime = vals.get("Lifetime")
    wacc = vals.get("WACC")
    fom = vals.get("FOM")

    if capex is None or lifetime is None or wacc is None or fom is None:
        # Renewable technologies have no CAPEX/WACC/Lifetime data in your JSON
        fixed_costs[tech] = None
        fixed_costs_repartition[tech] = None
     
        continue
    
    fixed_costs_repartition[tech] = {}
    annualized_investment = vpm(wacc, lifetime, capex)
    fixed_costs[tech] = annualized_investment + fom

    fixed_costs_repartition[tech]["invest_overnight"] = capex/lifetime
    fixed_costs_repartition[tech]["capital"] = annualized_investment - capex/lifetime
    fixed_costs_repartition[tech]["FOM"] = fom

In [ ]:
# Plotting

# List of technologies to exclude from plotting
exclude_techs = {"Solar PV", "Wind Onshore", "Wind Offshore", "Hydro ROR", "Deficit"}

# Filter only technologies that have cost data and are not excluded
techs = [t for t, v in fixed_costs_repartition.items() if v and t not in exclude_techs]

# Extract each component for filtered technologies
invest_overnight = [fixed_costs_repartition[t]["invest_overnight"] for t in techs]
capital_cost     = [fixed_costs_repartition[t]["capital"]          for t in techs]
fom_cost         = [fixed_costs_repartition[t]["FOM"]             for t in techs]

# Plot Cost Repartition with chosen WACC
fig = go.Figure()

fig.add_trace(go.Bar(
    name="Overnight Investment",
    x=techs,
    y=invest_overnight
))

fig.add_trace(go.Bar(
    name="Capital Cost (annualized - capex)",
    x=techs,
    y=capital_cost
))

fig.add_trace(go.Bar(
    name="FOM",
    x=techs,
    y=fom_cost
))

fig.update_layout(
    barmode="stack",
    title="Annualized fixed cost by technology (thermal)",
    xaxis_title="Technology",
    yaxis_title="Cost (€/MW/year)",
    template="plotly_white",
    width=1100,
    height=600,
    legend_title="Cost Component"
)

fig.show()

The WACC has a considerable impact on the overall capital costs. It depends on various factors, and it can be structural for some technologies, such as nuclear. Here is a WACC-sensitivity analysis for each technologies:

In [ ]:
# Plot cost distrib with different WACC

# Range of WACC to test
wacc_range = np.linspace(0, 0.15, 50)  # 0% to 15%

# Dictionary to store results for each tech, but only for included technologies
tech_costs_variation = {tech: [] for tech in tech_data.keys() if tech not in exclude_techs}

# Sweep over WACC values
for wacc in wacc_range:
    for tech, vals in tech_data.items():
        if tech in exclude_techs:
            continue  # Skip excluded techs
        capex = vals.get("CAPEX")
        lifetime = vals.get("Lifetime")
        fom = vals.get("FOM")
        
        if capex is None or lifetime is None or fom is None:
            continue  # Skip if data missing
        
        annualized_investment = vpm(wacc, lifetime, capex)
        total_fixed_cost = annualized_investment + fom
        tech_costs_variation[tech].append(total_fixed_cost)

# ---- Create interactive box plot with Plotly ----
fig = go.Figure()

for tech, costs in tech_costs_variation.items():
    fig.add_trace(go.Box(
        y=costs,
        name=tech,
        boxmean='sd',  # show mean and standard deviation
        marker_color='skyblue'
    ))

fig.update_layout(
    title="Variation of fixed costs with WACC (0–15%)",
    yaxis_title="Fixed costs (€/MWh)",
    xaxis_title="Technology",
    template="plotly_white"
)

fig.show()

#### Variable costs

The variable costs depends mainly on the prices of the fuel, but also on the co2 price for carbon intensive technologies. Let's analyze these

In [ ]:
# Parameter to change at will
co2_price = 100 # €/tCO2

In [ ]:
# ---- Compute variable costs ----
var_costs = {}
var_costs_repartition = {} #Repartition between VOM, fuel cost and co2 cost

for tech, vals in tech_data.items():
    fuel_type = vals.get("Fuel type")
    efficiency = vals.get("Efficiency")
    vom = vals.get("VOM")

    if fuel_type is None or efficiency is None or fuel_type not in fuel_data:
        var_costs[tech] = vom
        var_costs_repartition[tech] = {}
        continue
    
    fuel_info = fuel_data[fuel_type]
    fuel_cost_per_ton = fuel_info["cost_per_ton"]
    fuel_energy_density = fuel_info["energy_density_per_ton"]
    co2_intensity = fuel_info["co2_emissions"]/efficiency #tCO2/MWh_e
    
    fuel_cost = fuel_cost_per_ton / fuel_energy_density / efficiency
    co2_cost = co2_price * co2_intensity
    total_var_cost = fuel_cost + co2_cost + vom

    var_costs[tech] = total_var_cost
    var_costs_repartition[tech] = {
        "fuel_cost": fuel_cost,
        "co2_cost": co2_cost,
        "VOM": vom
    }

In [ ]:
# ---- Plot ----
technologies = []
fuel_costs = []
co2_costs = []
vom_costs = []

for tech, parts in var_costs_repartition.items():
    if parts == {}:
        continue  # skip non-thermal technologies

    technologies.append(tech)
    fuel_costs.append(parts.get("fuel_cost", 0))
    co2_costs.append(parts.get("co2_cost", 0))
    vom_costs.append(parts.get("VOM", 0))

# --- Plot ---
fig = go.Figure()

fig.add_trace(go.Bar(
    x=technologies,
    y=fuel_costs,
    name="Fuel cost",
))

fig.add_trace(go.Bar(
    x=technologies,
    y=co2_costs,
    name="CO₂ cost",
))

fig.add_trace(go.Bar(
    x=technologies,
    y=vom_costs,
    name="VOM",
))

fig.update_layout(
    title=f"Variable Cost Repartition (€/MWh) | CO2 price = {co2_price} €/tCO2",
    barmode="stack",
    xaxis_title="Technology",
    yaxis_title="€/MWh",
    font=dict(size=14)
)

fig.show()

In [ ]:
# ---- Test sensitivity of variable costs to CO₂ price ----
# Range of CO₂ prices to test
co2_price_range = np.linspace(0, 200, 25)  # €/tCO₂

# Dictionary to store variable costs for each tech
tech_var_costs_variation = {tech: [] for tech in tech_data.keys()}

# --- Sweep over CO₂ prices ---
for co2_price_var in co2_price_range:
    for tech, vals in tech_data.items():
        fuel_type  = vals.get("Fuel type")
        efficiency = vals.get("Efficiency")
        vom        = vals.get("VOM", 0)
        
        if fuel_type is None or fuel_type not in fuel_data or efficiency is None:
            continue  # Skip non-thermal or missing data

        fuel_info = fuel_data[fuel_type]
        fuel_cost_per_ton   = fuel_info["cost_per_ton"]
        fuel_energy_density = fuel_info["energy_density_per_ton"]
        co2_intensity       = fuel_info["co2_emissions"]

        # Fuel cost €/MWh_e
        fuel_cost = fuel_cost_per_ton / fuel_energy_density / efficiency

        # CO₂ cost €/MWh_e
        co2_cost = co2_price_var * co2_intensity / efficiency

        # Total variable cost
        total_var_cost = fuel_cost + co2_cost + vom
        tech_var_costs_variation[tech].append(total_var_cost)

# ---- Create interactive box plot with Plotly ----
fig = go.Figure()

for tech, costs in tech_var_costs_variation.items():
    fig.add_trace(go.Box(
        y=costs,
        name=tech,
        boxmean='sd',  # show mean and standard deviation
        marker_color='lightgreen'
    ))

fig.update_layout(
    title="Sensitivity to CO₂ Price (0–200 €/tCO₂)",
    yaxis_title="Variable costs (€/MWh)",
    xaxis_title="Technology",
    template="plotly_white"
)

fig.show()

Questions:
- Which technologies are the most affected?
- How does the CO2 price level affect the **merit order**?

In [ ]:
# Test sensitivity to fuel price

# Range of fuel price increase (from 0 to 100% with a step of 10)
fuel_price_range = np.linspace(0, 100, 10)  # 0% to 100%

# Dictionary to store variable costs for each tech
tech_var_costs_variation = {tech: [] for tech in tech_data.keys()}

# --- Sweep over fuel price increase ---
for fuel_price_var in fuel_price_range:
    for tech, vals in tech_data.items():
        fuel_type = vals.get("Fuel type")
        efficiency = vals.get("Efficiency")
        vom        = vals.get("VOM", 0)
        
        if fuel_type is None or fuel_type not in fuel_data or efficiency is None:
            continue  # Skip non-thermal or missing data

        fuel_info = fuel_data[fuel_type]
        fuel_cost_per_ton = fuel_info["cost_per_ton"] * (1 + fuel_price_var / 100)
        fuel_energy_density = fuel_info["energy_density_per_ton"]
        co2_intensity = fuel_info["co2_emissions"] / efficiency  # tCO2/MWh_e

        # Fuel cost €/MWh_e
        fuel_cost = fuel_cost_per_ton / fuel_energy_density / efficiency
        # CO₂ cost €/MWh_e (use fixed CO₂ price as in the earlier correct computation)
        co2_cost = co2_price * co2_intensity
        # Total variable cost
        total_var_cost = fuel_cost + co2_cost + vom
        tech_var_costs_variation[tech].append(total_var_cost)

# ---- Create interactive box plot with Plotly ----
fig = go.Figure()

for tech, costs in tech_var_costs_variation.items():
    fig.add_trace(go.Box(
        y=costs,
        name=tech,
        boxmean='sd',  # show mean and standard deviation
        marker_color='lightgreen'
    ))

fig.update_layout(
    title="Sensibility to fuel price increase (0–100%)",
    yaxis_title="Variable costs (€/MWh)",
    xaxis_title="Technology",
    template="plotly_white"
)

fig.show()

Questions:
- Which technologies are the most affected by fuel price increases?
- Which countries produce these fuels? Are they located in the EU?
- Can you relate with EU ernergy independence goals?
- What could be your recommendations regarding your country's future energy mix?

#### GHG Emissions

As we are aiming to decarbionize our energy system in Europe, it is also important to achieve not only a reliable and cost effective network, but to also set some ambitious GHG emissions reduction goals.

It is important not to only consider marginal emissions, that are due to the operation, but the overall lifetime emissions (*from well-to-grave*). Indeed, while capital intensive capacities, such as Nuclear, or renewables have very low operational emissions, their construction and end of life concentrate most of the emissions.

To lighten your decisions here are some indicators that may help:

IPCC data (https://www.ipcc.ch/site/assets/uploads/2018/02/ipcc_wg3_ar5_annex-iii.pdf)

*More details can be found in this annex. Latest IPCC reports also provide some updated data, but not as comprehensive*

In [ ]:
# Collect IPCC data
ipcc_df = pd.read_csv('data/fuel_sources/IPCC_AR5_TableAIII2_full.csv')

# Plot IPCC lifecycle emissions data

# Prepare the data for plotting
technologies = ipcc_df["Technology"]
min_values = ipcc_df["Lifecycle_emissions_Min"]
median_values = ipcc_df["Lifecycle_emissions_Median"]
max_values = ipcc_df["Lifecycle_emissions_Max"]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=min_values,
    y=technologies,
    mode='markers',
    marker=dict(color='green', symbol='line-ns-open', size=12),
    name='Min'
))
fig.add_trace(go.Scatter(
    x=median_values,
    y=technologies,
    mode='markers',
    marker=dict(color='blue', symbol='circle', size=12),
    name='Median'
))
fig.add_trace(go.Scatter(
    x=max_values,
    y=technologies,
    mode='markers',
    marker=dict(color='red', symbol='line-ns-open', size=12),
    name='Max'
))

# Add lines showing min to max for each technology
for idx, tech in enumerate(technologies):
    fig.add_shape(
        type='line',
        x0=min_values.iloc[idx], x1=max_values.iloc[idx],
        y0=tech, y1=tech,
        line=dict(color='gray', width=2),
        xref='x', yref='y'
    )

fig.update_layout(
    title="Lifecycle GHG Emissions by Technology",
    xaxis_title="Lifecycle Emissions (gCO₂eq/kWhe)",
    yaxis_title="Technology",
    template="plotly_white",
    legend_title="Statistic",
    height=500
)

fig.show()


**WARNING: DATA MISSING FOR OIL AND OCGT + 'Coal PC' emmbed to Lignite & Hard coal**

ERAA data (indirect) - Operational emissions

In [ ]:
# Calculate operational CO2 intensities for each technology (kg CO2/MWh)
tech_intensity = {}

for tech, vals in tech_data.items():
    fuel_type  = vals.get("Fuel type")
    efficiency = vals.get("Efficiency")
    if fuel_type is None or fuel_type not in fuel_data or efficiency is None:
        continue  # Skip non-thermal or missing data

    fuel_info = fuel_data[fuel_type]
    co2_intensity = fuel_info["co2_emissions"] * 1000 / efficiency  # kgCO2eq/MWh(el)
    tech_intensity[tech] = co2_intensity

# ---- Create a bar plot with Plotly for technology intensities ----
import plotly.graph_objs as go

tech_names = list(tech_intensity.keys())
intensities = [tech_intensity[tech] for tech in tech_names]

fig = go.Figure(go.Bar(
    x=intensities,
    y=tech_names,
    orientation='h',
    marker=dict(color='grey'),
    text=[f"{val:.0f}" for val in intensities],
    textposition='auto',
))
fig.update_layout(
    title="Operational CO₂ intensity by thermal technology (kgCO₂/MWh)",
    xaxis_title="CO₂ Intensity (kgCO₂/MWh)",
    yaxis_title="Technology",
    template="plotly_white",
    height=400
)
fig.show()


#### Beyond GHG

Systematic approach should always be considered when designing a new (energy) system. Limiting our analysis to GHG is convenient as it enables to compare means of production with on a singular dimension, with a normalized unit: CO2eq.

However, the environmental transition cannot be restricted to GHG emission reduction. The 9-boundary circle is a relevant representation of the Earth system critical components (https://www.stockholmresilience.org/research/planetary-boundaries.html):

<img src="images/PB_2025.jpg" alt="Planetary Boundaries" width="500">

Usual indicators that are usually considered for power generation can be:
- Fresh water usage
- Land-use
- *(Critical)* minerals usage

If you want to further investigate these dimensions and compare different technologies, you can look for comparative **Life Cycle Assessements (cLCA)**. They develop normalized quantitative frameworks for different dimensions of the environmental impacts.

(A good point of entry: https://www-sciencedirect-com.ezproxy.universite-paris-saclay.fr/science/article/pii/S1364032113005534?via%3Dihub)

### 4.2.3 - Boiteux stacking method

Now you hopefully have a better idea of the technologies available, their economic and environmental characteristics.

But how to build the most economic-efficient power system, to cover the gross power demand? This is the question Marcel Boiteux, eminent economist and former CEO of EDF, answered.

***NOTEBOOK BREAK- BOITEUX'S STACKING METHOD***

Questions:
- What is the goal of Boiteux's stacking method?
- Why ranging the hours of the year by power demand?
- How to get the yearly minimum activation time for a power plant?
- How to visually conclude on the capacity to install for each technology of power plant?
- What is the impact of the VOLL on the capacity sizing? Can you relate with risk management ? How to ensure an average shortage duration using the VOLL? 
- What is you decide to have a different VOLL compared to your neighbor?
- What are the limits of this methodology?

Application to your country

1. Compute minimum activation durations for each technologies

In [ ]:
co2_price = 100 # €/tcO2eq

In [ ]:
# ============================================================
# 1. Compute fixed & variable costs
# ============================================================

# ---- Fixed annual costs ----
fixed_costs = {}
fixed_costs_repartition = {}

for tech, vals in tech_data.items():
    capex = vals.get("CAPEX")
    lifetime = vals.get("Lifetime")
    wacc = vals.get("WACC")
    fom = vals.get("FOM")

    if capex is None or lifetime is None or wacc is None or fom is None:
        # Renewable technologies have no CAPEX/WACC/Lifetime data in your JSON
        fixed_costs[tech] = None
        fixed_costs_repartition[tech] = None
        continue

    fixed_costs_repartition[tech] = {}
    annualized_investment = vpm(wacc, lifetime, capex)
    fixed_costs[tech] = annualized_investment + fom

    fixed_costs_repartition[tech]["invest_overnight"] = capex / lifetime
    fixed_costs_repartition[tech]["capital"] = annualized_investment - capex / lifetime
    fixed_costs_repartition[tech]["FOM"] = fom


# ---- Variable costs ----
var_costs = {}
var_costs_repartition = {}  # Repartition between VOM, fuel cost and CO2 cost

for tech, vals in tech_data.items():
    fuel_type = vals.get("Fuel type")
    efficiency = vals.get("Efficiency")
    vom = vals.get("VOM")

    if fuel_type is None or efficiency is None or fuel_type not in fuel_data:
        var_costs[tech] = vom
        var_costs_repartition[tech] = {}
        continue

    fuel_info = fuel_data[fuel_type]
    fuel_cost_per_ton = fuel_info["cost_per_ton"]
    fuel_energy_density = fuel_info["energy_density_per_ton"]
    co2_intensity = fuel_info["co2_emissions"] / efficiency  # tCO2/MWh_e

    fuel_cost = fuel_cost_per_ton / fuel_energy_density / efficiency
    co2_cost = co2_price * co2_intensity
    total_var_cost = fuel_cost + co2_cost + vom

    var_costs[tech] = total_var_cost
    var_costs_repartition[tech] = {
        "fuel_cost": fuel_cost,
        "co2_cost": co2_cost,
        "VOM": vom,
    }


# ---- Commitable technologies with valid fixed cost data ----
techs = [
    tech for tech, vals in tech_data.items()
    if vals.get("Commitable") and fixed_costs.get(tech) is not None
]


# ============================================================
# 2. Summary table
# ============================================================

summary_data = [
    {
        "Technology": tech,
        "Fixed cost (EUR/MW/year)": fixed_costs[tech],
        "Variable cost (EUR/MWh)": var_costs[tech],
    }
    for tech in techs
]

df_costs = pd.DataFrame(summary_data).set_index("Technology").round(2)

display(df_costs)  # nicely rendered table in Jupyter


# ============================================================
# 3. Boiteux stacking chart with intersections
# ============================================================

full_load_hours = np.linspace(0, 8760, 200)

fig = go.Figure()

for tech in techs:
    fc = fixed_costs[tech]
    vc = var_costs[tech]
    y = fc + full_load_hours * vc

    fig.add_trace(
        go.Scatter(
            x=full_load_hours,
            y=y,
            mode="lines",
            name=tech,
            hovertemplate=(
                f"<b>{tech}</b><br>"
                "Full load hours: %{x:.0f} h<br>"
                "Annual cost: %{y:,.0f} EUR/MW<extra></extra>"
            ),
        )
    )

# --- Pairwise intersections (for plotting) ---
intersections_x = []
intersections_y = []
intersections_text = []

for tech1, tech2 in combinations(techs, 2):
    fc1, vc1 = fixed_costs[tech1], var_costs[tech1]
    fc2, vc2 = fixed_costs[tech2], var_costs[tech2]

    if vc1 == vc2:
        continue  # parallel lines, no intersection

    x_star = (fc2 - fc1) / (vc1 - vc2)
    y_star = fc1 + x_star * vc1

    if 0 <= x_star <= 8760:
        intersections_x.append(x_star)
        intersections_y.append(y_star)
        intersections_text.append(f"{tech1} / {tech2}<br>{x_star:.0f} h — {y_star:,.0f} EUR/MW")

fig.add_trace(
    go.Scatter(
        x=intersections_x,
        y=intersections_y,
        mode="markers",
        marker=dict(size=9, color="black", symbol="x"),
        name="Intersections",
        hovertext=intersections_text,
        hoverinfo="text",
    )
)

fig.update_layout(
    title="Economic competitivity ranges by technology",
    xaxis_title="Full load hours (h/year)",
    yaxis_title="Annualized cost (EUR/MW/year)",
    hovermode="closest",
    template="plotly_white",
    legend_title="Technology",
    width=950,
    height=600,
)

fig.show()


# ============================================================
# 4. Minimum activation duration per technology (lower envelope)
# ============================================================

x_grid = np.linspace(0, 8760, 5000)
cost_matrix = np.array([
    fixed_costs[tech] + x_grid * var_costs[tech] for tech in techs
])

argmin_idx = np.argmin(cost_matrix, axis=0)

print("\nMinimal activation duration :\n")
for i, tech in enumerate(techs):
    matching_x = x_grid[argmin_idx == i]
    if matching_x.size == 0:
        print(f"  {tech:<15} : never optimal (outside minimal enveloppe)")
    elif matching_x.min() == x_grid[0]:
        # This tech is optimal starting at x=0, so the lower bound (0) is
        # trivial/uninformative. Use the upper bound of its interval instead.
        hours = matching_x.max()+1
    else:
        hours = matching_x.min()+1

    if matching_x.size > 0:
        print(f"  {tech:<15} : {hours:,.0f} h/an")

2. Apply these duration on the load duration curve to find the capacity to install

In [ ]:
# Choose the demand reference year
plot_year=2025

# Choose the climate year to consider
cy = "Avg_WS"

Reminder on the load duraction curve of your case study

In [ ]:
# ---- Plot ----
fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=ldc[plot_year].index,          # Hour rank
        y=ldc[plot_year][cy]/1e3,      # Sorted load values
        mode="lines",
        name=str(year)
    )
)

fig.update_layout(
    title=f"Load Duration Curves – {country_choice.capitalize()} - Average Weather Scenario",
    xaxis_title="Hours",
    yaxis_title="Demand (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600
)

fig.show()

Question: Case you guess the capacities to install to be economically optimal ?

#### Solution

In [ ]:
# ============================================================
# 5. Boiteux graphical stacking on the Load Duration Curve
# ============================================================

# Color map
tech_color_map = {
    "Nuclear": "rgb(200, 160, 0)",     # dark yellow
    "CCGT": "rgb(140, 140, 140)",      # medium grey
    "OCGT": "rgb(80, 80, 80)",         # dark grey
    "Deficit": "rgb(220, 0, 0)",       # red
    "Oil": "rgb(127, 0, 255)",         # purple
    "Hard Coal": "rgb(20,20,20)",      # black
    "Lignite": "rgb(40,40,40)",        # black
    "Biomass": "rgb(165,42,42)"        # brown
}

# --- Step 1: merit order (peak -> base) and exact switching hours ---

order_idx = []
breakpoints = []
prev = None
for j, idx in enumerate(argmin_idx):
    if idx != prev:
        order_idx.append(idx)
        breakpoints.append(x_grid[j])
        prev = idx

order_techs = [techs[i] for i in order_idx]
stacking_order = list(reversed(order_techs))
stacking_breakpoints = list(reversed(breakpoints))


# --- Step 2: read cumulative capacity directly on the LDC at each breakpoint ---

demand_curve = ldc[plot_year][cy].sort_index() / 1e3  # GW, indexed by hour rank

def ldc_value_at(hour):
    return np.interp(hour, demand_curve.index, demand_curve.values)

cumulative_capacity = {}
prev_capacity = 0.0
for tech, bp in zip(stacking_order, stacking_breakpoints):
    cap_at_breakpoint = ldc_value_at(bp) if bp > 0 else demand_curve.iloc[0]
    cumulative_capacity[tech] = max(cap_at_breakpoint, prev_capacity)
    prev_capacity = cumulative_capacity[tech]

cumulative_capacity[stacking_order[-1]] = demand_curve.iloc[0]

# --- Step 2bis: own (non-cumulative) capacity to install per technology ---

own_capacity_to_install = {}
prev_cap = 0.0
for tech in stacking_order:
    own_capacity_to_install[tech] = cumulative_capacity[tech] - prev_cap
    prev_cap = cumulative_capacity[tech]

print("Own capacity to install per technology (GW):\n")
for tech in stacking_order:
    print(f"  {tech:<15} : {own_capacity_to_install[tech]:,.2f} GW")

# --- Step 3: build the stacked area plot ---

x_hours = demand_curve.index.values
y_demand = demand_curve.values

fig = go.Figure()
prev_layer = np.zeros_like(x_hours, dtype=float)
areas = {}  # <-- NEW: annual generation per technology (GWh)

for tech in stacking_order:
    layer_top = np.minimum(cumulative_capacity[tech], y_demand)
    layer_top = np.maximum(layer_top, prev_layer)

    own_capacity = layer_top - prev_layer
    areas[tech] = np.trapezoid(own_capacity, x_hours)  # GW * h = GWh   <-- NEW

    fig.add_trace(
        go.Scatter(
            x=x_hours,
            y=layer_top,
            mode="lines",
            line=dict(width=0.5, color=tech_color_map.get(tech, "rgb(100,100,100)")),
            fillcolor=tech_color_map.get(tech, "rgb(100,100,100)"),
            fill="tozeroy" if tech == stacking_order[0] else "tonexty",
            name=tech,
            customdata=own_capacity,
            hovertemplate=(
                f"<b>{tech}</b><br>"
                "Hour: %{x:.0f}<br>"
                "Capacity: %{customdata:.2f} GW<extra></extra>"
            ),
        )
    )
    prev_layer = layer_top


fig.add_trace(
    go.Scatter(
        x=x_hours,
        y=y_demand,
        mode="lines",
        name="Load duration curve",
        line=dict(color="black", width=2, dash="dot"),
    )
)

fig.update_layout(
    title=f"Boiteux Stacking on Load Duration Curve – {country_choice.capitalize()} - WS {cy} - {plot_year}",
    xaxis_title="Hours",
    yaxis_title="Capacity / Demand (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600,
)

fig.show()

Questions:
- What happens when you chnage the climatic year?
- Are all climiatic year equiprobable?
- On which one shoul you size your system?

#### Assessment of this first 100% thermal mix

In [ ]:
# ============================================================
# 6. Economic & environmental assessment of the resulting mix
# ============================================================

unit_factor = 1e3  # areas is in GWh -> convert to MWh

# --- Exclude the fictive Deficit technology ---
assessed_techs = [t for t in stacking_order if t != "Deficit"]

print("Annual generation per technology (GWh/year):\n")
for tech in assessed_techs:
    print(f"  {tech:<15} : {areas[tech]:,.0f} GWh/year")


# ------------------------------------------------------------
# 6.1 Economic assessment: total annual system bill
# ------------------------------------------------------------

bill_breakdown = []
for tech in assessed_techs:
    capacity_MW = own_capacity_to_install[tech] * 1e3  # GW -> MW
    generation_MWh = areas[tech] * unit_factor

    fixed_bill = fixed_costs[tech] * capacity_MW
    variable_bill = var_costs[tech] * generation_MWh
    total_bill = fixed_bill + variable_bill

    bill_breakdown.append({
        "Technology": tech,
        "Capacity (GW)": own_capacity_to_install[tech],
        "Generation (MWh)": generation_MWh,
        "Fixed cost (EUR/year)": fixed_bill,
        "Variable cost (EUR/year)": variable_bill,
        "Total cost (EUR/year)": total_bill,
    })

df_bill = pd.DataFrame(bill_breakdown).set_index("Technology")
total_system_bill = df_bill["Total cost (EUR/year)"].sum()

print(f"\nTotal annual system bill (excl. Deficit): {total_system_bill:,.0f} EUR/year\n")
display(df_bill.round(0))


# ------------------------------------------------------------
# 6.2 Environmental assessment — 1) Operational (direct) emissions
# ------------------------------------------------------------

operational_emissions = {}
for tech in assessed_techs:
    fuel_type = tech_data[tech]["Fuel type"]
    efficiency = tech_data[tech]["Efficiency"]
    co2_intensity = fuel_data[fuel_type]["co2_emissions"] / efficiency  # tCO2/MWh_e

    generation_MWh = areas[tech] * unit_factor
    operational_emissions[tech] = co2_intensity * generation_MWh  # tCO2/year

total_operational_emissions = sum(operational_emissions.values())

print(f"\nTotal operational (direct) emissions: {total_operational_emissions:,.0f} tCO2/year\n")
for tech in assessed_techs:
    print(f"  {tech:<15} : {operational_emissions[tech]:,.0f} tCO2/year")


# ------------------------------------------------------------
# 6.2 Environmental assessment — 2) System (lifecycle) emissions
#     using IPCC AR5 median values
# ------------------------------------------------------------

# Mapping our technology names -> IPCC AR5 Table A.III.2 technology rows.
# NOTE: two special cases are handled outside the direct IPCC lookup:
#   - Oil: not covered by IPCC AR5 Table A.III.2 at all. We assume
#     840 gCO2eq/kWh (order-of-magnitude literature value for oil-fired
#     lifecycle emissions, since no IPCC figure exists for this fuel).
#   - OCGT: IPCC AR5 only reports a "Gas - Combined Cycle" figure, not a
#     separate open-cycle value. We approximate OCGT lifecycle emissions
#     by scaling the CCGT median with the efficiency ratio:
#         OCGT_emissions = CCGT_emissions * (eff_CCGT / eff_OCGT)
#     This reflects that, for the same fuel, a less efficient conversion
#     process emits proportionally more per unit of electricity produced.
ipcc_mapping = {
    "Nuclear":   ("Nuclear", "exact match"),
    "CCGT":      ("Gas - Combined Cycle", "exact match"),
    "OCGT":      ("Gas - Combined Cycle", "APPROXIMATION: scaled by efficiency ratio CCGT/OCGT (see note above)"),
    "Hard Coal": ("Coal - PC", "exact match"),
    "Lignite":   ("Coal - PC", "APPROXIMATION: IPCC AR5 does not separate lignite from hard coal"),
    "Biomass":   ("Biomass - dedicated", "assumed dedicated (not cofiring)"),
    "Oil":       (None, "APPROXIMATION: not covered by IPCC AR5; assuming 840 gCO2eq/kWh (literature order-of-magnitude value)"),
}

print("\nIPCC technology mapping used:\n")
for tech, (ipcc_tech, note) in ipcc_mapping.items():
    if tech in assessed_techs:
        print(f"  {tech:<15} -> {ipcc_tech!s:<25} ({note})")

ipcc_median_lookup = ipcc_df.set_index("Technology")["Lifecycle_emissions_Median"]

# Fixed assumption for Oil (no IPCC AR5 figure available)
OIL_LIFECYCLE_EMISSIONS_gCO2_per_kWh = 840

# Conversion: gCO2eq/kWh -> tCO2eq/MWh is a division by 1000
# (since 1 gCO2eq/kWh = 1 kgCO2eq/MWh, then kg -> t divides by another 1000... 
#  wait: 1 gCO2eq/kWh = 1 kgCO2eq/MWh directly, so kgCO2eq/MWh -> tCO2eq/MWh
#  requires dividing by 1000 once more)
GCO2_PER_KWH_TO_TCO2_PER_MWH = 1 / 1000  # kg/MWh -> t/MWh

system_emissions = {}
for tech in assessed_techs:
    ipcc_tech, _ = ipcc_mapping.get(tech, (None, "no mapping defined"))
    generation_MWh = areas[tech] * unit_factor

    if tech == "Oil":
        median_gCO2_per_kWh = OIL_LIFECYCLE_EMISSIONS_gCO2_per_kWh

    elif tech == "OCGT":
        ccgt_median = ipcc_median_lookup["Gas - Combined Cycle"]
        eff_ccgt = tech_data["CCGT"]["Efficiency"]
        eff_ocgt = tech_data["OCGT"]["Efficiency"]
        median_gCO2_per_kWh = ccgt_median * (eff_ccgt / eff_ocgt)

    elif ipcc_tech is None or ipcc_tech not in ipcc_median_lookup.index:
        print(f"  WARNING: no lifecycle emission factor available for '{tech}' — skipped")
        continue

    else:
        median_gCO2_per_kWh = ipcc_median_lookup[ipcc_tech]

    carbon_intensity_tCO2_per_MWh = median_gCO2_per_kWh * GCO2_PER_KWH_TO_TCO2_PER_MWH
    system_emissions[tech] = carbon_intensity_tCO2_per_MWh * generation_MWh  # tCO2eq/year

total_system_emissions = sum(system_emissions.values())

print(f"\nTotal system (lifecycle) emissions: {total_system_emissions:,.0f} tCO2eq/year\n")
for tech in assessed_techs:
    if tech in system_emissions:
        print(f"  {tech:<15} : {system_emissions[tech]:,.0f} tCO2eq/year")


# ------------------------------------------------------------
# 6.3 Summary comparison table
# ------------------------------------------------------------

summary_env = pd.DataFrame({
    "Operational emissions (tCO2/year)": pd.Series(operational_emissions),
    "System (lifecycle) emissions (tCO2eq/year)": pd.Series(system_emissions),
}).round(0)

display(summary_env)

print(f"\nTOTAL BILL                  : {total_system_bill:,.0f} EUR/year")
print(f"TOTAL OPERATIONAL EMISSIONS : {total_operational_emissions:,.0f} tCO2/year")
print(f"TOTAL SYSTEM EMISSIONS      : {total_system_emissions:,.0f} tCO2eq/year")

Question:
- What happens if you change the price of CO2?

## 4.3 - Introduction of variable Renewable Energy Sources (vRES)

Nuclear and Biomass are hopefully not the only low-carbon technologies available to achieve the energy tranition. Renewables such as:
- Hydro Power (Run-of-River)
- Solar PV
- Onshore/Offshore Wind
are also low carbon, and can help achieve or transition goals.

However, contrarily to nuclear and biomass, these technologies are not controllable. It does not make them random or useless, as they can be predictible up to some extent.

**The goal of this section is to assess the value they can bring to the system.**

*Notes:*
- *We do not consider storages yet, as they are more complex to represent*
- *While this can hinder the vRES, it does not change the qualitative impacts we are going to witness*

### 4.3.1 - vRES profiles

As they are weather dependent, the profiles are usually described with *capacity factors*, ratios ranging from 0 to 1. It represents the generation per MW installed.

#### Solar PV

In [ ]:
# ---- Import and Plot ----
res_capacity_folder = "data/ERAA_2023-2/res_capa-factors"
solar_file = f"{res_capacity_folder}/capa_factor_solar_pv_{plot_year}_{country_choice}.csv"

try:
    df = pd.read_csv(solar_file, sep=";")
    df["date"] = pd.to_datetime(df["date"])
    climatic_years = sorted(df["climatic_year"].unique())

    fig = go.Figure()
    for clim_year in climatic_years:
        tmp = df[df["climatic_year"] == clim_year]
        fig.add_trace(
            go.Scatter(
                x=tmp["date"],
                y=tmp["value"] * 100,
                mode="lines",
                name=f"Clim. year {clim_year}"
            )
        )

    fig.update_layout(
        title=f"Solar PV Production Profiles for {country_choice.title()} ({plot_year})",
        xaxis_title="Date",
        yaxis_title="Generation (%)",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Climatic year",
        width=1100,
        height=600
    )

    # Format x-axis ticks to show only month abbreviations
    fig.update_xaxes(
        dtick="M1",
        tickformat="%b"
    )

    fig.show()
except Exception as e:
    print(f"Could not plot solar profile {solar_file}: {e}")

In [ ]:
# ---- Import and Plot: quantile envelopes across climatic years ----
res_capacity_folder = "data/ERAA_2023-2/res_capa-factors"
solar_file = f"{res_capacity_folder}/capa_factor_solar_pv_{plot_year}_{country_choice}.csv"

try:
    df = pd.read_csv(solar_file, sep=";")
    df["date"] = pd.to_datetime(df["date"])

    # Use month-day-hour as the common x-axis (drop the year component)
    df["x_label"] = df["date"].dt.strftime("%m-%d %H:%M")

    # Pivot: rows = x_label (chronological order), columns = climatic_year
    pivot = df.pivot_table(index="x_label", columns="climatic_year", values="value")
    pivot = pivot.reindex(df.sort_values("date")["x_label"].unique())  # keep chronological order

    # Compute quantiles across climatic years, at each timestamp
    quantile_levels = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]  # min, P10, P25, median, P75, P90, max
    quantiles = pivot.quantile(quantile_levels, axis=1).T  # index = x_label, columns = quantile levels

    # Convert x_label back to a real datetime axis (common reference year)
    # so that month-based tick formatting works correctly.
    x_vals = pd.to_datetime(quantiles.index, format="%m-%d %H:%M")

    fig = go.Figure()

    # --- Outer envelope: min-max (light shading) ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.0] * 100,
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[1.0] * 100,
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(255,165,0,0.15)",
        name="Min–Max range",
        hoverinfo="skip",
    ))

    # --- Inner envelope: P10-P90 (medium shading) ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.1] * 100,
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.9] * 100,
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(255,165,0,0.3)",
        name="P10–P90 range",
        hoverinfo="skip",
    ))

    # --- Interquartile envelope: P25-P75 (darker shading) ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.25] * 100,
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.75] * 100,
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(255,165,0,0.45)",
        name="P25–P75 range",
        hoverinfo="skip",
    ))

    # --- Median line ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.5] * 100,
        mode="lines", line=dict(color="rgb(200,100,0)", width=1.5),
        name="Median",
    ))

    fig.update_layout(
        title=f"Solar PV Production Profiles for {country_choice.title()} ({plot_year}) — Quantile Envelopes",
        xaxis_title="Date",
        yaxis_title="Generation (%)",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Statistic",
        width=1100,
        height=600
    )

    # Format x-axis ticks to show only month abbreviations
    fig.update_xaxes(
        dtick="M1",
        tickformat="%b"
    )

    fig.show()

except Exception as e:
    print(f"Could not plot solar profile {solar_file}: {e}")

We do not have a lot of weather scenarios in our practical session, which is why this lacks clarity. Below Is a screenshot of a similar project, but including 36 WS from ERAA2025 (**FOR FRANCE**)

<img src="images/SolarPV_quantiles_36WS" alt="Solar PV profile (quantiles, ERAA 2024, 36WS)" width="800">

Questions:
- What are the main patterns?
- Is there a high volatility?
- Do you think the prediction rate can be good for solar PV?
- Is the production aligned with consumption (daily and seasonal analysis)?


#### Wind Onshore

In [ ]:
# ---- Import and Plot ----
res_capacity_folder = "data/ERAA_2023-2/res_capa-factors"
wind_onshore_file = f"{res_capacity_folder}/capa_factor_wind_onshore_{plot_year}_{country_choice}.csv"

try:
    df = pd.read_csv(wind_onshore_file, sep=";")
    df["date"] = pd.to_datetime(df["date"])
    climatic_years = sorted(df["climatic_year"].unique())

    fig = go.Figure()
    for clim_year in climatic_years:
        tmp = df[df["climatic_year"] == clim_year]
        fig.add_trace(
            go.Scatter(
                x=tmp["date"],
                y=tmp["value"] * 100,
                mode="lines",
                name=f"Clim. year {clim_year}"
            )
        )

    fig.update_layout(
        title=f"Onshore Wind Production Profiles for {country_choice.title()} ({plot_year})",
        xaxis_title="Date",
        yaxis_title="Generation (%)",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Climatic year",
        width=1100,
        height=600
    )

    # Format x-axis ticks to show only month abbreviations
    fig.update_xaxes(
        dtick="M1",
        tickformat="%b"
    )

    fig.show()
except Exception as e:
    print(f"Could not plot wind onshore profile {wind_onshore_file}: {e}")

In [ ]:
# ---- Import and Plot: quantile envelopes across climatic years ----
res_capacity_folder = "data/ERAA_2023-2/res_capa-factors"
wind_onshore_file = f"{res_capacity_folder}/capa_factor_wind_onshore_{plot_year}_{country_choice}.csv"

try:
    df = pd.read_csv(wind_onshore_file, sep=";")
    df["date"] = pd.to_datetime(df["date"])

    # Use month-day-hour as the common x-axis (drop the year component)
    df["x_label"] = df["date"].dt.strftime("%m-%d %H:%M")

    # Pivot: rows = x_label (chronological order), columns = climatic_year
    pivot = df.pivot_table(index="x_label", columns="climatic_year", values="value")
    pivot = pivot.reindex(df.sort_values("date")["x_label"].unique())  # keep chronological order

    # Compute quantiles across climatic years, at each timestamp
    quantile_levels = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]  # min, P10, P25, median, P75, P90, max
    quantiles = pivot.quantile(quantile_levels, axis=1).T  # index = x_label, columns = quantile levels

    # Convert x_label back to a real datetime axis (common reference year)
    # so that month-based tick formatting works correctly.
    x_vals = pd.to_datetime(quantiles.index, format="%m-%d %H:%M")

    fig = go.Figure()

    # --- Outer envelope: min-max (light shading) ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.0] * 100,
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[1.0] * 100,
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(255,165,0,0.15)",
        name="Min–Max range",
        hoverinfo="skip",
    ))

    # --- Inner envelope: P10-P90 (medium shading) ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.1] * 100,
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.9] * 100,
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(255,165,0,0.3)",
        name="P10–P90 range",
        hoverinfo="skip",
    ))

    # --- Interquartile envelope: P25-P75 (darker shading) ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.25] * 100,
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.75] * 100,
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(255,165,0,0.45)",
        name="P25–P75 range",
        hoverinfo="skip",
    ))

    # --- Median line ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.5] * 100,
        mode="lines", line=dict(color="rgb(200,100,0)", width=1.5),
        name="Median",
    ))

    fig.update_layout(
        title=f"Onshore Wind Production Profiles for {country_choice.title()} ({plot_year}) — Quantile Envelopes",
        xaxis_title="Date",
        yaxis_title="Generation (%)",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Statistic",
        width=1100,
        height=600
    )

    # Format x-axis ticks to show only month abbreviations
    fig.update_xaxes(
        dtick="M1",
        tickformat="%b"
    )

    fig.show()

except Exception as e:
    print(f"Could not plot wind onshore profile {wind_onshore_file}: {e}")

We do not have a lot of weather scenarios in our practical session, which is why this lacks clarity. Below is a screenshot of a similar project, but including 36 WS from ERAA2025 (**FOR FRANCE**).

<img src="images/wind_onshore_ERAA2024_36WS" alt="Onshore Wind profile (quantiles, ERAA 2024, 36WS)" width="800">

Questions:
- What are the main patterns?
- Is there a high volatility?
- Do you think the prediction rate can be good for solar PV?
- Any complementarity with solar PV?
- Is the production aligned with consumption (daily and seasonal analysis)?

#### Wind Offshore

In [ ]:
# ---- Import and Plot ----
res_capacity_folder = "data/ERAA_2023-2/res_capa-factors"
wind_offshore_file = f"{res_capacity_folder}/capa_factor_wind_offshore_{plot_year}_{country_choice}.csv"

try:
    df = pd.read_csv(wind_offshore_file, sep=";")
    df["date"] = pd.to_datetime(df["date"])
    climatic_years = sorted(df["climatic_year"].unique())

    fig = go.Figure()
    for clim_year in climatic_years:
        tmp = df[df["climatic_year"] == clim_year]
        fig.add_trace(
            go.Scatter(
                x=tmp["date"],
                y=tmp["value"] * 100,
                mode="lines",
                name=f"Clim. year {clim_year}"
            )
        )

    fig.update_layout(
        title=f"Offshore Wind Production Profiles for {country_choice.title()} ({plot_year})",
        xaxis_title="Date",
        yaxis_title="Generation (%)",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Climatic year",
        width=1100,
        height=600
    )

    # Format x-axis ticks to show only month abbreviations
    fig.update_xaxes(
        dtick="M1",
        tickformat="%b"
    )

    fig.show()
except Exception as e:
    print(f"Could not plot wind offshore profile {wind_offshore_file}: {e}")

In [ ]:
# ---- Import and Plot: quantile envelopes across climatic years ----
res_capacity_folder = "data/ERAA_2023-2/res_capa-factors"
wind_offshore_file = f"{res_capacity_folder}/capa_factor_wind_offshore_{plot_year}_{country_choice}.csv"

try:
    df = pd.read_csv(wind_offshore_file, sep=";")
    df["date"] = pd.to_datetime(df["date"])

    # Use month-day-hour as the common x-axis (drop the year component)
    df["x_label"] = df["date"].dt.strftime("%m-%d %H:%M")

    # Pivot: rows = x_label (chronological order), columns = climatic_year
    pivot = df.pivot_table(index="x_label", columns="climatic_year", values="value")
    pivot = pivot.reindex(df.sort_values("date")["x_label"].unique())  # keep chronological order

    # Compute quantiles across climatic years, at each timestamp
    quantile_levels = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]  # min, P10, P25, median, P75, P90, max
    quantiles = pivot.quantile(quantile_levels, axis=1).T  # index = x_label, columns = quantile levels

    # Convert x_label back to a real datetime axis (common reference year)
    # so that month-based tick formatting works correctly.
    x_vals = pd.to_datetime(quantiles.index, format="%m-%d %H:%M")

    fig = go.Figure()

    # --- Outer envelope: min-max (light shading) ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.0] * 100,
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[1.0] * 100,
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(255,165,0,0.15)",
        name="Min–Max range",
        hoverinfo="skip",
    ))

    # --- Inner envelope: P10-P90 (medium shading) ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.1] * 100,
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.9] * 100,
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(255,165,0,0.3)",
        name="P10–P90 range",
        hoverinfo="skip",
    ))

    # --- Interquartile envelope: P25-P75 (darker shading) ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.25] * 100,
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.75] * 100,
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(255,165,0,0.45)",
        name="P25–P75 range",
        hoverinfo="skip",
    ))

    # --- Median line ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.5] * 100,
        mode="lines", line=dict(color="rgb(200,100,0)", width=1.5),
        name="Median",
    ))

    fig.update_layout(
        title=f"Offshore Wind Production Profiles for {country_choice.title()} ({plot_year}) — Quantile Envelopes",
        xaxis_title="Date",
        yaxis_title="Generation (%)",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Statistic",
        width=1100,
        height=600
    )

    # Format x-axis ticks to show only month abbreviations
    fig.update_xaxes(
        dtick="M1",
        tickformat="%b"
    )

    fig.show()

except Exception as e:
    print(f"Could not plot wind offshore profile {wind_offshore_file}: {e}")

We do not have a lot of weather scenarios in our practical session, which is why this lacks clarity. Below is a screenshot of a similar project, but including 36 WS from ERAA2025 (**FOR FRANCE**)

<img src="images/wind_offshore_ERAA2024_36WS" alt="Onshore Wind profile (quantiles, ERAA 2024, 36WS)" width="800">

Questions:
- What are the main patterns?
- Is there a high volatility?
- Do you think the prediction rate can be good for solar PV?
- Any complementarity with solar PV?
- Is the production aligned with consumption (daily and seasonal analysis)?

#### Hydro ROR

From the data analysis, it seems this data includes both run or river and small pondage actually ("éclusée" for the french speakers).

This excludes pumped hydro and dams.

As the run of river is not a capacity that one can develop as wished (geographical constraints, we will only consider the capacities provided by the national TSOs). Therefore the analysis can directly be done in GW produced

In [ ]:
# ---- Import RoR data, build hourly timeseries, overlay all (non-leap) climatic years ----
ror_file = "data/ERAA_2023-2/hydro/PECD-hydro-daily-ror-generation.csv"  # adjust path as needed

def is_leap_year(y):
    return y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)

try:
    df = pd.read_csv(ror_file, sep=";")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # Filter for the chosen country/zone
    df = df[df["zone"] == country_choice].copy()

    # Drop leap climatic years to keep a consistent day-of-year <-> calendar
    # date mapping across all years (avoids a 1-day shift after Feb 29).
    df = df[~df["climatic_year"].apply(is_leap_year)].copy()

    # Map day-of-year to a common non-leap reference year, so all climatic
    # years overlay on the same calendar axis (year itself is meaningless
    # here, only month/day matters).
    REFERENCE_YEAR = 2001  # any non-leap year works
    df["ref_date"] = pd.to_datetime(
        str(REFERENCE_YEAR) + df["day"].astype(str).str.zfill(3),
        format="%Y%j"
    )

    # ---- Expand daily values to hourly (uniform split: daily value / 24) ----
    hourly_rows = []
    for _, row in df.iterrows():
        base_date = row["ref_date"]
        hourly_value = row["value"] / 24
        for h in range(24):
            hourly_rows.append({
                "date": base_date + pd.Timedelta(hours=h),
                "climatic_year": row["climatic_year"],
                "value": hourly_value
            })

    hourly_df = pd.DataFrame(hourly_rows).sort_values(["climatic_year", "date"]).reset_index(drop=True)

    # ---- Plot: all climatic years overlaid on a common calendar axis ----
    climatic_years = sorted(hourly_df["climatic_year"].unique())

    fig = go.Figure()
    for clim_year in climatic_years:
        tmp = hourly_df[hourly_df["climatic_year"] == clim_year]
        fig.add_trace(
            go.Scatter(
                x=tmp["date"],
                y=tmp["value"],
                mode="lines",
                name=f"Clim. year {clim_year}",
                line=dict(width=1)
            )
        )

    fig.update_layout(
        title=f"Run-of-River Hourly Production Profiles for {country_choice.title()}",
        xaxis_title="Date",
        yaxis_title="Generation (GW)",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Climatic year",
        width=1100,
        height=600
    )

    fig.update_xaxes(
        dtick="M1",
        tickformat="%b"
    )

    fig.show()

except Exception as e:
    print(f"Could not plot RoR profile {ror_file}: {e}")

In [ ]:
# ---- Import RoR data, build hourly timeseries, plot quantile envelopes ----
ror_file = "data/ERAA_2023-2/hydro/PECD-hydro-daily-ror-generation.csv"  # adjust path as needed

def is_leap_year(y):
    return y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)

try:
    df = pd.read_csv(ror_file, sep=";")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # Filter for the chosen country/zone
    df = df[df["zone"] == country_choice].copy()

    # Drop leap climatic years to keep a consistent day-of-year <-> calendar
    # date mapping across all years.
    df = df[~df["climatic_year"].apply(is_leap_year)].copy()

    # Map day-of-year to a common non-leap reference year
    REFERENCE_YEAR = 2001
    df["ref_date"] = pd.to_datetime(
        str(REFERENCE_YEAR) + df["day"].astype(str).str.zfill(3),
        format="%Y%j"
    )

    # ---- Expand daily values to hourly (uniform split: daily value / 24) ----
    hourly_rows = []
    for _, row in df.iterrows():
        base_date = row["ref_date"]
        hourly_value = row["value"] / 24
        for h in range(24):
            hourly_rows.append({
                "date": base_date + pd.Timedelta(hours=h),
                "climatic_year": row["climatic_year"],
                "value": hourly_value
            })

    hourly_df = pd.DataFrame(hourly_rows).sort_values(["climatic_year", "date"]).reset_index(drop=True)

    # ---- Pivot: rows = date (chronological order), columns = climatic_year ----
    pivot = hourly_df.pivot_table(index="date", columns="climatic_year", values="value")
    pivot = pivot.sort_index()

    # ---- Compute quantiles across climatic years, at each timestamp ----
    quantile_levels = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
    quantiles = pivot.quantile(quantile_levels, axis=1).T

    x_vals = quantiles.index

    fig = go.Figure()

    # --- Outer envelope: min-max (light shading) ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.0],
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[1.0],
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(255,165,0,0.15)",
        name="Min–Max range",
        hoverinfo="skip",
    ))

    # --- Inner envelope: P10-P90 (medium shading) ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.1],
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.9],
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(255,165,0,0.3)",
        name="P10–P90 range",
        hoverinfo="skip",
    ))

    # --- Interquartile envelope: P25-P75 (darker shading) ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.25],
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.75],
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(255,165,0,0.45)",
        name="P25–P75 range",
        hoverinfo="skip",
    ))

    # --- Median line ---
    fig.add_trace(go.Scatter(
        x=x_vals, y=quantiles[0.5],
        mode="lines", line=dict(color="rgb(200,100,0)", width=1.5),
        name="Median",
    ))

    fig.update_layout(
        title=f"Run-of-River Hourly Production Profiles for {country_choice.title()} — Quantile Envelopes",
        xaxis_title="Date",
        yaxis_title="Generation (GW)",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Statistic",
        width=1100,
        height=600
    )

    fig.update_xaxes(
        dtick="M1",
        tickformat="%b"
    )

    fig.show()

except Exception as e:
    print(f"Could not plot RoR profile {ror_file}: {e}")

Questions:
- What are the main patterns?
- Is there a high volatility?
- Do you think the prediction rate can be good for solar PV?
- Is the production aligned with consumption (daily and seasonal analysis)?

We are not going to further analyze this production as we won't have the opportunity to choose the developed capacity

### 4.3.2 - Impacts of net demand

As it is not controllable, we need to assess how naturally each type of vRES contributes to the demand supply. As they have null marginal costs, they are almost always nominated in the merit order, which justifies this approach.

#### Integrating effect of hydro ROR

In [ ]:
# Create the hourly time series for hydro ROR, for each climatic year
df_ror = pd.read_csv(ror_file, sep=";")
df_ror["value"] = pd.to_numeric(df_ror["value"], errors="coerce")
df_ror = df_ror[df_ror["zone"] == country_choice].copy()

hourly_index = pd.date_range(start="1900-01-01", periods=8760, freq="h")

hydro_ROR_ts = {}

for year in ERAA_climatic_years:
    df_year = df_ror[df_ror["climatic_year"] == year].sort_values("day")

    # Expand daily values to hourly (uniform split: daily value / 24)
    hourly_values = np.repeat(df_year["value"].values / 24, 24)

    key = f"WS{year}"
    hydro_ROR_ts[key] = hourly_values

hydro_ROR_ts = pd.DataFrame(hydro_ROR_ts, index=hourly_index) * 1000  # GW -> MW

In [ ]:
# ---- Step 1: net demand = gross demand - hydro RoR generation for each year and climatic year ----
net_demand = {}
for year in ERAA_years:
    gross_demand = demands[year].drop(columns="Avg_WS")
    gross_demand.columns = gross_demand.columns.astype(str)
    hydro_ROR_ts.columns = hydro_ROR_ts.columns.astype(str)
    for climatic_year in hydro_ROR_ts.columns:
        if climatic_year in gross_demand.columns:
            gross = gross_demand[climatic_year]
            hydro = hydro_ROR_ts[climatic_year]
            # Subtract by position (.values), not by index label, since
            # gross_demand is indexed on real `year` dates while
            # hydro_ROR_ts is indexed on a fixed reference year (1900).
            net = pd.Series(gross.values - hydro.values, index=gross.index, name=climatic_year)
            if year not in net_demand:
                net_demand[year] = {}
            net_demand[year][climatic_year] = net
            
# ---- Step 2: net load duration curve for each year and climatic year (unchanged) ----
net_ldc = {}
for year in net_demand:
    net_ldc[year] = {}
    for climatic_year in net_demand[year]:
        net_ldc[year][climatic_year] = pd.Series(
            net_demand[year][climatic_year].sort_values(ascending=False).values,
            name=climatic_year
        ).reset_index(drop=True)
    net_ldc[year] = pd.DataFrame(net_ldc[year])

In [ ]:
plot_year = 2033

In [ ]:
# ---- Plot LDC (dashed) and net LDC (solid) for each climatic year, target year ----

climatic_years = [c for c in ldc[plot_year].columns if c != "Avg_WS"]

palette = pc.qualitative.Plotly  # or any other qualitative palette
color_map = {cy: palette[i % len(palette)] for i, cy in enumerate(climatic_years)}

fig = go.Figure()

for cy in climatic_years:
    color = color_map[cy]

    # Gross LDC (dashed)
    fig.add_trace(
        go.Scatter(
            x=ldc[plot_year].index,
            y=ldc[plot_year][cy] / 1e3,
            mode="lines",
            name=f"{cy} — gross",
            line=dict(color=color, dash="dot"),
            legendgroup=cy,
        )
    )

    # Net LDC (solid)
    fig.add_trace(
        go.Scatter(
            x=net_ldc[plot_year].index,
            y=net_ldc[plot_year][cy] / 1e3,
            mode="lines",
            name=f"{cy} — net",
            line=dict(color=color, dash="solid"),
            legendgroup=cy,
        )
    )

fig.update_layout(
    title=f"Gross vs Net Load Duration Curves — {plot_year} - {country_choice}",
    xaxis_title="Hours",
    yaxis_title="Demand (GW)",
    template="plotly_white",
    hovermode="x unified",
    legend_title="Climatic year",
    width=1100,
    height=600,
)

fig.show()

Questions:
- Does Hydro ROR contribute to the suecurity of supply equally during all year?
- Is its contribution volatile in respect to the climatic years considered?

***From now on, we will consider that hydro ROR is fatal, and work based on the demand, net of hydro ROR generation***

#### Integrating effect of solar PV

What happens if you start adding more and more solar on thr system? Let's investigate that question through the prism of net load duration curve

In [ ]:
# ============================================================
# 0. Setup: choose target year / climatic year, and solar capacities to test
# ============================================================
target_year = 2033          # e.g. 2025 or 2033
target_ws = "WS1982"             # pick the climatic year (column) to analyze
solar_pv_capacities = np.arange(0, 52500, 2500)  # MW, 0 to 50000 GW... wait, 50 GW, step 2.5 GW
zoom_hours_low = 0 # low boundary for scarcity hours to zoom in for analysis
zoom_hours_high = 100 # downward boudary for scarciest hours to zoom in for analysis

In [ ]:
# ============================================================
# 1. Load solar PV capacity factors, build hourly matrix aligned with net_demand
# ============================================================
res_capacity_folder = "data/ERAA_2023-2/res_capa-factors"
solar_file = f"{res_capacity_folder}/capa_factor_solar_pv_{target_year}_{country_choice}.csv"

df_solar = pd.read_csv(solar_file, sep=";")
df_solar["date"] = pd.to_datetime(df_solar["date"])
df_solar["value"] = pd.to_numeric(df_solar["value"], errors="coerce")

solar_cf = df_solar.pivot_table(index="date", columns="climatic_year", values="value").sort_index()
solar_cf.columns = [f"WS{cy}" for cy in solar_cf.columns]  # match net_demand column naming


# ============================================================
# 2. Compute net demand with increasing solar PV, for the target climatic year
# ============================================================
base_net_demand = net_demand[target_year][target_ws]  # Series, MW, indexed on target_year real dates

net_demands = {}
for capa in solar_pv_capacities:
    net_demands[capa] = pd.Series(
        base_net_demand.values - capa * solar_cf[target_ws].values,
        index=base_net_demand.index
    )


# ============================================================
# 3. Net load duration curves per solar capacity
# ============================================================
net_ldc = {}
for capa in solar_pv_capacities:
    df = net_demands[capa].copy()
    net_ldc[capa] = df.sort_values(ascending=False).reset_index(drop=True)

fig = go.Figure()
for capa in solar_pv_capacities:
    fig.add_trace(
        go.Scatter(
            x=net_ldc[capa].index,
            y=net_ldc[capa] / 1e3,
            mode="lines",
            name=f"{int(capa/1000)} GW"
        )
    )

fig.update_layout(
    title=f"Impact of Solar PV on Net Load Duration Curves – {target_ws} - {country_choice}",
    xaxis_title="Hours",
    yaxis_title="Net Demand (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600,
    legend_title="Solar Capacity"
)
fig.show()

print(
    "We can see that the impact is not the same on the hours with lowest and highest consumption! "
    "Indeed, there usually is a significant correlation between solar production, the weather, and the demand!\n"
    "Let's push the analysis a step further to clearly observe the marginal impacts of adding solar PV in the mix!"
)


# ============================================================
# 4. Marginal contribution of solar PV
# ============================================================
baseline = net_ldc[0].values
net_ldc_diff = {}
for capa in solar_pv_capacities:
    net_ldc_diff[capa] = (baseline - net_ldc[capa].values) / 1e3

x = np.arange(len(baseline))

fig = go.Figure()
for capa in solar_pv_capacities:
    fig.add_trace(
        go.Scatter(
            x=x,
            y=net_ldc_diff[capa],
            mode="lines",
            name=f"{round(capa/1000, 1)} GW"
        )
    )

fig.update_layout(
    title=f"Marginal Net Load Reduction from Solar – {target_ws} - {country_choice}",
    xaxis_title="Hour Rank (Load Duration Curve)",
    yaxis_title="Net Load Reduction (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600,
    legend_title="Solar Capacity"
)
fig.show()

print(
    "It is even clearer now! How about zooming on a specific period and really highlighting "
    "the marginal contribution of each additional GW of solar on the demand?\n"
    "Advice: let's start with the scarcest hours of the year, which are really the most important "
    "to size the power mix and limit deficit!"
)


# ============================================================
# 5. Average marginal value over the zoomed (scarcest) hours
# ============================================================
avg_ws_reduction_solar_pv = {}
for capa in solar_pv_capacities:
    avg_ws_reduction_solar_pv[capa] = np.mean(net_ldc_diff[capa][zoom_hours_low:zoom_hours_high])

x_vals = [c / 1000 for c in solar_pv_capacities]
y_vals = [avg_ws_reduction_solar_pv[c] for c in solar_pv_capacities]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_vals,
        y=y_vals,
        mode="lines+markers",
        name="Average reduction"
    )
)
fig.update_layout(
    title=(
        f"Value of Solar PV on {target_year} Net Demand - {country_choice} \n"
        f"(Avg reduction over [{zoom_hours_low};{zoom_hours_high}] LDC hours - {target_ws})"
    ),
    xaxis_title="Installed Solar Capacity (GW)",
    yaxis_title="Average Net Load Reduction (GW)",
    template="plotly_white",
    width=900,
    height=500
)
fig.show()


# ============================================================
# 6. Extract the datetimes of the top net-load hours (before/after solar)
# ============================================================
capa_test = [0, 5000, 10000, 15000, 20000]

sorted_with_time = {}
for c in capa_test:
    # Sort WITHOUT resetting index → keep original timestamps
    sorted_with_time[c] = net_demands[c].sort_values(ascending=False)
    top_hours_datetimes = sorted_with_time[c].index[zoom_hours_low:zoom_hours_high]
    top_hours_values = sorted_with_time[c].values[zoom_hours_low:zoom_hours_high]

    print("\n============================================")
    print(f"Top {zoom_hours_high - zoom_hours_low} highest net-load hours for {target_ws} (Capa = {c/1000} GW)")
    print("============================================")
    for dt, val in zip(top_hours_datetimes, top_hours_values):
        print(f"{dt}  -->  {val/1e3:.2f} GW")

We can focus on the peakload reduction, which is a core indicator for power system sizing, and assess the marginal contribution of solar PV across all climatic years considered:

In [ ]:
# ============================================================
# Value of Solar PV on net demand — all climatic years, for a given target_year
# ============================================================

climatic_years_available = [
    cy for cy in net_demand[target_year].keys() if cy in solar_cf.columns
]

avg_reduction_by_cy = {}  # avg_reduction_by_cy[cy][capa] = avg reduction (GW)

for cy in climatic_years_available:
    base_net_demand = net_demand[target_year][cy]

    net_ldc_cy = {}
    for capa in solar_pv_capacities:
        net_demand_capa = base_net_demand.values - capa * solar_cf[cy].values
        net_ldc_cy[capa] = np.sort(net_demand_capa)[::-1]  # descending, like the LDC

    baseline = net_ldc_cy[0]
    avg_reduction_by_cy[cy] = {}
    for capa in solar_pv_capacities:
        diff = (baseline - net_ldc_cy[capa]) / 1e3  # GW
        avg_reduction_by_cy[cy][capa] = np.mean(diff[zoom_hours_low:zoom_hours_high])


# ---- Plot: one curve per climatic year ----
fig = go.Figure()

for cy in climatic_years_available:
    x_vals = [c / 1000 for c in solar_pv_capacities]
    y_vals = [avg_reduction_by_cy[cy][c] for c in solar_pv_capacities]

    fig.add_trace(
        go.Scatter(
            x=x_vals,
            y=y_vals,
            mode="lines+markers",
            name=cy
        )
    )

fig.update_layout(
    title=(
        f"Value of Solar PV on {target_year} Net Demand — {country_choice} — All Climatic Years "
        f"(Avg reduction over [{zoom_hours_low};{zoom_hours_high}] LDC hours)"
    ),
    xaxis_title="Installed Solar Capacity (GW)",
    yaxis_title="Average Net Load Reduction (GW)",
    template="plotly_white",
    legend_title="Climatic year",
    width=1000,
    height=550
)

fig.show()

Impacts on indicators defined in the Demand Section

In [ ]:
# ---- Compute indicators for all climatic years for 2025 and 2033 ----

# ============================================================
# Step 1: load solar capacity factors for EACH year (2025, 2033, ...)
# ============================================================
res_capacity_folder = "data/ERAA_2023-2/res_capa-factors"

solar_cf_by_year = {}
for year in ERAA_years:
    solar_file = f"{res_capacity_folder}/capa_factor_solar_pv_{year}_{country_choice}.csv"
    df_solar = pd.read_csv(solar_file, sep=";")
    df_solar["date"] = pd.to_datetime(df_solar["date"])
    df_solar["value"] = pd.to_numeric(df_solar["value"], errors="coerce")

    cf = df_solar.pivot_table(index="date", columns="climatic_year", values="value").sort_index()
    cf.columns = [f"WS{cy}" for cy in cf.columns]
    solar_cf_by_year[year] = cf


# ============================================================
# Step 2: compute indicators on NET demand, for each year / capacity / climatic year
# ============================================================
results_net = []

for year in ERAA_years:
    climatic_years_available = [
        cy for cy in net_demand[year].keys() if cy in solar_cf_by_year[year].columns
    ]

    for capa in solar_pv_capacities:
        for cy in climatic_years_available:
            base = net_demand[year][cy]  # MW, indexed on real `year` dates
            solar_gen = capa * solar_cf_by_year[year][cy].values  # MW

            net = pd.Series(
                base.values - solar_gen,
                index=pd.to_datetime(base.index)
            )

            # 1) Total annual net demand (TWh) — MWh sum / 1e6
            total_demand_twh = net.sum() / 1e6

            # 2) Peak net demand (GW) — MW / 1e3
            peak_load = net.max() / 1e3

            # 3) Seasonal variation (GW): spread of monthly averages
            monthly_avg = net.resample("ME").mean()
            seasonal_variation = (monthly_avg.max() - monthly_avg.min()) / 1e3

            # 4) Daily variation (GW): max daily (max-min) spread
            daily_spread = net.resample("D").apply(lambda x: x.max() - x.min())
            daily_variation = daily_spread.max() / 1e3

            results_net.append({
                "Year": year,
                "Climatic_year": cy,
                "Solar_capacity_MW": capa,
                "Total_Demand_TWh": total_demand_twh,
                "Peak_Load_GW": peak_load,
                "Seasonal_Variation_GW": seasonal_variation,
                "Max_Daily_Variation_GW": daily_variation,
            })

results_net_df = pd.DataFrame(results_net)

# ============================================================
# Step 2bis: additional indicators (daily & weekly variation) on NET demand
# ============================================================

results_net = []

for year in ERAA_years:
    climatic_years_available = [
        cy for cy in net_demand[year].keys() if cy in solar_cf_by_year[year].columns
    ]

    for capa in solar_pv_capacities:
        for cy in climatic_years_available:
            base = net_demand[year][cy]
            solar_gen = capa * solar_cf_by_year[year][cy].values

            net = pd.Series(
                base.values - solar_gen,
                index=pd.to_datetime(base.index)
            )

            # 1) Total annual net demand (TWh)
            total_demand_twh = net.sum() / 1e6

            # 2) Peak net demand (GW)
            peak_load = net.max() / 1e3

            # 3) Seasonal variation (GW)
            monthly_avg = net.resample("ME").mean()
            seasonal_variation = (monthly_avg.max() - monthly_avg.min()) / 1e3

            # 4) Max daily variation (GW)
            daily_spread = net.resample("D").apply(lambda x: x.max() - x.min())
            max_daily_variation = daily_spread.max() / 1e3

            # 5) Mean daily variation (GW) — average of daily max-min spread
            mean_daily_variation = daily_spread.mean() / 1e3


            results_net.append({
                "Year": year,
                "Climatic_year": cy,
                "Solar_capacity_MW": capa,
                "Total_Demand_TWh": total_demand_twh,
                "Peak_Load_GW": peak_load,
                "Seasonal_Variation_GW": seasonal_variation,
                "Max_Daily_Variation_GW": max_daily_variation,
                "Mean_Daily_Variation_GW": mean_daily_variation,
            })

results_net_df = pd.DataFrame(results_net)


# ============================================================
# Step 3: aggregate min/max across climatic years, per (Year, Solar_capacity)
# ============================================================
indicators = [
    "Total_Demand_TWh",
    "Peak_Load_GW",
    "Seasonal_Variation_GW",
    "Max_Daily_Variation_GW",
    "Mean_Daily_Variation_GW",
]

agg = (
    results_net_df
    .groupby(["Year", "Solar_capacity_MW"])[indicators]
    .agg(["min", "max"])
)


# ============================================================
# Step 4: plot — one figure per indicator, min-max range band per year
# ============================================================
year_colors = {
    2025: "31, 119, 180",   # RGB triplet as string, for rgba()
    2033: "255, 127, 14",
}

for indicator in indicators:
    fig = go.Figure()

    for year in ERAA_years:
        sub = agg.loc[year, indicator].sort_index()
        x_vals = sub.index / 1000  # MW -> GW

        color_rgb = year_colors.get(year, "100, 100, 100")

        fig.add_trace(go.Scatter(
            x=x_vals, y=sub["min"],
            mode="lines", line=dict(width=0),
            showlegend=False, hoverinfo="skip",
        ))
        fig.add_trace(go.Scatter(
            x=x_vals, y=sub["max"],
            mode="lines", line=dict(width=0),
            fill="tonexty", fillcolor=f"rgba({color_rgb},0.25)",
            name=f"{year} (min–max range)",
            hoverinfo="skip",
        ))

    fig.update_layout(
        title=f"{indicator} vs Solar PV Capacity — Range Across Climatic Years",
        xaxis_title="Installed Solar Capacity (GW)",
        yaxis_title=indicator.replace("_", " "),
        template="plotly_white",
        legend_title="Year",
        width=1000,
        height=500,
    )

    fig.show()

Illustration of the daily variation problematic - The Duck curve effect

Here below a case study for The Netherlands:

<img src="images/duck_curve_nl.png" alt="Duck curve in Netherlands - A rising issue" width="500">

*(Optional) You can try to plot the duck effect for your case study country with varying Solar PV penetration in the mix.*

Questions:
- Does solar power provide equal contribution to power system adequacy across all year?
- How can you explain what looks like a "cap" for peakload reduction?
- Is this cap the same in 2025 and 2033?
- How could we increase the value of solar pV for peakload reduction?
- How does it affect the overall consumption?
- How does it affect the daily and seasonal variations?
- How to mitigate these negative impacts? What are the drawbacks of the solutions ?
- At this point, what would you advise for your country regArding the solar capacity development?

#### Integrating effect of onshore wind

In [ ]:
# ============================================================
# 0. Setup: choose target year / climatic year, and wind onshore capacities to test
# ============================================================
target_year = 2033          # e.g. 2025 or 2033
target_ws = "WS1982"             # pick the climatic year (column) to analyze
wind_onshore_capacities = np.arange(0, 52500, 2500)  # MW, 0 to 50 GW, step 2.5 GW
zoom_hours_low = 0 # low boundary for scarcity hours to zoom in for analysis
zoom_hours_high = 100 # downward boundary for scarciest hours to zoom in for analysis

In [ ]:
# ============================================================
# 1. Load wind onshore capacity factors, build hourly matrix aligned with net_demand
# ============================================================
res_capacity_folder = "data/ERAA_2023-2/res_capa-factors"
wind_onshore_file = f"{res_capacity_folder}/capa_factor_wind_onshore_{target_year}_{country_choice}.csv"
df_wind = pd.read_csv(wind_onshore_file, sep=";")
df_wind["date"] = pd.to_datetime(df_wind["date"])
df_wind["value"] = pd.to_numeric(df_wind["value"], errors="coerce")
wind_cf = df_wind.pivot_table(index="date", columns="climatic_year", values="value").sort_index()
wind_cf.columns = [f"WS{cy}" for cy in wind_cf.columns]  # match net_demand column naming

# ============================================================
# 2. Compute net demand with increasing wind onshore, for the target climatic year
# ============================================================
base_net_demand = net_demand[target_year][target_ws]  # Series, MW, indexed on target_year real dates
net_demands = {}
for capa in wind_onshore_capacities:
    net_demands[capa] = pd.Series(
        base_net_demand.values - capa * wind_cf[target_ws].values,
        index=base_net_demand.index
    )

# ============================================================
# 3. Net load duration curves per wind onshore capacity
# ============================================================
net_ldc = {}
for capa in wind_onshore_capacities:
    df = net_demands[capa].copy()
    net_ldc[capa] = df.sort_values(ascending=False).reset_index(drop=True)

fig = go.Figure()
for capa in wind_onshore_capacities:
    fig.add_trace(
        go.Scatter(
            x=net_ldc[capa].index,
            y=net_ldc[capa] / 1e3,
            mode="lines",
            name=f"{int(capa/1000)} GW"
        )
    )
fig.update_layout(
    title=f"Impact of Wind Onshore on Net Load Duration Curves – {target_ws}",
    xaxis_title="Hours",
    yaxis_title="Net Demand (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600,
    legend_title="Wind Onshore Capacity"
)
fig.show()
print(
    "We can see that the impact is not the same on the hours with lowest and highest consumption! "
    "Indeed, there usually is a significant correlation between wind production, the weather, and the demand!\n"
    "Let's push the analysis a step further to clearly observe the marginal impacts of adding wind onshore in the mix!"
)

# ============================================================
# 4. Marginal contribution of wind onshore
# ============================================================
baseline = net_ldc[0].values
net_ldc_diff = {}
for capa in wind_onshore_capacities:
    net_ldc_diff[capa] = (baseline - net_ldc[capa].values) / 1e3
x = np.arange(len(baseline))
fig = go.Figure()
for capa in wind_onshore_capacities:
    fig.add_trace(
        go.Scatter(
            x=x,
            y=net_ldc_diff[capa],
            mode="lines",
            name=f"{round(capa/1000, 1)} GW"
        )
    )
fig.update_layout(
    title=f"Marginal Net Load Reduction from Wind Onshore – {target_ws}",
    xaxis_title="Hour Rank (Load Duration Curve)",
    yaxis_title="Net Load Reduction (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600,
    legend_title="Wind Onshore Capacity"
)
fig.show()
print(
    "It is even clearer now! How about zooming on a specific period and really highlighting "
    "the marginal contribution of each additional GW of wind onshore on the demand?\n"
    "Advice: let's start with the scarcest hours of the year, which are really the most important "
    "to size the power mix and limit deficit!"
)

# ============================================================
# 5. Average marginal value over the zoomed (scarcest) hours
# ============================================================
avg_ws_reduction_wind_onshore = {}
for capa in wind_onshore_capacities:
    avg_ws_reduction_wind_onshore[capa] = np.mean(net_ldc_diff[capa][zoom_hours_low:zoom_hours_high])
x_vals = [c / 1000 for c in wind_onshore_capacities]
y_vals = [avg_ws_reduction_wind_onshore[c] for c in wind_onshore_capacities]
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_vals,
        y=y_vals,
        mode="lines+markers",
        name="Average reduction"
    )
)
fig.update_layout(
    title=(
        f"Value of Wind Onshore on {target_year} Net Demand "
        f"(Avg reduction over [{zoom_hours_low};{zoom_hours_high}] LDC hours - {target_ws})"
    ),
    xaxis_title="Installed Wind Onshore Capacity (GW)",
    yaxis_title="Average Net Load Reduction (GW)",
    template="plotly_white",
    width=900,
    height=500
)
fig.show()

# ============================================================
# 6. Extract the datetimes of the top net-load hours (before/after wind onshore)
# ============================================================
capa_test = [0, 5000, 10000, 15000, 20000]
sorted_with_time = {}
for c in capa_test:
    # Sort WITHOUT resetting index → keep original timestamps
    sorted_with_time[c] = net_demands[c].sort_values(ascending=False)
    top_hours_datetimes = sorted_with_time[c].index[zoom_hours_low:zoom_hours_high]
    top_hours_values = sorted_with_time[c].values[zoom_hours_low:zoom_hours_high]
    print("\n============================================")
    print(f"Top {zoom_hours_high - zoom_hours_low} highest net-load hours for {target_ws} (Capa = {c/1000} GW)")
    print("============================================")
    for dt, val in zip(top_hours_datetimes, top_hours_values):
        print(f"{dt}  -->  {val/1e3:.2f} GW")

In [ ]:
# ============================================================
# Value of Wind Onshore on net demand — all climatic years, for a given target_year
# ============================================================

climatic_years_available = [
    cy for cy in net_demand[target_year].keys() if cy in wind_cf.columns
]

avg_reduction_by_cy = {}  # avg_reduction_by_cy[cy][capa] = avg reduction (GW)

for cy in climatic_years_available:
    base_net_demand = net_demand[target_year][cy]

    net_ldc_cy = {}
    for capa in wind_onshore_capacities:
        net_demand_capa = base_net_demand.values - capa * wind_cf[cy].values
        net_ldc_cy[capa] = np.sort(net_demand_capa)[::-1]  # descending, like the LDC

    baseline = net_ldc_cy[0]
    avg_reduction_by_cy[cy] = {}
    for capa in wind_onshore_capacities:
        diff = (baseline - net_ldc_cy[capa]) / 1e3  # GW
        avg_reduction_by_cy[cy][capa] = np.mean(diff[zoom_hours_low:zoom_hours_high])


# ---- Plot: one curve per climatic year ----
fig = go.Figure()

for cy in climatic_years_available:
    x_vals = [c / 1000 for c in wind_onshore_capacities]
    y_vals = [avg_reduction_by_cy[cy][c] for c in wind_onshore_capacities]

    fig.add_trace(
        go.Scatter(
            x=x_vals,
            y=y_vals,
            mode="lines+markers",
            name=cy
        )
    )

fig.update_layout(
    title=(
        f"Value of Wind Onshore on {target_year} Net Demand — All Climatic Years "
        f"(Avg reduction over [{zoom_hours_low};{zoom_hours_high}] LDC hours)"
    ),
    xaxis_title="Installed Wind Onshore Capacity (GW)",
    yaxis_title="Average Net Load Reduction (GW)",
    template="plotly_white",
    legend_title="Climatic year",
    width=1000,
    height=550
)

fig.show()

Impacts on indicators defined in the Demand Section

In [ ]:
# ---- Compute indicators for all climatic years for 2025 and 2033 ----
# ============================================================
# Step 1: load wind onshore capacity factors for EACH year (2025, 2033, ...)
# ============================================================
res_capacity_folder = "data/ERAA_2023-2/res_capa-factors"
wind_cf_by_year = {}
for year in ERAA_years:
    wind_onshore_file = f"{res_capacity_folder}/capa_factor_wind_onshore_{year}_{country_choice}.csv"
    df_wind = pd.read_csv(wind_onshore_file, sep=";")
    df_wind["date"] = pd.to_datetime(df_wind["date"])
    df_wind["value"] = pd.to_numeric(df_wind["value"], errors="coerce")
    cf = df_wind.pivot_table(index="date", columns="climatic_year", values="value").sort_index()
    cf.columns = [f"WS{cy}" for cy in cf.columns]
    wind_cf_by_year[year] = cf

# ============================================================
# Step 2: compute indicators on NET demand, for each year / capacity / climatic year
# ============================================================
results_net = []
for year in ERAA_years:
    climatic_years_available = [
        cy for cy in net_demand[year].keys() if cy in wind_cf_by_year[year].columns
    ]
    for capa in wind_onshore_capacities:
        for cy in climatic_years_available:
            base = net_demand[year][cy]  # MW, indexed on real `year` dates
            wind_gen = capa * wind_cf_by_year[year][cy].values  # MW
            net = pd.Series(
                base.values - wind_gen,
                index=pd.to_datetime(base.index)
            )
            # 1) Total annual net demand (TWh) — MWh sum / 1e6
            total_demand_twh = net.sum() / 1e6
            # 2) Peak net demand (GW) — MW / 1e3
            peak_load = net.max() / 1e3
            # 3) Seasonal variation (GW): spread of monthly averages
            monthly_avg = net.resample("ME").mean()
            seasonal_variation = (monthly_avg.max() - monthly_avg.min()) / 1e3
            # 4) Daily variation (GW): max daily (max-min) spread
            daily_spread = net.resample("D").apply(lambda x: x.max() - x.min())
            daily_variation = daily_spread.max() / 1e3
            results_net.append({
                "Year": year,
                "Climatic_year": cy,
                "Wind_Onshore_capacity_MW": capa,
                "Total_Demand_TWh": total_demand_twh,
                "Peak_Load_GW": peak_load,
                "Seasonal_Variation_GW": seasonal_variation,
                "Max_Daily_Variation_GW": daily_variation,
            })
results_net_df = pd.DataFrame(results_net)

# ============================================================
# Step 2bis: additional indicators (daily & weekly variation) on NET demand
# ============================================================
results_net = []
for year in ERAA_years:
    climatic_years_available = [
        cy for cy in net_demand[year].keys() if cy in wind_cf_by_year[year].columns
    ]
    for capa in wind_onshore_capacities:
        for cy in climatic_years_available:
            base = net_demand[year][cy]
            wind_gen = capa * wind_cf_by_year[year][cy].values
            net = pd.Series(
                base.values - wind_gen,
                index=pd.to_datetime(base.index)
            )
            # 1) Total annual net demand (TWh)
            total_demand_twh = net.sum() / 1e6
            # 2) Peak net demand (GW)
            peak_load = net.max() / 1e3
            # 3) Seasonal variation (GW)
            monthly_avg = net.resample("ME").mean()
            seasonal_variation = (monthly_avg.max() - monthly_avg.min()) / 1e3
            # 4) Max daily variation (GW)
            daily_spread = net.resample("D").apply(lambda x: x.max() - x.min())
            max_daily_variation = daily_spread.max() / 1e3
            # 5) Mean daily variation (GW) — average of daily max-min spread
            mean_daily_variation = daily_spread.mean() / 1e3
            results_net.append({
                "Year": year,
                "Climatic_year": cy,
                "Wind_Onshore_capacity_MW": capa,
                "Total_Demand_TWh": total_demand_twh,
                "Peak_Load_GW": peak_load,
                "Seasonal_Variation_GW": seasonal_variation,
                "Max_Daily_Variation_GW": max_daily_variation,
                "Mean_Daily_Variation_GW": mean_daily_variation,
            })
results_net_df = pd.DataFrame(results_net)

# ============================================================
# Step 3: aggregate min/max across climatic years, per (Year, Wind_Onshore_capacity)
# ============================================================
indicators = [
    "Total_Demand_TWh",
    "Peak_Load_GW",
    "Seasonal_Variation_GW",
    "Max_Daily_Variation_GW",
    "Mean_Daily_Variation_GW",
]
agg = (
    results_net_df
    .groupby(["Year", "Wind_Onshore_capacity_MW"])[indicators]
    .agg(["min", "max"])
)

# ============================================================
# Step 4: plot — one figure per indicator, min-max range band per year
# ============================================================
year_colors = {
    2025: "31, 119, 180",   # RGB triplet as string, for rgba()
    2033: "255, 127, 14",
}
for indicator in indicators:
    fig = go.Figure()
    for year in ERAA_years:
        sub = agg.loc[year, indicator].sort_index()
        x_vals = sub.index / 1000  # MW -> GW
        color_rgb = year_colors.get(year, "100, 100, 100")
        fig.add_trace(go.Scatter(
            x=x_vals, y=sub["min"],
            mode="lines", line=dict(width=0),
            showlegend=False, hoverinfo="skip",
        ))
        fig.add_trace(go.Scatter(
            x=x_vals, y=sub["max"],
            mode="lines", line=dict(width=0),
            fill="tonexty", fillcolor=f"rgba({color_rgb},0.25)",
            name=f"{year} (min–max range)",
            hoverinfo="skip",
        ))
    fig.update_layout(
        title=f"{indicator} vs Wind Onshore Capacity — Range Across Climatic Years",
        xaxis_title="Installed Wind Onshore Capacity (GW)",
        yaxis_title=indicator.replace("_", " "),
        template="plotly_white",
        legend_title="Year",
        width=1000,
        height=500,
    )
    fig.show()

Questions:
- What are the main difference in terms of impacts compared to solar PV?
- Why does the marginal impact on peakload "saturates" less then for solar PV?
- Are the contributions to the peakload reduaction varying significantly across weather scenarios?
- What about net demand volatility? What would be the drawbacks of adding large amounts of wind power in the power system?

#### Integrating effect of wind offshore

In [ ]:
# ============================================================
# 0. Setup: choose target year / climatic year, and wind offshore capacities to test
# ============================================================
target_year = 2033
target_ws = "WS1982"
wind_offshore_capacities = np.arange(0, 52500, 2500)  # MW, 0 to 50 GW, step 2.5 GW
zoom_hours_low = 0
zoom_hours_high = 100

In [ ]:
# ============================================================
# 1. Load wind offshore capacity factors, build hourly matrix aligned with net_demand
# ============================================================
res_capacity_folder = "data/ERAA_2023-2/res_capa-factors"
wind_offshore_file = f"{res_capacity_folder}/capa_factor_wind_offshore_{target_year}_{country_choice}.csv"
df_wind_off = pd.read_csv(wind_offshore_file, sep=";")
df_wind_off["date"] = pd.to_datetime(df_wind_off["date"])
df_wind_off["value"] = pd.to_numeric(df_wind_off["value"], errors="coerce")
wind_offshore_cf = df_wind_off.pivot_table(index="date", columns="climatic_year", values="value").sort_index()
wind_offshore_cf.columns = [f"WS{cy}" for cy in wind_offshore_cf.columns]

# ============================================================
# 2. Compute net demand with increasing wind offshore, for the target climatic year
# ============================================================
base_net_demand = net_demand[target_year][target_ws]
net_demands = {}
for capa in wind_offshore_capacities:
    net_demands[capa] = pd.Series(
        base_net_demand.values - capa * wind_offshore_cf[target_ws].values,
        index=base_net_demand.index
    )

# ============================================================
# 3. Net load duration curves per wind offshore capacity
# ============================================================
net_ldc = {}
for capa in wind_offshore_capacities:
    df = net_demands[capa].copy()
    net_ldc[capa] = df.sort_values(ascending=False).reset_index(drop=True)

fig = go.Figure()
for capa in wind_offshore_capacities:
    fig.add_trace(
        go.Scatter(
            x=net_ldc[capa].index,
            y=net_ldc[capa] / 1e3,
            mode="lines",
            name=f"{int(capa/1000)} GW"
        )
    )
fig.update_layout(
    title=f"Impact of Wind Offshore on Net Load Duration Curves – {target_ws}",
    xaxis_title="Hours",
    yaxis_title="Net Demand (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600,
    legend_title="Wind Offshore Capacity"
)
fig.show()
print(
    "We can see that the impact is not the same on the hours with lowest and highest consumption! "
    "Indeed, there usually is a significant correlation between wind production, the weather, and the demand!\n"
    "Let's push the analysis a step further to clearly observe the marginal impacts of adding wind offshore in the mix!"
)

# ============================================================
# 4. Marginal contribution of wind offshore
# ============================================================
baseline = net_ldc[0].values
net_ldc_diff = {}
for capa in wind_offshore_capacities:
    net_ldc_diff[capa] = (baseline - net_ldc[capa].values) / 1e3
x = np.arange(len(baseline))
fig = go.Figure()
for capa in wind_offshore_capacities:
    fig.add_trace(
        go.Scatter(
            x=x,
            y=net_ldc_diff[capa],
            mode="lines",
            name=f"{round(capa/1000, 1)} GW"
        )
    )
fig.update_layout(
    title=f"Marginal Net Load Reduction from Wind Offshore – {target_ws}",
    xaxis_title="Hour Rank (Load Duration Curve)",
    yaxis_title="Net Load Reduction (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600,
    legend_title="Wind Offshore Capacity"
)
fig.show()
print(
    "It is even clearer now! How about zooming on a specific period and really highlighting "
    "the marginal contribution of each additional GW of wind offshore on the demand?\n"
    "Advice: let's start with the scarcest hours of the year, which are really the most important "
    "to size the power mix and limit deficit!"
)

# ============================================================
# 5. Average marginal value over the zoomed (scarcest) hours
# ============================================================
avg_ws_reduction_wind_offshore = {}
for capa in wind_offshore_capacities:
    avg_ws_reduction_wind_offshore[capa] = np.mean(net_ldc_diff[capa][zoom_hours_low:zoom_hours_high])
x_vals = [c / 1000 for c in wind_offshore_capacities]
y_vals = [avg_ws_reduction_wind_offshore[c] for c in wind_offshore_capacities]
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_vals,
        y=y_vals,
        mode="lines+markers",
        name="Average reduction"
    )
)
fig.update_layout(
    title=(
        f"Value of Wind Offshore on {target_year} Net Demand "
        f"(Avg reduction over [{zoom_hours_low};{zoom_hours_high}] LDC hours - {target_ws})"
    ),
    xaxis_title="Installed Wind Offshore Capacity (GW)",
    yaxis_title="Average Net Load Reduction (GW)",
    template="plotly_white",
    width=900,
    height=500
)
fig.show()

# ============================================================
# 6. Extract the datetimes of the top net-load hours (before/after wind offshore)
# ============================================================
capa_test = [0, 5000, 10000, 15000, 20000]
sorted_with_time = {}
for c in capa_test:
    sorted_with_time[c] = net_demands[c].sort_values(ascending=False)
    top_hours_datetimes = sorted_with_time[c].index[zoom_hours_low:zoom_hours_high]
    top_hours_values = sorted_with_time[c].values[zoom_hours_low:zoom_hours_high]
    print("\n============================================")
    print(f"Top {zoom_hours_high - zoom_hours_low} highest net-load hours for {target_ws} (Capa = {c/1000} GW)")
    print("============================================")
    for dt, val in zip(top_hours_datetimes, top_hours_values):
        print(f"{dt}  -->  {val/1e3:.2f} GW")

In [ ]:
# ============================================================
# Value of Wind Offshore on net demand — all climatic years, for a given target_year
# ============================================================

climatic_years_available = [
    cy for cy in net_demand[target_year].keys() if cy in wind_offshore_cf.columns
]

avg_reduction_by_cy = {}

for cy in climatic_years_available:
    base_net_demand = net_demand[target_year][cy]

    net_ldc_cy = {}
    for capa in wind_offshore_capacities:
        net_demand_capa = base_net_demand.values - capa * wind_offshore_cf[cy].values
        net_ldc_cy[capa] = np.sort(net_demand_capa)[::-1]

    baseline = net_ldc_cy[0]
    avg_reduction_by_cy[cy] = {}
    for capa in wind_offshore_capacities:
        diff = (baseline - net_ldc_cy[capa]) / 1e3
        avg_reduction_by_cy[cy][capa] = np.mean(diff[zoom_hours_low:zoom_hours_high])

fig = go.Figure()
for cy in climatic_years_available:
    x_vals = [c / 1000 for c in wind_offshore_capacities]
    y_vals = [avg_reduction_by_cy[cy][c] for c in wind_offshore_capacities]
    fig.add_trace(
        go.Scatter(
            x=x_vals,
            y=y_vals,
            mode="lines+markers",
            name=cy
        )
    )

fig.update_layout(
    title=(
        f"Value of Wind Offshore on {target_year} Net Demand — All Climatic Years "
        f"(Avg reduction over [{zoom_hours_low};{zoom_hours_high}] LDC hours)"
    ),
    xaxis_title="Installed Wind Offshore Capacity (GW)",
    yaxis_title="Average Net Load Reduction (GW)",
    template="plotly_white",
    legend_title="Climatic year",
    width=1000,
    height=550
)
fig.show()

In [ ]:
# ---- Compute indicators for all climatic years for 2025 and 2033 ----
# ============================================================
# Step 1: load wind offshore capacity factors for EACH year (2025, 2033, ...)
# ============================================================
res_capacity_folder = "data/ERAA_2023-2/res_capa-factors"
wind_offshore_cf_by_year = {}
for year in ERAA_years:
    wind_offshore_file = f"{res_capacity_folder}/capa_factor_wind_offshore_{year}_{country_choice}.csv"
    df_wind_off = pd.read_csv(wind_offshore_file, sep=";")
    df_wind_off["date"] = pd.to_datetime(df_wind_off["date"])
    df_wind_off["value"] = pd.to_numeric(df_wind_off["value"], errors="coerce")
    cf = df_wind_off.pivot_table(index="date", columns="climatic_year", values="value").sort_index()
    cf.columns = [f"WS{cy}" for cy in cf.columns]
    wind_offshore_cf_by_year[year] = cf

# ============================================================
# Step 2: compute indicators on NET demand, for each year / capacity / climatic year
# ============================================================
results_net = []
for year in ERAA_years:
    climatic_years_available = [
        cy for cy in net_demand[year].keys() if cy in wind_offshore_cf_by_year[year].columns
    ]
    for capa in wind_offshore_capacities:
        for cy in climatic_years_available:
            base = net_demand[year][cy]
            wind_gen = capa * wind_offshore_cf_by_year[year][cy].values
            net = pd.Series(
                base.values - wind_gen,
                index=pd.to_datetime(base.index)
            )
            total_demand_twh = net.sum() / 1e6
            peak_load = net.max() / 1e3
            monthly_avg = net.resample("ME").mean()
            seasonal_variation = (monthly_avg.max() - monthly_avg.min()) / 1e3
            daily_spread = net.resample("D").apply(lambda x: x.max() - x.min())
            daily_variation = daily_spread.max() / 1e3
            results_net.append({
                "Year": year,
                "Climatic_year": cy,
                "Wind_Offshore_capacity_MW": capa,
                "Total_Demand_TWh": total_demand_twh,
                "Peak_Load_GW": peak_load,
                "Seasonal_Variation_GW": seasonal_variation,
                "Max_Daily_Variation_GW": daily_variation,
            })
results_net_df = pd.DataFrame(results_net)

# ============================================================
# Step 2bis: additional indicators (daily & weekly variation) on NET demand
# ============================================================
results_net = []
for year in ERAA_years:
    climatic_years_available = [
        cy for cy in net_demand[year].keys() if cy in wind_offshore_cf_by_year[year].columns
    ]
    for capa in wind_offshore_capacities:
        for cy in climatic_years_available:
            base = net_demand[year][cy]
            wind_gen = capa * wind_offshore_cf_by_year[year][cy].values
            net = pd.Series(
                base.values - wind_gen,
                index=pd.to_datetime(base.index)
            )
            total_demand_twh = net.sum() / 1e6
            peak_load = net.max() / 1e3
            monthly_avg = net.resample("ME").mean()
            seasonal_variation = (monthly_avg.max() - monthly_avg.min()) / 1e3
            daily_spread = net.resample("D").apply(lambda x: x.max() - x.min())
            max_daily_variation = daily_spread.max() / 1e3
            mean_daily_variation = daily_spread.mean() / 1e3
            results_net.append({
                "Year": year,
                "Climatic_year": cy,
                "Wind_Offshore_capacity_MW": capa,
                "Total_Demand_TWh": total_demand_twh,
                "Peak_Load_GW": peak_load,
                "Seasonal_Variation_GW": seasonal_variation,
                "Max_Daily_Variation_GW": max_daily_variation,
                "Mean_Daily_Variation_GW": mean_daily_variation,
            })
results_net_df = pd.DataFrame(results_net)

# ============================================================
# Step 3: aggregate min/max across climatic years, per (Year, Wind_Offshore_capacity)
# ============================================================
indicators = [
    "Total_Demand_TWh",
    "Peak_Load_GW",
    "Seasonal_Variation_GW",
    "Max_Daily_Variation_GW",
    "Mean_Daily_Variation_GW",
]
agg = (
    results_net_df
    .groupby(["Year", "Wind_Offshore_capacity_MW"])[indicators]
    .agg(["min", "max"])
)

# ============================================================
# Step 4: plot — one figure per indicator, min-max range band per year
# ============================================================
year_colors = {
    2025: "31, 119, 180",
    2033: "255, 127, 14",
}
for indicator in indicators:
    fig = go.Figure()
    for year in ERAA_years:
        sub = agg.loc[year, indicator].sort_index()
        x_vals = sub.index / 1000
        color_rgb = year_colors.get(year, "100, 100, 100")
        fig.add_trace(go.Scatter(
            x=x_vals, y=sub["min"],
            mode="lines", line=dict(width=0),
            showlegend=False, hoverinfo="skip",
        ))
        fig.add_trace(go.Scatter(
            x=x_vals, y=sub["max"],
            mode="lines", line=dict(width=0),
            fill="tonexty", fillcolor=f"rgba({color_rgb},0.25)",
            name=f"{year} (min–max range)",
            hoverinfo="skip",
        ))
    fig.update_layout(
        title=f"{indicator} vs Wind Offshore Capacity — Range Across Climatic Years",
        xaxis_title="Installed Wind Offshore Capacity (GW)",
        yaxis_title=indicator.replace("_", " "),
        template="plotly_white",
        legend_title="Year",
        width=1000,
        height=500,
    )
    fig.show()

Questions:
- Can you spot differences between wind offshore and wind onshore contribution to the power system?
- Could you explain them?
- What do you think are the drawbacks of this technology?

#### Comparative analysis

In [ ]:
# ============================================================
# 0. Setup: choose target year / climatic year, and solar capacities to test
# ============================================================
target_year = 2033          # e.g. 2025 or 2033

zoom_hours_low = 0 # low boundary for scarcity hours to zoom in for analysis
zoom_hours_high = 100 # downward boudary for scarciest hours to zoom in for analysis

In [ ]:
# ============================================================
# Marginal contribution of Solar PV, Wind Onshore, Wind Offshore
# on peak (scarcest) hours net demand reduction — all climatic years
# ============================================================

# --- Common climatic years across net_demand and all three RES capacity factor sets ---
climatic_years_available = [
    cy for cy in net_demand[target_year].keys()
    if cy in solar_cf.columns and cy in wind_cf.columns and cy in wind_offshore_cf.columns
]

# --- Color map: one color per climatic year ---
palette = pc.qualitative.Plotly
color_map = {cy: palette[i % len(palette)] for i, cy in enumerate(climatic_years_available)}

# --- Line style per technology ---
tech_styles = {
    "Solar PV": {"cf": solar_cf, "capacities": solar_pv_capacities, "dash": "solid"},
    "Wind Onshore": {"cf": wind_cf, "capacities": wind_onshore_capacities, "dash": "dash"},
    "Wind Offshore": {"cf": wind_offshore_cf, "capacities": wind_offshore_capacities, "dash": "dot"},
}

# --- Compute average reduction over zoomed (scarcest) hours, per tech / cy / capacity ---
avg_reduction = {}  # avg_reduction[tech][cy][capa] = avg reduction (GW)

for tech, params in tech_styles.items():
    cf = params["cf"]
    capacities = params["capacities"]
    avg_reduction[tech] = {}

    for cy in climatic_years_available:
        base_net_demand = net_demand[target_year][cy]

        net_ldc_cy = {}
        for capa in capacities:
            net_demand_capa = base_net_demand.values - capa * cf[cy].values
            net_ldc_cy[capa] = np.sort(net_demand_capa)[::-1]

        baseline = net_ldc_cy[0]
        avg_reduction[tech][cy] = {}
        for capa in capacities:
            diff = (baseline - net_ldc_cy[capa]) / 1e3  # GW
            avg_reduction[tech][cy][capa] = np.mean(diff[zoom_hours_low:zoom_hours_high])


# --- Plot: one trace per (technology, climatic year), color = cy, dash = tech ---
fig = go.Figure()

for tech, params in tech_styles.items():
    capacities = params["capacities"]
    dash = params["dash"]

    for cy in climatic_years_available:
        x_vals = [c / 1000 for c in capacities]
        y_vals = [avg_reduction[tech][cy][c] for c in capacities]

        fig.add_trace(
            go.Scatter(
                x=x_vals,
                y=y_vals,
                mode="lines",
                name=f"{tech} — {cy}",
                line=dict(color=color_map[cy], dash=dash),
                legendgroup=cy,
            )
        )

fig.update_layout(
    title=(
        f"Marginal Value of Solar PV, Wind Onshore & Wind Offshore on {target_year} Net Demand "
        f"(Avg reduction over [{zoom_hours_low};{zoom_hours_high}] LDC hours)"
    ),
    xaxis_title="Installed Capacity (GW)",
    yaxis_title="Average Net Load Reduction (GW)",
    template="plotly_white",
    legend_title="Technology — Climatic year",
    width=1100,
    height=650
)

fig.show()

Question:
- Are there any positive/negative correlations?
- Do you think contribution are additive? Why?
- To reduce manage peakload reduction, what would you advocate?

### 4.3.3 - Economic and environmental characteristics

#### Economic parameters

In [ ]:
# ---- Load economic data JSON files ----
with open("data/fuel_sources/technology.json", "r") as f:
    tech_data = json.load(f)
with open("data/fuel_sources/fuels.json", "r") as f:
    fuel_data = json.load(f)

In [ ]:
# ---- Compute fixed annual costs ----
fixed_costs = {}
fixed_costs_repartition = {}

for tech, vals in tech_data.items():
    capex = vals.get("CAPEX")
    lifetime = vals.get("Lifetime")
    wacc = vals.get("WACC")
    fom = vals.get("FOM")
    commitable = vals.get("Commitable")
    
    fixed_costs_repartition[tech] = {}
    annualized_investment = vpm(wacc, lifetime, capex)
    fixed_costs[tech] = annualized_investment + fom

    fixed_costs_repartition[tech]["invest_overnight"] = capex/lifetime
    fixed_costs_repartition[tech]["capital"] = annualized_investment - capex/lifetime
    fixed_costs_repartition[tech]["FOM"] = fom

In [ ]:
# Plotting

# Select non-commitable (renewable / must-run) technologies with valid cost data
techs = [
    t for t, v in fixed_costs_repartition.items()
    if v and not tech_data[t]["Commitable"]
]

# Extract each component for filtered technologies
invest_overnight = [fixed_costs_repartition[t]["invest_overnight"] for t in techs]
capital_cost     = [fixed_costs_repartition[t]["capital"]          for t in techs]
fom_cost         = [fixed_costs_repartition[t]["FOM"]             for t in techs]

# Plot Cost Repartition with chosen WACC
fig = go.Figure()

fig.add_trace(go.Bar(
    name="Overnight Investment",
    x=techs,
    y=invest_overnight
))

fig.add_trace(go.Bar(
    name="Capital Cost (annualized - capex)",
    x=techs,
    y=capital_cost
))

fig.add_trace(go.Bar(
    name="FOM",
    x=techs,
    y=fom_cost
))

fig.update_layout(
    barmode="stack",
    title="Annualized fixed cost by technology (renewables)",
    xaxis_title="Technology",
    yaxis_title="Cost (€/MW/year)",
    template="plotly_white",
    width=1100,
    height=600,
    legend_title="Cost Component"
)

fig.show()

How does this compare with thermal technologies?

In [ ]:
# Plotting — all technologies (thermal + renewable), sorted by total cost,
# renewable technology labels in dark green

# Select all technologies with valid cost data, excluding the fictive Deficit
techs = [
    t for t, v in fixed_costs_repartition.items()
    if v and t != "Deficit"
]

# Sort from cheapest to most expensive (total annualized fixed cost)
techs = sorted(techs, key=lambda t: fixed_costs[t])

# Extract each component for filtered technologies (in sorted order)
invest_overnight = [fixed_costs_repartition[t]["invest_overnight"] for t in techs]
capital_cost     = [fixed_costs_repartition[t]["capital"]          for t in techs]
fom_cost         = [fixed_costs_repartition[t]["FOM"]             for t in techs]

# Build colored tick labels: dark green for renewables (non-commitable), default otherwise
ticktext = [
    f'<span style="color:darkgreen">{t}</span>' if not tech_data[t]["Commitable"] else t
    for t in techs
]

# Plot Cost Repartition with chosen WACC
fig = go.Figure()

fig.add_trace(go.Bar(
    name="Overnight Investment",
    x=techs,
    y=invest_overnight
))

fig.add_trace(go.Bar(
    name="Capital Cost (annualized - capex)",
    x=techs,
    y=capital_cost
))

fig.add_trace(go.Bar(
    name="FOM",
    x=techs,
    y=fom_cost
))

fig.update_layout(
    barmode="stack",
    title="Annualized fixed cost by technology (all technologies)",
    xaxis_title="Technology",
    yaxis_title="Cost (€/MW/year)",
    template="plotly_white",
    width=1100,
    height=600,
    legend_title="Cost Component"
)

fig.update_xaxes(
    tickmode="array",
    tickvals=techs,
    ticktext=ticktext
)

fig.show()

Note: Efficiencies are not considered in this graph. To have a fairer comparison, we could divide the costs of Soalr and Wind technologies by the average annual capacity factor for technology (which would multiply the costs by a factor of 2 to 5 ....)

Questions:
- Can you remember which thermal power plants is considered peak/base ?
- As Hydro ROR, Solar PV and Wind Onshore do not have any variable costs, and appear to have lower fixed costs, why not building solely these technologies?
- Can you think of other monetary units that could to some extent represent the real cost of a generation technology for the power system?

*(Optional) Further Investigations*: A commonly used indicator is the **Levelized Cost of Energy (LCOE)**. Eventhough it has a lot of limitations, and do not provide a good understanding of the value an asset has for the system, it enables to have an order of magnitude of the compeitivity of a technology (thermal or vRES). It can be intepreted as the minimal price the energy generated by an asset must be sold to over its operational lifetime for the project to be afforable.
If you want to have a quick look on the LCOE in your country for different techologies, the IEA provided a usefull open data LCOE calculator you can access here: https://www.iea.org/data-and-statistics/data-tools/levelised-cost-of-electricity-calculator

*(Optional) Questions*:
- *Are the WACCs the same for each technology by default in data provided by ERAA?*
- *What could influence the WACC value?*
- *Could you justify having a different WACC then your neighbors?*

What if the WACC changes?

In [ ]:
# ============================================================
# Sensitivity of annualized fixed cost to WACC — renewable technologies
# ============================================================

wacc_range = np.linspace(0.04, 0.15, 11)

renewable_techs = [t for t in tech_data if not tech_data[t]["Commitable"] and t != "Deficit"]

fig = go.Figure()

for tech in renewable_techs:
    capex = tech_data[tech]["CAPEX"]
    lifetime = tech_data[tech]["Lifetime"]
    fom = tech_data[tech]["FOM"]

    costs = [vpm(w, lifetime, capex) + fom for w in wacc_range]

    fig.add_trace(
        go.Scatter(
            x=wacc_range * 100,  # as percentage
            y=costs,
            mode="lines",
            name=tech
        )
    )

fig.update_layout(
    title="Sensitivity of Annualized Fixed Cost to WACC — Renewable Technologies",
    xaxis_title="WACC (%)",
    yaxis_title="Annualized Fixed Cost (€/MW/year)",
    template="plotly_white",
    hovermode="x unified",
    width=1000,
    height=600,
    legend_title="Technology"
)

fig.show()

#### Environmental considerations

If you want to investigate beyond-GHG impacts, comparative LCAs may be a good source of information. Here is on done by the United Nations Economic Commission for Europe: https://unece.org/sites/default/files/2022-04/LCA_3_FINAL%20March%202022.pdf

Categories of impact usually assessed:

<img src="images/lca_categories.png" alt="LCA - Categories of impacts" width="500">


# 5 - Storage

As you previously witnessed, flexibility is quite valuable for balancing the system while being economically efficient. In that regard, storage technologies help:
- Limit peak demands by charging during unstressed hours (*load shifting* and *peak shaving*)
- Mitigate the undesired structural variability increases due to renewable penetration (*e.g. duck curve*)

**Note**: Storages have other values for the power system (e.g. providing balancing services), but these are out of scope in this practical session
 

Storage technologies can be:
- **Chemical**: ammonia, hydrogen, synthetic fuels, methanol, ...
- **Electrochemical**: Batteries, hybrid supercapacitors, ...
- **Electrical**: Supercapacitors, SMES, ...
- **Mechanical**: Pumped Hydro (and other gravity based storages), Air compression, flywheels, ...
- **Thermal**

As all technologies are not mature enough, or not economically competitive yet, or very complex to model, we are only going to consider in this partical session Battery Energy Storage Systems (BESS, typically Li-On Battery), and pumped hydro.

**Hydro water management is very complex to model**, but can be structural for several countries. For this reason, we intergate it, but you will not be able for now to use it. Also, as the capacity is limited, you will only use ERAA data for installed pumped hydro capacity within PyPSA. The expected overall effect of pumped hydro is basically to provide some seasonal flexibility, and charge during low net demand periods of the year (typically in summers in the past, but this may evolve), and discharge during high net demand periods of the year (typically during winter).

**As for batteries, their management is less complex.** We are going to model them here in a very simple way. Batteries provide short term flexibility. They are expected to charge during low net demand hours of the day, and discharge during high net demand periods. 

## 5.1 - Batteries

We model battery as intraday load shifting assets with loss (due to efficency). This simplifies the funcitoning of batteries, as they can in practice load shift over several days if needed, and often use their capacity to participate to balancing services. But this simplified model help understand the the basic impact of batteries

In [ ]:
# Battery Characteristics 

bat_eff = 0.81 # Round trip efficiency
bat_duration = 2 # hours, gives the nominal energy in MWh for a given nominal power in MW
cycle_day = 1 # Number of days between two charge/discharge cycles

### 5.1.1 - Impact without vRES

In [ ]:
target_year = 2033
target_ws = "WS1989"

In [ ]:
# ---- Simple intraday battery load-shifting model ----

def _water_fill(values, energy_budget, power_cap, mode):
    """
    Distribute `energy_budget` (MWh) across `values` (MW, one point per hour),
    never exceeding `power_cap` (MW) at any single hour, by leveling towards
    a common threshold starting from the extreme (lowest for charge, highest
    for discharge).

    mode="charge"   : raises the lowest values first (valley filling)
    mode="discharge": lowers the highest values first (peak shaving)
    """
    values = values.copy()
    n = len(values)
    room = np.full(n, power_cap, dtype=float)  # remaining power headroom per hour
    remaining_energy = energy_budget

    while remaining_energy > 1e-9 and room.max() > 1e-9:
        active = room > 1e-9
        if mode == "charge":
            extreme_val = values[active].min()
            at_extreme = active & (values == extreme_val)
            others = values[active & (values > extreme_val)]
            next_level = others.min() if len(others) > 0 else np.inf
            delta_level = next_level - extreme_val
        else:  # discharge
            extreme_val = values[active].max()
            at_extreme = active & (values == extreme_val)
            others = values[active & (values < extreme_val)]
            next_level = others.max() if len(others) > 0 else -np.inf
            delta_level = extreme_val - next_level

        n_at_extreme = at_extreme.sum()
        delta_room = room[at_extreme].min()
        delta_energy = remaining_energy / n_at_extreme

        candidates = [d for d in [delta_level, delta_room, delta_energy] if np.isfinite(d) and d > 0]
        delta = min(candidates) if candidates else 0

        if delta <= 1e-9:
            break  # no more room / no more level to reach

        if mode == "charge":
            values[at_extreme] += delta
        else:
            values[at_extreme] -= delta
        room[at_extreme] -= delta
        remaining_energy -= delta * n_at_extreme

    return values

def apply_battery_shifting(net_demand_series, battery_power_mw,
                            bat_duration=2, bat_eff=0.81, cycle_day=1):
    """
    Intraday (or every `cycle_day` days) battery load-shifting via
    water-filling: charges the lowest-demand hours and discharges the
    highest-demand hours towards a common level, never exceeding the
    battery's nominal power at any hour, until the energy budget is used.

    - Charge energy budget   = battery_power_mw * bat_duration          (MWh)
    - Discharge energy budget = bat_eff * battery_power_mw * bat_duration (MWh)
    - Instantaneous power cap = battery_power_mw for BOTH charge and discharge
      (efficiency losses reduce total usable energy, not the power cap)
    """
    modified = net_demand_series.copy().astype(float)
    hours_per_cycle = 24 * cycle_day
    n_hours = len(modified)

    charge_energy = battery_power_mw * bat_duration
    discharge_energy = bat_eff * battery_power_mw * bat_duration

    for start in range(0, n_hours, hours_per_cycle):
        end = start + hours_per_cycle
        window_idx = modified.index[start:end]

        if len(window_idx) < hours_per_cycle:
            break

        day_values = modified.loc[window_idx].values

        day_values = _water_fill(day_values, charge_energy, battery_power_mw, mode="charge")
        day_values = _water_fill(day_values, discharge_energy, battery_power_mw, mode="discharge")

        modified.loc[window_idx] = day_values

    return modified

In [ ]:
# ============================================================
# Example application: one target year / climatic year, one battery power level
# ============================================================
battery_power_mw = 10000  # MW of battery fleet, adjust as you want

In [ ]:
# ---- Plots ----

base_net_demand = net_demand[target_year][target_ws]
net_demand_with_battery = apply_battery_shifting(
    base_net_demand,
    battery_power_mw=battery_power_mw,
    bat_duration=bat_duration,
    bat_eff=bat_eff,
    cycle_day=cycle_day,
)

# ============================================================
# Plot: net demand time series, with vs without battery (browsable)
# ============================================================

# ============================================================
# Plot: net demand time series, with vs without battery
# ============================================================

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=base_net_demand.index,
        y=base_net_demand,
        mode="lines",
        name="Net demand (no battery)"
    )
)

fig.add_trace(
    go.Scatter(
        x=net_demand_with_battery.index,
        y=net_demand_with_battery,
        mode="lines",
        name=f"Net demand ({battery_power_mw/1e3:.0f} GW battery)"
    )
)

fig.update_layout(
    title=f"Net Demand — With vs Without Battery Storage ({target_ws})",
    xaxis_title="Date",
    yaxis_title="Net Demand (MW)",
    hovermode="x unified",
    template="plotly_white",
    legend_title="Scenario",
    width=1100,
    height=600
)

# Set labels
fig.update_xaxes(
    dtick=None,
    tickformat="%d %b\n%H:%M"
)

fig.show()
# ============================================================
# Plot: net demand duration curve, before vs after battery
# ============================================================
ldc_before = base_net_demand.sort_values(ascending=False).reset_index(drop=True)
ldc_after = net_demand_with_battery.sort_values(ascending=False).reset_index(drop=True)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ldc_before.index, y=ldc_before / 1e3,
    mode="lines", name="Net demand (no battery)",
    line=dict(dash="dot", color="black")
))
fig.add_trace(go.Scatter(
    x=ldc_after.index, y=ldc_after / 1e3,
    mode="lines", name=f"Net demand ({battery_power_mw/1e3:.0f} GW battery)",
    line=dict(color="royalblue")
))

fig.update_layout(
    title=f"Impact of Battery Storage on Net Load Duration Curve – {target_ws}",
    xaxis_title="Hours",
    yaxis_title="Net Demand (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600
)
fig.show()

Questions:
- What is the impact?
- Does it help with seasonal variation? With peakload? With daily variations?
- Do you think it would help mitigate undesired net demand variation increase cause by (i) Solar PV? (ii) Onshore Wind? (iii) Offshore Wind?

Let's further analyze batteries impact to identify when marginal contribution starts to cap

In [ ]:
# ============================================================
# 0. Setup
# ============================================================
battery_capacities = np.arange(0, 55000, 5000)  # MW, 0 to 50 GW, step 5 GW
target_year = 2033
target_ws = "WS1989"
zoom_hours_low = 0
zoom_hours_high = 100

In [ ]:
# ============================================================
# 1. Compute net demand with increasing battery power, for the target climatic year
# ============================================================
base_net_demand = net_demand[target_year][target_ws]

net_demands = {}
for capa in battery_capacities:
    if capa == 0:
        net_demands[capa] = base_net_demand.copy()
    else:
        net_demands[capa] = apply_battery_shifting(
            base_net_demand,
            battery_power_mw=capa,
            bat_duration=bat_duration,
            bat_eff=bat_eff,
            cycle_day=cycle_day,
        )

# ============================================================
# 2. Net load duration curves per battery capacity
# ============================================================
net_ldc = {}
for capa in battery_capacities:
    df = net_demands[capa].copy()
    net_ldc[capa] = df.sort_values(ascending=False).reset_index(drop=True)

fig = go.Figure()
for capa in battery_capacities:
    fig.add_trace(
        go.Scatter(
            x=net_ldc[capa].index,
            y=net_ldc[capa] / 1e3,
            mode="lines",
            name=f"{int(capa/1000)} GW"
        )
    )
fig.update_layout(
    title=f"Impact of Battery Storage on Net Load Duration Curves – {target_ws}",
    xaxis_title="Hours",
    yaxis_title="Net Demand (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600,
    legend_title="Battery Capacity"
)
fig.show()
print(
    "Unlike solar or wind, batteries do not simply subtract generation from demand: "
    "they shift energy from low- to high-demand hours, shaving the peak and filling the valley "
    "of the load duration curve simultaneously — with a slight net energy increase due to round-trip losses.\n"
    "Let's look at the marginal impact of adding battery capacity!"
)

# ============================================================
# 3. Marginal contribution of batteries
# ============================================================
baseline = net_ldc[0].values
net_ldc_diff = {}
for capa in battery_capacities:
    net_ldc_diff[capa] = (baseline - net_ldc[capa].values) / 1e3
x = np.arange(len(baseline))
fig = go.Figure()
for capa in battery_capacities:
    fig.add_trace(
        go.Scatter(
            x=x,
            y=net_ldc_diff[capa],
            mode="lines",
            name=f"{round(capa/1000, 1)} GW"
        )
    )
fig.update_layout(
    title=f"Marginal Net Load Reduction from Batteries – {target_ws}",
    xaxis_title="Hour Rank (Load Duration Curve)",
    yaxis_title="Net Load Reduction (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600,
    legend_title="Battery Capacity"
)
fig.show()
print(
    "Note that the marginal reduction can turn NEGATIVE at the low end of the LDC (the valley-filling hours), "
    "since batteries increase demand there rather than reducing it — this is expected and different from solar/wind."
)

# ============================================================
# 4. Average marginal value over the zoomed (scarcest) hours
# ============================================================
avg_ws_reduction_battery = {}
for capa in battery_capacities:
    avg_ws_reduction_battery[capa] = np.mean(net_ldc_diff[capa][zoom_hours_low:zoom_hours_high])
x_vals = [c / 1000 for c in battery_capacities]
y_vals = [avg_ws_reduction_battery[c] for c in battery_capacities]
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_vals,
        y=y_vals,
        mode="lines+markers",
        name="Average reduction"
    )
)
fig.update_layout(
    title=(
        f"Value of Batteries on {target_year} Net Demand "
        f"(Avg reduction over [{zoom_hours_low};{zoom_hours_high}] LDC hours - {target_ws})"
    ),
    xaxis_title="Battery Capacity (GW)",
    yaxis_title="Average Net Load Reduction (GW)",
    template="plotly_white",
    width=900,
    height=500
)
fig.show()

# ============================================================
# 5. Extract the datetimes of the top net-load hours (before/after batteries)
# ============================================================
capa_test = [0, 10000, 20000, 30000, 40000]
sorted_with_time = {}
for c in capa_test:
    sorted_with_time[c] = net_demands[c].sort_values(ascending=False)
    top_hours_datetimes = sorted_with_time[c].index[zoom_hours_low:zoom_hours_high]
    top_hours_values = sorted_with_time[c].values[zoom_hours_low:zoom_hours_high]
    print("\n============================================")
    print(f"Top {zoom_hours_high - zoom_hours_low} highest net-load hours for {target_ws} (Battery = {c/1000} GW)")
    print("============================================")
    for dt, val in zip(top_hours_datetimes, top_hours_values):
        print(f"{dt}  -->  {val/1e3:.2f} GW")

The decrease in contribution is just explained by the simplicity of the algorithm: we ensure a full cycle. Past some point, when the consumption is flat (or at least cannot become flatter), enforcing a full cycle increases slightly the "peaks".

However it already provide a valuable information on the order of magnitude of the battery capacity that could help the system.

Let's have a look at all climatic years

In [ ]:
# ============================================================
# Value of Batteries on net demand — all climatic years, for a given target_year
# ============================================================

climatic_years_available = list(net_demand[target_year].keys())

avg_reduction_by_cy = {}

for cy in climatic_years_available:
    base = net_demand[target_year][cy]

    net_ldc_cy = {}
    for capa in battery_capacities:
        if capa == 0:
            net_capa = base.values
        else:
            net_capa = apply_battery_shifting(
                base, battery_power_mw=capa,
                bat_duration=bat_duration, bat_eff=bat_eff, cycle_day=cycle_day
            ).values
        net_ldc_cy[capa] = np.sort(net_capa)[::-1]

    baseline = net_ldc_cy[0]
    avg_reduction_by_cy[cy] = {}
    for capa in battery_capacities:
        diff = (baseline - net_ldc_cy[capa]) / 1e3
        avg_reduction_by_cy[cy][capa] = np.mean(diff[zoom_hours_low:zoom_hours_high])

fig = go.Figure()
for cy in climatic_years_available:
    x_vals = [c / 1000 for c in battery_capacities]
    y_vals = [avg_reduction_by_cy[cy][c] for c in battery_capacities]
    fig.add_trace(
        go.Scatter(
            x=x_vals,
            y=y_vals,
            mode="lines+markers",
            name=cy
        )
    )

fig.update_layout(
    title=(
        f"Value of Batteries on {target_year} Net Demand — All Climatic Years "
        f"(Avg reduction over [{zoom_hours_low};{zoom_hours_high}] LDC hours)"
    ),
    xaxis_title="Battery Capacity (GW)",
    yaxis_title="Average Net Load Reduction (GW)",
    template="plotly_white",
    legend_title="Climatic year",
    width=1000,
    height=550
)
fig.show()

### 5.1.2 - Impact with vRES

We witnessed that solar PV and wind technologies add some volatility. In particular at the daily horizon, maybe this can increase the value a battery have in a power system with high penetration of vRES?

Based on the previous section, choose solar pV, wind onshore/offshore capacities, and we will re-run the analysis to reassess the value of batteries in such system

In [ ]:
# ============================================================
# 0. Chosen renewable capacities for this section
# ============================================================
capa_solar_pv = 15000 # in MW
capa_wind_onshore = 12000 # in MW
capa_wind_offshore = 8000 # in MW

In [ ]:
# ---- Build net_demand_res = net_demand (hydro ROR already subtracted) ----
#    minus solar PV, wind onshore, wind offshore at the chosen capacities
# ============================================================
# Reuses solar_cf_by_year, wind_cf_by_year, wind_offshore_cf_by_year
# already built in the earlier "system indicators" sections.

net_demand_res = {}

for year in ERAA_years:
    net_demand_res[year] = {}

    climatic_years_available = [
        cy for cy in net_demand[year].keys()
        if cy in solar_cf_by_year[year].columns
        and cy in wind_cf_by_year[year].columns
        and cy in wind_offshore_cf_by_year[year].columns
    ]

    for cy in climatic_years_available:
        base = net_demand[year][cy]  # MW, hydro ROR already subtracted

        res_generation = (
            capa_solar_pv * solar_cf_by_year[year][cy].values
            + capa_wind_onshore * wind_cf_by_year[year][cy].values
            + capa_wind_offshore * wind_offshore_cf_by_year[year][cy].values
        )

        net_demand_res[year][cy] = pd.Series(
            base.values - res_generation,
            index=pd.to_datetime(base.index)
        )

In [ ]:
# ============================================================
# 2. Compute net_demand_res + increasing battery power, for the target climatic year
# ============================================================
base_net_demand_res = net_demand_res[target_year][target_ws]

net_demands_res = {}
for capa in battery_capacities:
    if capa == 0:
        net_demands_res[capa] = base_net_demand_res.copy()
    else:
        net_demands_res[capa] = apply_battery_shifting(
            base_net_demand_res,
            battery_power_mw=capa,
            bat_duration=bat_duration,
            bat_eff=bat_eff,
            cycle_day=cycle_day,
        )

# ============================================================
# 3. Net (RES-adjusted) load duration curves per battery capacity
# ============================================================
net_ldc_res = {}
for capa in battery_capacities:
    df = net_demands_res[capa].copy()
    net_ldc_res[capa] = df.sort_values(ascending=False).reset_index(drop=True)

fig = go.Figure()
for capa in battery_capacities:
    fig.add_trace(
        go.Scatter(
            x=net_ldc_res[capa].index,
            y=net_ldc_res[capa] / 1e3,
            mode="lines",
            name=f"{int(capa/1000)} GW"
        )
    )
fig.update_layout(
    title=(
        f"Impact of Battery Storage on Net Load Duration Curves (Solar {capa_solar_pv/1e3:.0f} GW, "
        f"Wind Onshore {capa_wind_onshore/1e3:.0f} GW, Wind Offshore {capa_wind_offshore/1e3:.0f} GW) – {target_ws}"
    ),
    xaxis_title="Hours",
    yaxis_title="Net Demand (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600,
    legend_title="Battery Capacity"
)
fig.show()

# ============================================================
# 4. Marginal contribution of batteries
# ============================================================
baseline = net_ldc_res[0].values
net_ldc_diff = {}
for capa in battery_capacities:
    net_ldc_diff[capa] = (baseline - net_ldc_res[capa].values) / 1e3
x = np.arange(len(baseline))
fig = go.Figure()
for capa in battery_capacities:
    fig.add_trace(
        go.Scatter(
            x=x,
            y=net_ldc_diff[capa],
            mode="lines",
            name=f"{round(capa/1000, 1)} GW"
        )
    )
fig.update_layout(
    title=f"Marginal Net Load Reduction from Batteries (with RES mix) – {target_ws}",
    xaxis_title="Hour Rank (Load Duration Curve)",
    yaxis_title="Net Load Reduction (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600,
    legend_title="Battery Capacity"
)
fig.show()

# ============================================================
# 5. Average marginal value over the zoomed (scarcest) hours
# ============================================================
avg_ws_reduction_battery_res = {}
for capa in battery_capacities:
    avg_ws_reduction_battery_res[capa] = np.mean(net_ldc_diff[capa][zoom_hours_low:zoom_hours_high])
x_vals = [c / 1000 for c in battery_capacities]
y_vals = [avg_ws_reduction_battery_res[c] for c in battery_capacities]
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_vals,
        y=y_vals,
        mode="lines+markers",
        name="Average reduction"
    )
)
fig.update_layout(
    title=(
        f"Value of Batteries on {target_year} Net Demand (with RES mix) "
        f"(Avg reduction over [{zoom_hours_low};{zoom_hours_high}] LDC hours - {target_ws})"
    ),
    xaxis_title="Battery Capacity (GW)",
    yaxis_title="Average Net Load Reduction (GW)",
    template="plotly_white",
    width=900,
    height=500
)
fig.show()

# ============================================================
# 6. Extract the datetimes of the top net-load hours (before/after batteries)
# ============================================================
capa_test = [0, 10000, 20000, 30000, 40000]
sorted_with_time = {}
for c in capa_test:
    sorted_with_time[c] = net_demands_res[c].sort_values(ascending=False)
    top_hours_datetimes = sorted_with_time[c].index[zoom_hours_low:zoom_hours_high]
    top_hours_values = sorted_with_time[c].values[zoom_hours_low:zoom_hours_high]
    print("\n============================================")
    print(f"Top {zoom_hours_high - zoom_hours_low} highest net-load hours for {target_ws} (Battery = {c/1000} GW)")
    print("============================================")
    for dt, val in zip(top_hours_datetimes, top_hours_values):
        print(f"{dt}  -->  {val/1e3:.2f} GW")

In [ ]:
# ---- Value of Batteries on net_demand_res — all climatic years, for a given target_year ----
# ============================================================

climatic_years_available = list(net_demand_res[target_year].keys())

avg_reduction_by_cy = {}

for cy in climatic_years_available:
    base = net_demand_res[target_year][cy]

    net_ldc_cy = {}
    for capa in battery_capacities:
        if capa == 0:
            net_capa = base.values
        else:
            net_capa = apply_battery_shifting(
                base, battery_power_mw=capa,
                bat_duration=bat_duration, bat_eff=bat_eff, cycle_day=cycle_day
            ).values
        net_ldc_cy[capa] = np.sort(net_capa)[::-1]

    baseline = net_ldc_cy[0]
    avg_reduction_by_cy[cy] = {}
    for capa in battery_capacities:
        diff = (baseline - net_ldc_cy[capa]) / 1e3
        avg_reduction_by_cy[cy][capa] = np.mean(diff[zoom_hours_low:zoom_hours_high])

fig = go.Figure()
for cy in climatic_years_available:
    x_vals = [c / 1000 for c in battery_capacities]
    y_vals = [avg_reduction_by_cy[cy][c] for c in battery_capacities]
    fig.add_trace(
        go.Scatter(
            x=x_vals,
            y=y_vals,
            mode="lines+markers",
            name=cy
        )
    )

fig.update_layout(
    title=(
        f"Value of Batteries on {target_year} Net Demand (with RES mix) — All Climatic Years "
        f"(Avg reduction over [{zoom_hours_low};{zoom_hours_high}] LDC hours)"
    ),
    xaxis_title="Battery Capacity (GW)",
    yaxis_title="Average Net Load Reduction (GW)",
    template="plotly_white",
    legend_title="Climatic year",
    width=1000,
    height=550
)
fig.show()

In [ ]:
# ============================================================
# Step 1-2: compute indicators on net_demand_res + batteries
# ============================================================
results_net = []
for year in ERAA_years:
    climatic_years_available = list(net_demand_res[year].keys())
    for capa in battery_capacities:
        for cy in climatic_years_available:
            base = net_demand_res[year][cy]
            if capa == 0:
                net = base.copy()
            else:
                net = apply_battery_shifting(
                    base, battery_power_mw=capa,
                    bat_duration=bat_duration, bat_eff=bat_eff, cycle_day=cycle_day
                )

            total_demand_twh = net.sum() / 1e6
            peak_load = net.max() / 1e3
            monthly_avg = net.resample("ME").mean()
            seasonal_variation = (monthly_avg.max() - monthly_avg.min()) / 1e3
            daily_spread = net.resample("D").apply(lambda x: x.max() - x.min())
            max_daily_variation = daily_spread.max() / 1e3
            mean_daily_variation = daily_spread.mean() / 1e3

            results_net.append({
                "Year": year,
                "Climatic_year": cy,
                "Battery_capacity_MW": capa,
                "Total_Demand_TWh": total_demand_twh,
                "Peak_Load_GW": peak_load,
                "Seasonal_Variation_GW": seasonal_variation,
                "Max_Daily_Variation_GW": max_daily_variation,
                "Mean_Daily_Variation_GW": mean_daily_variation,
            })
results_net_df = pd.DataFrame(results_net)

# ============================================================
# Step 3: aggregate min/max across climatic years
# ============================================================
indicators = [
    "Total_Demand_TWh",
    "Peak_Load_GW",
    "Seasonal_Variation_GW",
    "Max_Daily_Variation_GW",
    "Mean_Daily_Variation_GW",
]
agg = (
    results_net_df
    .groupby(["Year", "Battery_capacity_MW"])[indicators]
    .agg(["min", "max"])
)

# ============================================================
# Step 4: plot
# ============================================================
year_colors = {
    2025: "31, 119, 180",
    2033: "255, 127, 14",
}
for indicator in indicators:
    fig = go.Figure()
    for year in ERAA_years:
        sub = agg.loc[year, indicator].sort_index()
        x_vals = sub.index / 1000
        color_rgb = year_colors.get(year, "100, 100, 100")
        fig.add_trace(go.Scatter(
            x=x_vals, y=sub["min"],
            mode="lines", line=dict(width=0),
            showlegend=False, hoverinfo="skip",
        ))
        fig.add_trace(go.Scatter(
            x=x_vals, y=sub["max"],
            mode="lines", line=dict(width=0),
            fill="tonexty", fillcolor=f"rgba({color_rgb},0.25)",
            name=f"{year} (min–max range)",
            hoverinfo="skip",
        ))
    fig.update_layout(
        title=f"{indicator} vs Battery Capacity (with RES mix) — Range Across Climatic Years",
        xaxis_title="Battery Capacity (GW)",
        yaxis_title=indicator.replace("_", " "),
        template="plotly_white",
        legend_title="Year",
        width=1000,
        height=500,
    )
    fig.show()

Questions:
- What are your conclusions on the value batteries can bring to the power system?
- Can you expand on the synergies with renewables?
- What impact the teh "battery duration have? Do 3-4h batteries provide more value ?
- What are the limitations of battery in terms of power system contribution?
- What other storage systems could adress these limitations?
- In your study case country, what would be an order of magnitude of batteries you could develop?

To have a point of comparison, you could check IEA's Flexibility 2026 report, and look at the relative capacity of batteries in different countries comapred to the peakload (analysis provided directely in the report). In can range up to 14% in 2024 in UK (max in Europe), and up to 26% in California already! But in most countries it is still a few pct (e.g. Germany: 5% of peakload) (Source: https://www.iea.org/reports/electricity-2026/flexibility)

**Cannibalization effects:** Developing too many batteries could lead to overlap with other flexible assets value, typically hydro power. This is a real concern in prospective studies that cannot be overlooked.

### 5.1.3 Economic parameters

Batteries are getting more affordable, leading to increasing capacity development. However, their cost should not be overlooked in the context of capacity sizing. 


Look at references:
- NREL (https://docs.nlr.gov/docs/fy25osti/93281.pdf): for 4h batteries, they consider in 2030 a range from 200 to 350$/kWh
 and they also propsoe this interesting graph:
<img src="images/battery_costs_NREL.png" alt="Current battery storage costs (NRL report)" width="500">



We can discuss these hypothesis, but we should agree on a price. 300€/kWh?

### 5.1.4 - Environmenatal considerations

Eventhough batteries do not directly emit GHG during operation, they and require critical and rare metals, and are still hard to recycle today. Similarly to renewables, if this is something you want to further invesigate, go check LCAs on batteries.

## 5.2 Pumped Hydro

**POINT D'ATTENTION -> TO BE DISCUSSED**
*Not sure we can model it simply....* -> For the scandinavia, we should discuss at this point how to include pumped hydro
(idea: same as batteries for a limited volume over the year)

*cf proposals at section 7.3*

# 6 - Exploring optimal power mix design

Now you have an idea of the value that can bring vRES and batteries in your system.

You also know that commitable generation technologies can be optimally (*under several hypothesis...*) sized using Boiteux stacking methodology. It is not the case for non controllable technologies.

To continue using this stacking methodology, we will need to choose arbitrarily vRES capacities, as well as battery capacities. It wille nable to compute a new net load duration curve, to which we will apply the Boiteux stacking methodology.

Secondly, we will see what an optimization approach will suggest for optimal cpaacity sizing. The vRES capacity will then either be fixed mannually, via a constraint, or indirectly by fixing a CO2 price level. Both approach are feasible.

## 6.1 - Optimal power mix with vRES deployment target 

In [ ]:
# ============================================================
# 0. Inputs: chosen capacities and target scenario
# ============================================================
capa_solar_pv = 15000        # MW
capa_wind_onshore = 12000    # MW
capa_wind_offshore = 8000    # MW
capa_battery = 10000         # MW (assumed 2h duration, see cost assumptions below)

target_year = 2033           # ERAA projection year (2025 or 2033)
target_ws = "WS1982"         # target climatic year

co2_price = 100 # €/gCO2eq

# Battery cost assumptions (not yet in technology.json — see cost research above)
battery_capex_eur_per_kwh = 300      # €/kWh
battery_duration_h = 4               # hours
battery_lifetime = 15                # years
battery_wacc = 0.06                  # -
battery_fom_pct = 0.025              # fraction of CAPEX (€/kW) per year
battery_vom = 0.0                    # €/MWh

In [ ]:
# 1. Net load = gross demand - hydro RoR - solar PV - wind onshore
#    - wind offshore, then battery load-shifting applied
# ============================================================

# Renewable generation (MW) at the target year / climatic year
res_generation = (
    capa_solar_pv * solar_cf_by_year[target_year][target_ws].values
    + capa_wind_onshore * wind_cf_by_year[target_year][target_ws].values
    + capa_wind_offshore * wind_offshore_cf_by_year[target_year][target_ws].values
)

# net_demand[target_year][target_ws] already has hydro RoR subtracted
base_net_demand = net_demand[target_year][target_ws]

net_demand_no_battery = pd.Series(
    base_net_demand.values - res_generation,
    index=pd.to_datetime(base_net_demand.index)
)

if capa_battery > 0:
    net_load = apply_battery_shifting(
        net_demand_no_battery,
        battery_power_mw=capa_battery,
        bat_duration=battery_duration_h,
        bat_eff=bat_eff,
        cycle_day=cycle_day,
    )
else:
    net_load = net_demand_no_battery.copy()

In [ ]:
# ============================================================
# 2. Plot: gross LDC (dashed) vs net LDC (solid)
# ============================================================

gross_demand = demands[target_year][target_ws]

gross_ldc_series = gross_demand.sort_values(ascending=False).reset_index(drop=True) / 1e3
net_ldc_series = net_load.sort_values(ascending=False).reset_index(drop=True) / 1e3

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=gross_ldc_series.index,
        y=gross_ldc_series,
        mode="lines",
        name="Gross demand LDC",
        line=dict(color="black", dash="dot")
    )
)
fig.add_trace(
    go.Scatter(
        x=net_ldc_series.index,
        y=net_ldc_series,
        mode="lines",
        name="Net LDC (RES + battery)",
        line=dict(color="royalblue")
    )
)

fig.update_layout(
    title=(
        f"Gross vs Net Load Duration Curve — {country_choice.capitalize()} - WS {target_ws} - {target_year}<br>"
        f"<sub>Solar {capa_solar_pv/1e3:.0f} GW, Wind Onshore {capa_wind_onshore/1e3:.0f} GW, "
        f"Wind Offshore {capa_wind_offshore/1e3:.0f} GW, Battery {capa_battery/1e3:.0f} GW</sub>"
    ),
    xaxis_title="Hours",
    yaxis_title="Demand (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600
)
fig.show()

In [ ]:
# 3.0 Recompute merit order inputs with a dedicated, collision-free name
#     (techs, argmin_idx, x_grid may have been overwritten elsewhere
#     in the notebook by other plotting sections)
# ============================================================

commitable_techs = [
    tech for tech, vals in tech_data.items()
    if vals.get("Commitable") and fixed_costs.get(tech) is not None
]

x_grid = np.linspace(0, 8760, 5000)
cost_matrix = np.array([
    fixed_costs[tech] + x_grid * var_costs[tech] for tech in commitable_techs
])
argmin_idx = np.argmin(cost_matrix, axis=0)

# ============================================================
# 3. Boiteux graphical stacking applied to the NET load duration curve
# ============================================================
# Reuses the merit order (argmin_idx, x_grid, techs) already computed
# earlier from the cost curves — this ordering only depends on
# fixed_costs / var_costs, not on the demand curve, so it doesn't need
# to be recomputed for a different RES/battery scenario.

# --- Step 1: merit order (peak -> base) and exact switching hours ---
order_idx = []
breakpoints = []
prev = None
for j, idx in enumerate(argmin_idx):
    if idx != prev:
        order_idx.append(idx)
        breakpoints.append(x_grid[j])
        prev = idx

order_techs = [commitable_techs[i] for i in order_idx]
stacking_order = list(reversed(order_techs))
stacking_breakpoints = list(reversed(breakpoints))

# --- Step 2: read cumulative capacity directly on the NET LDC at each breakpoint ---
demand_curve = net_ldc_series  # GW, indexed by hour rank

def ldc_value_at(hour):
    return np.interp(hour, demand_curve.index, demand_curve.values)

cumulative_capacity = {}
prev_capacity = 0.0
for tech, bp in zip(stacking_order, stacking_breakpoints):
    cap_at_breakpoint = ldc_value_at(bp) if bp > 0 else demand_curve.iloc[0]
    cumulative_capacity[tech] = max(cap_at_breakpoint, prev_capacity)
    prev_capacity = cumulative_capacity[tech]

cumulative_capacity[stacking_order[-1]] = demand_curve.iloc[0]

# --- Step 2bis: own (non-cumulative) capacity to install per technology ---
own_capacity_to_install = {}
prev_cap = 0.0
for tech in stacking_order:
    own_capacity_to_install[tech] = cumulative_capacity[tech] - prev_cap
    prev_cap = cumulative_capacity[tech]

print("Thermal capacity to install per technology (GW):\n")
for tech in stacking_order:
    print(f"  {tech:<15} : {own_capacity_to_install[tech]:,.2f} GW")

# --- Step 3: build the stacked area plot ---
x_hours = demand_curve.index.values
y_demand = demand_curve.values

fig = go.Figure()
prev_layer = np.zeros_like(x_hours, dtype=float)
thermal_generation_gwh = {}  # annual generation per technology (GWh)

for tech in stacking_order:
    layer_top = np.minimum(cumulative_capacity[tech], y_demand)
    layer_top = np.maximum(layer_top, prev_layer)

    own_capacity = layer_top - prev_layer
    thermal_generation_gwh[tech] = np.trapezoid(own_capacity, x_hours)

    fig.add_trace(
        go.Scatter(
            x=x_hours,
            y=layer_top,
            mode="lines",
            line=dict(width=0.5, color=tech_color_map.get(tech, "rgb(100,100,100)")),
            fillcolor=tech_color_map.get(tech, "rgb(100,100,100)"),
            fill="tozeroy" if tech == stacking_order[0] else "tonexty",
            name=tech,
            customdata=own_capacity,
            hovertemplate=(
                f"<b>{tech}</b><br>"
                "Hour: %{x:.0f}<br>"
                "Capacity: %{customdata:.2f} GW<extra></extra>"
            ),
        )
    )
    prev_layer = layer_top

fig.add_trace(
    go.Scatter(
        x=x_hours,
        y=y_demand,
        mode="lines",
        name="Net load duration curve",
        line=dict(color="black", width=2, dash="dot"),
    )
)

fig.update_layout(
    title=(
        f"Boiteux Stacking on Net Load Duration Curve – {country_choice.capitalize()} - "
        f"WS {target_ws} - {target_year}"
    ),
    xaxis_title="Hours",
    yaxis_title="Capacity / Net Demand (GW)",
    template="plotly_white",
    hovermode="x unified",
    width=1100,
    height=600,
)
fig.show()

In [ ]:
# Recompute fixed & variable costs from current tech_data / co2_price
#     (students may have edited technology.json parameters or the CO2
#     price since the earlier cost sections were first run)
# ============================================================

fixed_costs = {}
fixed_costs_repartition = {}

# ---- Load economic data JSON files ----
with open("data/fuel_sources/technology.json", "r") as f:
    tech_data = json.load(f)
with open("data/fuel_sources/fuels.json", "r") as f:
    fuel_data = json.load(f)

for tech, vals in tech_data.items():
    capex = vals.get("CAPEX")
    lifetime = vals.get("Lifetime")
    wacc = vals.get("WACC")
    fom = vals.get("FOM")

    if capex is None or lifetime is None or wacc is None or fom is None:
        fixed_costs[tech] = None
        fixed_costs_repartition[tech] = None
        continue

    fixed_costs_repartition[tech] = {}
    annualized_investment = vpm(wacc, lifetime, capex)
    fixed_costs[tech] = annualized_investment + fom

    fixed_costs_repartition[tech]["invest_overnight"] = capex / lifetime
    fixed_costs_repartition[tech]["capital"] = annualized_investment - capex / lifetime
    fixed_costs_repartition[tech]["FOM"] = fom


var_costs = {}
var_costs_repartition = {}

for tech, vals in tech_data.items():
    fuel_type = vals.get("Fuel type")
    efficiency = vals.get("Efficiency")
    vom = vals.get("VOM")

    if fuel_type is None or efficiency is None or fuel_type not in fuel_data:
        var_costs[tech] = vom
        var_costs_repartition[tech] = {}
        continue

    fuel_info = fuel_data[fuel_type]
    fuel_cost_per_ton = fuel_info["cost_per_ton"]
    fuel_energy_density = fuel_info["energy_density_per_ton"]
    co2_intensity = fuel_info["co2_emissions"] / efficiency

    fuel_cost = fuel_cost_per_ton / fuel_energy_density / efficiency
    co2_cost = co2_price * co2_intensity
    total_var_cost = fuel_cost + co2_cost + vom

    var_costs[tech] = total_var_cost
    var_costs_repartition[tech] = {
        "fuel_cost": fuel_cost,
        "co2_cost": co2_cost,
        "VOM": vom,
    }

# ============================================================
# 4. Total system cost
# ============================================================

# --- Thermal fleet (from Boiteux stacking), excluding the fictive Deficit ---
thermal_bill = 0.0
thermal_breakdown = []
for tech in stacking_order:
    if tech == "Deficit":
        continue
    capacity_mw = own_capacity_to_install[tech] * 1e3       # GW -> MW
    generation_mwh = thermal_generation_gwh[tech] * 1e3     # GWh -> MWh

    fixed_bill = fixed_costs[tech] * capacity_mw
    variable_bill = var_costs[tech] * generation_mwh
    tech_bill = fixed_bill + variable_bill
    thermal_bill += tech_bill

    thermal_breakdown.append({
        "Technology": tech,
        "Capacity (GW)": own_capacity_to_install[tech],
        "Generation (GWh)": thermal_generation_gwh[tech],
        "Total cost (M€/year)": tech_bill / 1e6,
    })

# --- Renewables (fixed cost + VOM if any) ---
res_breakdown = []
res_bill = 0.0

res_specs = {
    "Solar PV": (capa_solar_pv, solar_cf_by_year[target_year][target_ws]),
    "Wind Onshore": (capa_wind_onshore, wind_cf_by_year[target_year][target_ws]),
    "Wind Offshore": (capa_wind_offshore, wind_offshore_cf_by_year[target_year][target_ws]),
}

for tech, (capa_mw, cf) in res_specs.items():
    generation_mwh = capa_mw * cf.sum()  # MW * (sum of hourly capacity factors) = MWh
    fixed_bill = fixed_costs[tech] * capa_mw
    variable_bill = var_costs[tech] * generation_mwh
    tech_bill = fixed_bill + variable_bill
    res_bill += tech_bill

    res_breakdown.append({
        "Technology": tech,
        "Capacity (GW)": capa_mw / 1e3,
        "Generation (GWh)": generation_mwh / 1e3,
        "Total cost (M€/year)": tech_bill / 1e6,
    })

# --- Battery (fixed cost only, assumed VOM = 0) ---
battery_capex_eur_per_mw = battery_capex_eur_per_kwh * 1000 * battery_duration_h  # €/kWh -> €/MW
battery_annualized_investment = vpm(battery_wacc, battery_lifetime, battery_capex_eur_per_mw)
battery_fom_eur_per_mw = battery_fom_pct * battery_capex_eur_per_mw
battery_fixed_cost_per_mw = battery_annualized_investment + battery_fom_eur_per_mw
battery_bill = battery_fixed_cost_per_mw * capa_battery

# --- Combine and display ---
thermal_df = pd.DataFrame(thermal_breakdown)
res_df = pd.DataFrame(res_breakdown)

print("=== Thermal fleet ===")
display(thermal_df.round(2))
print(f"Thermal subtotal: {thermal_bill/1e6:,.1f} M€/year\n")

print("=== Renewables ===")
display(res_df.round(2))
print(f"Renewables subtotal: {res_bill/1e6:,.1f} M€/year\n")

print("=== Battery ===")
print(f"  Capacity          : {capa_battery/1e3:.2f} GW ({battery_duration_h}h duration)")
print(f"  Annualized cost   : {battery_bill/1e6:,.1f} M€/year")
print(f"  (CAPEX assumption : {battery_capex_eur_per_kwh} €/kWh, "
      f"WACC {battery_wacc*100:.0f}%, Lifetime {battery_lifetime} yr, "
      f"FOM {battery_fom_pct*100:.1f}% of CAPEX/kW/yr)\n")

total_system_cost = thermal_bill + res_bill + battery_bill
print(f"TOTAL SYSTEM COST (excl. Hydro RoR & Deficit): {total_system_cost/1e6:,.1f} M€/year")

You can try to change the parameters and see how it affects the thermal mix and the total system cost

## 6.2 - Optimal power mix (wo vRES target)

Previously, you arbitrarily decided on vRES & batteries capacity development. Let's try to optimize the system cost now.

As we are overlooking dynamic constraints, in the previous section, we could just apply Boiteux Stacking methodology to get optimal thermal capacities. Now, since we also want to compute optima vRES and battery capacities, we need to use optimisation problem models. We will only use linear programming in this practical session.

NB: You can still set ambitious environmental goals, either by setting:
- A constraint on the minimal capacity for each technology
- Setting a co2 price

Note: We also could strict contrainsts on CO2 emissions, but as we do not have consolidated in this session on well-to-grave emissions, it is not very relevant

In [ ]:
# pip install pulp # uncomment if not installed yet 
import pulp

In [ ]:
# ============================================================
# 0. Optimization setup: target scenario, CO2 price, cost recompute
# ============================================================

target_year = 2025
target_ws = "WS1982"

# Carbon price, decentivizes use of polluting assets
co2_price = 0  # €/tCO2eq

# Limit on biomass based power generation (limit of stock...)
max_biomass_capa = 3000 # MW
# otherwise, a system woth almost only nuc + biomass a is most of the time the optimal solution

# Battery cost assumptions (not yet in technology.json — see cost research above)
battery_capex_eur_per_kwh = 120     # €/kWh
battery_duration_h = 2               # hours
bat_eff = 0.9                        # %, charge/discharge (one-way) efficiency
battery_lifetime = 15                # years
battery_wacc = 0.06                  # -
battery_fom_pct = 0.025              # fraction of CAPEX (€/kW) per year
battery_vom = 0.0                    # €/MWh

In [ ]:
# --- Recompute fixed & variable costs from the CURRENT tech_data / co2_price ---
# (dedicated names to avoid collisions with other sections of the notebook)
fixed_costs_opt = {}
var_costs_opt = {}

# ---- Load economic data JSON files ----
with open("data/fuel_sources/technology.json", "r") as f:
    tech_data = json.load(f)
with open("data/fuel_sources/fuels.json", "r") as f:
    fuel_data = json.load(f)

for tech, vals in tech_data.items():
    capex = vals.get("CAPEX")
    lifetime = vals.get("Lifetime")
    wacc = vals.get("WACC")
    fom = vals.get("FOM")
    if None in (capex, lifetime, wacc, fom):
        fixed_costs_opt[tech] = None
        continue
    annualized_investment = vpm(wacc, lifetime, capex)
    fixed_costs_opt[tech] = annualized_investment + fom

for tech, vals in tech_data.items():
    fuel_type = vals.get("Fuel type")
    efficiency = vals.get("Efficiency")
    vom = vals.get("VOM")
    if fuel_type is None or efficiency is None or fuel_type not in fuel_data:
        var_costs_opt[tech] = vom
        continue
    fuel_info = fuel_data[fuel_type]
    fuel_cost = fuel_info["cost_per_ton"] / fuel_info["energy_density_per_ton"] / efficiency
    co2_cost = co2_price * fuel_info["co2_emissions"] / efficiency
    var_costs_opt[tech] = fuel_cost + co2_cost + vom

commitable_techs_opt = [
    tech for tech, vals in tech_data.items()
    if vals.get("Commitable") and fixed_costs_opt.get(tech) is not None
]
renewable_techs_opt = ["Solar PV", "Wind Onshore", "Wind Offshore"]

# --- Battery cost (same assumptions as the descriptive section) ---
battery_capex_eur_per_mw = battery_capex_eur_per_kwh * 1000 * battery_duration_h
battery_fixed_cost_opt = vpm(battery_wacc, battery_lifetime, battery_capex_eur_per_mw) \
                          + battery_fom_pct * battery_capex_eur_per_mw

print("NOTE: 'Deficit' is treated here as an ordinary 'commitable'\n"
      "technology, with a very high variable cost (VOM = €20,000/MWh, the Value of\n"
      "Lost Load). The solver will only activate it if no other combination of capacities\n"
      "is less costly to cover demand at that instant — a non-zero Deficit capacity\n"
      "in the results therefore signals a structural need for load shedding / insufficient capacity.")

### 6.2.1 - First approach without additional contraint

In [ ]:
# 1. Build the LP: decision variables, constraints, objective
# ============================================================

# --- Demand to cover: gross demand minus hydro RoR (already computed elsewhere) ---
demand = net_demand[target_year][target_ws]
hours = list(range(len(demand)))
demand_values = demand.values  # MW

# --- Capacity factors for renewables, aligned on the same hours ---
cf = {
    "Solar PV": solar_cf_by_year[target_year][target_ws].values,
    "Wind Onshore": wind_cf_by_year[target_year][target_ws].values,
    "Wind Offshore": wind_offshore_cf_by_year[target_year][target_ws].values,
}

model = pulp.LpProblem("Capacity_Expansion", pulp.LpMinimize)

# --- Capacity decision variables (MW) ---
capacity = {
    tech: pulp.LpVariable(f"capacity_{tech}", lowBound=0)
    for tech in commitable_techs_opt + renewable_techs_opt
}
capacity_battery = pulp.LpVariable("capacity_battery", lowBound=0)  # MW (power)

# --- Hourly dispatch variables (MW) ---
gen = {
    tech: [pulp.LpVariable(f"gen_{tech}_{h}", lowBound=0) for h in hours]
    for tech in commitable_techs_opt
}
gen_res = {
    tech: [pulp.LpVariable(f"gen_res_{tech}_{h}", lowBound=0) for h in hours]
    for tech in renewable_techs_opt
}
charge = [pulp.LpVariable(f"charge_{h}", lowBound=0) for h in hours]
discharge = [pulp.LpVariable(f"discharge_{h}", lowBound=0) for h in hours]
soc = [pulp.LpVariable(f"soc_{h}", lowBound=0) for h in hours]  # state of charge (MWh)

# --- Constraints: max biomass capacity ---
model += capacity["Biomass"] <= max_biomass_capa

# --- Constraints: commitable dispatch capped by installed capacity ---
for tech in commitable_techs_opt:
    for h in hours:
        model += gen[tech][h] <= capacity[tech]

# --- Constraints: renewable dispatch capped by capacity * capacity factor (curtailable) ---
for tech in renewable_techs_opt:
    for h in hours:
        model += gen_res[tech][h] <= capacity[tech] * cf[tech][h]

# --- Constraints: battery power and energy limits ---
for h in hours:
    model += charge[h] <= capacity_battery
    model += discharge[h] <= capacity_battery
    model += soc[h] <= capacity_battery * battery_duration_h

# --- Constraint: battery state-of-charge balance (cyclic: end of year = start of year) ---
for h in hours:
    prev = soc[h - 1] if h > 0 else soc[hours[-1]]
    model += soc[h] == prev + charge[h]*bat_eff - discharge[h]/bat_eff

# --- Constraint: hourly supply-demand balance ---
for h in hours:
    supply = (
        pulp.lpSum(gen[tech][h] for tech in commitable_techs_opt)
        + pulp.lpSum(gen_res[tech][h] for tech in renewable_techs_opt)
        + discharge[h]
        - charge[h]
    )
    model += supply == demand_values[h]

# --- Objective: minimize annualized fixed cost + variable (OPEX) cost ---
fixed_term = pulp.lpSum(
    fixed_costs_opt[tech] * capacity[tech] for tech in commitable_techs_opt + renewable_techs_opt
) + battery_fixed_cost_opt * capacity_battery

variable_term = pulp.lpSum(
    var_costs_opt[tech] * pulp.lpSum(gen[tech][h] for h in hours)
    for tech in commitable_techs_opt
) + pulp.lpSum(
    var_costs_opt[tech] * pulp.lpSum(gen_res[tech][h] for h in hours)
    for tech in renewable_techs_opt
)

model += fixed_term + variable_term

In [ ]:
# 2. Solve
# ============================================================
solver = pulp.PULP_CBC_CMD(msg=True)  # msg=True shows solver progress; set False to silence
model.solve(solver)

print(f"\nSolver status: {pulp.LpStatus[model.status]}")
print(f"Total system cost: {pulp.value(model.objective)/1e6:,.1f} M€/year")

In [ ]:
# ============================================================
# 3. Full description of the optimal system: capacities, generation,
#    and cost breakdown (fixed/CAPEX vs variable/OPEX) by technology
# ============================================================

results_summary = []

for tech in commitable_techs_opt:
    cap = capacity[tech].value()
    generation = sum(gen[tech][h].value() for h in hours)
    fixed_cost = fixed_costs_opt[tech] * cap
    variable_cost = var_costs_opt[tech] * generation
    results_summary.append({
        "Technology": tech,
        "Capacity (MW)": cap,
        "Generation (GWh)": generation / 1e3,
        "Fixed cost (M€/yr)": fixed_cost / 1e6,
        "Variable cost (M€/yr)": variable_cost / 1e6,
        "Total cost (M€/yr)": (fixed_cost + variable_cost) / 1e6,
    })

for tech in renewable_techs_opt:
    cap = capacity[tech].value()
    generation = sum(gen_res[tech][h].value() for h in hours)
    fixed_cost = fixed_costs_opt[tech] * cap
    variable_cost = var_costs_opt[tech] * generation
    results_summary.append({
        "Technology": tech,
        "Capacity (MW)": cap,
        "Generation (GWh)": generation / 1e3,
        "Fixed cost (M€/yr)": fixed_cost / 1e6,
        "Variable cost (M€/yr)": variable_cost / 1e6,
        "Total cost (M€/yr)": (fixed_cost + variable_cost) / 1e6,
    })

battery_cap = capacity_battery.value()
battery_generation = sum(discharge[h].value() * bat_eff for h in hours)
battery_fixed_cost = battery_fixed_cost_opt * battery_cap
results_summary.append({
    "Technology": "Battery",
    "Capacity (MW)": battery_cap,
    "Generation (GWh)": battery_generation / 1e3,
    "Fixed cost (M€/yr)": battery_fixed_cost / 1e6,
    "Variable cost (M€/yr)": 0.0,   # VOM assumed zero, see cost assumptions
    "Total cost (M€/yr)": battery_fixed_cost / 1e6,
})

results_df = pd.DataFrame(results_summary).set_index("Technology")

print("=== Optimal system — full breakdown ===\n")
display(results_df.round(2))

# --- Aggregated CAPEX vs OPEX summary ---
total_fixed = results_df["Fixed cost (M€/yr)"].sum()
total_variable = results_df["Variable cost (M€/yr)"].sum()
total_cost = results_df["Total cost (M€/yr)"].sum()

print(f"\nTotal annualized CAPEX (fixed costs) : {total_fixed:,.1f} M€/year "
      f"({100*total_fixed/total_cost:.1f}%)")
print(f"Total OPEX (variable costs)          : {total_variable:,.1f} M€/year "
      f"({100*total_variable/total_cost:.1f}%)")
print(f"TOTAL SYSTEM COST                     : {total_cost:,.1f} M€/year")

# ============================================================
# 3bis. Total power system emissions (direct/operational only) — model
# ============================================================

emissions_by_tech = {}

for tech in commitable_techs_opt:
    fuel_type = tech_data[tech].get("Fuel type")
    efficiency = tech_data[tech].get("Efficiency")

    if fuel_type is None or efficiency is None or fuel_type not in fuel_data:
        # No fuel (e.g. Deficit) -> zero direct emissions
        emissions_by_tech[tech] = 0.0
        continue

    co2_intensity = fuel_data[fuel_type]["co2_emissions"] / efficiency  # tCO2/MWh_e
    generation = sum(gen[tech][h].value() for h in hours)  # MWh
    emissions_by_tech[tech] = co2_intensity * generation  # tCO2

# Renewables and battery have zero direct operational emissions
for tech in renewable_techs_opt:
    emissions_by_tech[tech] = 0.0
emissions_by_tech["Battery"] = 0.0

total_emissions = sum(emissions_by_tech.values())

print("=== Operational (direct) CO2 emissions by technology (model2) ===\n")
for tech, em in emissions_by_tech.items():
    if em > 0:
        print(f"  {tech:<15} : {em:,.0f} tCO2/year")

print(f"\nTOTAL SYSTEM EMISSIONS (operational only): {total_emissions:,.0f} tCO2/year")

Questions:
- What do you think about this power mix?
- Is it aligned with your vRES development ambitions?
- Is it aligned with EU decarbonization ambitions?
- Based on the power system cost distribution, do you think increasing the CO2 price would help in this framework?
- What other levers exist to strenghten the decarbonization?
- What are this simplistic model limitations?

### 6.2.2 - Constraints on emission reduction

Not happy with the operational carbon emissions of your power system? Let's set for an emmission reduction constraint, and see how it affects the optimal power system

In [ ]:
# ============================================================
# 0bis. CO2 reduction target parameter
# ============================================================
co2_reduction = 0.9  # e.g. 0.2 = -20% vs baseline model's emissions

emissions_cap = (1 - co2_reduction) * total_emissions  # tCO2/year
print(f"Baseline emissions   : {total_emissions:,.0f} tCO2/year")
print(f"Reduction target     : {co2_reduction*100:.0f}%")
print(f"Emissions cap (model1): {emissions_cap:,.0f} tCO2/year")

In [ ]:
# 1. Build model1: same as baseline model, + CO2 emissions cap
# ============================================================

model1 = pulp.LpProblem("Capacity_Expansion_CO2_constrained", pulp.LpMinimize)

# --- Capacity decision variables (MW) ---
capacity1 = {
    tech: pulp.LpVariable(f"capacity1_{tech}", lowBound=0)
    for tech in commitable_techs_opt + renewable_techs_opt
}
capacity_battery1 = pulp.LpVariable("capacity_battery1", lowBound=0)

# --- Hourly dispatch variables (MW) ---
gen1 = {
    tech: [pulp.LpVariable(f"gen1_{tech}_{h}", lowBound=0) for h in hours]
    for tech in commitable_techs_opt
}
gen_res1 = {
    tech: [pulp.LpVariable(f"gen_res1_{tech}_{h}", lowBound=0) for h in hours]
    for tech in renewable_techs_opt
}
charge1 = [pulp.LpVariable(f"charge1_{h}", lowBound=0) for h in hours]
discharge1 = [pulp.LpVariable(f"discharge1_{h}", lowBound=0) for h in hours]
soc1 = [pulp.LpVariable(f"soc1_{h}", lowBound=0) for h in hours]

# --- Constraint: max biomass capacity ---
model1 += capacity1["Biomass"] <= max_biomass_capa

# --- Constraint: commitable dispatch capped by installed capacity ---
for tech in commitable_techs_opt:
    for h in hours:
        model1 += gen1[tech][h] <= capacity1[tech]

# --- Constraint: renewable dispatch capped by capacity * capacity factor (curtailable) ---
for tech in renewable_techs_opt:
    for h in hours:
        model1 += gen_res1[tech][h] <= capacity1[tech] * cf[tech][h]

# --- Constraint: battery power and energy limits ---
for h in hours:
    model1 += charge1[h] <= capacity_battery1
    model1 += discharge1[h] <= capacity_battery1
    model1 += soc1[h] <= capacity_battery1 * battery_duration_h

# --- Constraint: battery state-of-charge balance (cyclic) ---
for h in hours:
    prev = soc1[h - 1] if h > 0 else soc1[hours[-1]]
    model1 += soc1[h] == prev + charge1[h] * bat_eff - discharge1[h] / bat_eff

# --- Constraint: hourly supply-demand balance ---
for h in hours:
    supply = (
        pulp.lpSum(gen1[tech][h] for tech in commitable_techs_opt)
        + pulp.lpSum(gen_res1[tech][h] for tech in renewable_techs_opt)
        + discharge1[h]
        - charge1[h]
    )
    model1 += supply == demand_values[h]

# --- NEW: CO2 emissions cap ---
co2_intensity_by_tech = {}
for tech in commitable_techs_opt:
    fuel_type = tech_data[tech].get("Fuel type")
    efficiency = tech_data[tech].get("Efficiency")
    if fuel_type is None or efficiency is None or fuel_type not in fuel_data:
        co2_intensity_by_tech[tech] = 0.0
    else:
        co2_intensity_by_tech[tech] = fuel_data[fuel_type]["co2_emissions"] / efficiency  # tCO2/MWh

model1 += (
    pulp.lpSum(
        co2_intensity_by_tech[tech] * pulp.lpSum(gen1[tech][h] for h in hours)
        for tech in commitable_techs_opt
    )
    <= emissions_cap,
    "CO2_cap"
)

# --- Objective: minimize annualized fixed cost + variable (OPEX) cost ---
fixed_term1 = pulp.lpSum(
    fixed_costs_opt[tech] * capacity1[tech] for tech in commitable_techs_opt + renewable_techs_opt
) + battery_fixed_cost_opt * capacity_battery1

variable_term1 = pulp.lpSum(
    var_costs_opt[tech] * pulp.lpSum(gen1[tech][h] for h in hours)
    for tech in commitable_techs_opt
) + pulp.lpSum(
    var_costs_opt[tech] * pulp.lpSum(gen_res1[tech][h] for h in hours)
    for tech in renewable_techs_opt
)

model1 += fixed_term1 + variable_term1

In [ ]:
# 2. Solve model1
# ============================================================
solver1 = pulp.PULP_CBC_CMD(msg=True)
model1.solve(solver1)

print(f"\nSolver status: {pulp.LpStatus[model1.status]}")
print(f"Total system cost (model1): {pulp.value(model1.objective)/1e6:,.1f} M€/year")

In [ ]:
# 3. Full description of the optimal system — model1
# ============================================================

results_summary1 = []

for tech in commitable_techs_opt:
    cap = capacity1[tech].value()
    generation = sum(gen1[tech][h].value() for h in hours)
    fixed_cost = fixed_costs_opt[tech] * cap
    variable_cost = var_costs_opt[tech] * generation
    results_summary1.append({
        "Technology": tech,
        "Capacity (MW)": cap,
        "Generation (GWh)": generation / 1e3,
        "Fixed cost (M€/yr)": fixed_cost / 1e6,
        "Variable cost (M€/yr)": variable_cost / 1e6,
        "Total cost (M€/yr)": (fixed_cost + variable_cost) / 1e6,
    })

for tech in renewable_techs_opt:
    cap = capacity1[tech].value()
    generation = sum(gen_res1[tech][h].value() for h in hours)
    fixed_cost = fixed_costs_opt[tech] * cap
    variable_cost = var_costs_opt[tech] * generation
    results_summary1.append({
        "Technology": tech,
        "Capacity (MW)": cap,
        "Generation (GWh)": generation / 1e3,
        "Fixed cost (M€/yr)": fixed_cost / 1e6,
        "Variable cost (M€/yr)": variable_cost / 1e6,
        "Total cost (M€/yr)": (fixed_cost + variable_cost) / 1e6,
    })

battery_cap1 = capacity_battery1.value()
battery_generation1 = sum(discharge1[h].value() * bat_eff for h in hours)
battery_fixed_cost1 = battery_fixed_cost_opt * battery_cap1
results_summary1.append({
    "Technology": "Battery",
    "Capacity (MW)": battery_cap1,
    "Generation (GWh)": battery_generation1 / 1e3,
    "Fixed cost (M€/yr)": battery_fixed_cost1 / 1e6,
    "Variable cost (M€/yr)": 0.0,
    "Total cost (M€/yr)": battery_fixed_cost1 / 1e6,
})

results_df1 = pd.DataFrame(results_summary1).set_index("Technology")

print("=== Optimal system with CO2 cap (model1) — full breakdown ===\n")
display(results_df1.round(2))

total_fixed1 = results_df1["Fixed cost (M€/yr)"].sum()
total_variable1 = results_df1["Variable cost (M€/yr)"].sum()
total_cost1 = results_df1["Total cost (M€/yr)"].sum()

print(f"\nTotal annualized CAPEX (fixed costs) : {total_fixed1:,.1f} M€/year "
      f"({100*total_fixed1/total_cost1:.1f}%)")
print(f"Total OPEX (variable costs)          : {total_variable1:,.1f} M€/year "
      f"({100*total_variable1/total_cost1:.1f}%)")
print(f"TOTAL SYSTEM COST                     : {total_cost1:,.1f} M€/year")

# --- Verify the emissions cap is respected ---
total_emissions1 = sum(
    co2_intensity_by_tech[tech] * sum(gen1[tech][h].value() for h in hours)
    for tech in commitable_techs_opt
)
print(f"\nTOTAL SYSTEM EMISSIONS (model1) : {total_emissions1:,.0f} tCO2/year")
print(f"Emissions cap                    : {emissions_cap:,.0f} tCO2/year")
print(f"Reduction vs baseline achieved   : {100*(1 - total_emissions1/total_emissions):.1f}% "
      f"(target: {co2_reduction*100:.0f}%)")

In [ ]:
# 3bis. Shadow price (dual value) of the CO2 emissions constraint
# ============================================================

co2_shadow_price = model1.constraints["CO2_cap"].pi  # €/tCO2

print(f"\nShadow price of the CO2 constraint : {-co2_shadow_price:,.2f} €/tCO2")

print(
    "\nInterpretation: this is the marginal value of relaxing the emissions cap by one\n"
    "additional tonne of CO2 — i.e. how much the total system cost would decrease\n"
    "(approximately) if the cap were raised by 1 tCO2/year, all else equal. Equivalently,\n"
    "it is the implicit carbon price the emissions constraint is imposing on the system,\n"
    "on top of the explicit CO2 price already included in the variable costs (co2_price).\n"
)

Questions:
- Does an emission reduction target systematically leads to an increase in reneable capacity? Why?
- Did you understand the meaning of the shadow price?
- How does it differ from the co2 price set before?
- Does it increase or decrease with an increase of the co2 price? Why?
- What is the co2 price inputed is 0? How does the emission shadow price evolve with the emmissions reduction target?

#### Plot the shadow price depending on carbon reduction targets

In [ ]:
# 0. Wrap model1 construction + solve into a reusable function
# ============================================================

def solve_co2_constrained_model(co2_reduction_target, verbose=False):
    """
    Build and solve the CO2-constrained capacity expansion model for a given
    reduction target (vs the unconstrained baseline `total_emissions`).
    Returns the shadow price of the CO2 cap constraint (€/tCO2, interpreted
    as an absolute value), or None if the solver did not reach optimality.
    """
    emissions_cap_local = (1 - co2_reduction_target) * total_emissions

    m = pulp.LpProblem("Capacity_Expansion_CO2_constrained", pulp.LpMinimize)

    capacity_l = {
        tech: pulp.LpVariable(f"capacity_{tech}", lowBound=0)
        for tech in commitable_techs_opt + renewable_techs_opt
    }
    capacity_battery_l = pulp.LpVariable("capacity_battery", lowBound=0)

    gen_l = {
        tech: [pulp.LpVariable(f"gen_{tech}_{h}", lowBound=0) for h in hours]
        for tech in commitable_techs_opt
    }
    gen_res_l = {
        tech: [pulp.LpVariable(f"gen_res_{tech}_{h}", lowBound=0) for h in hours]
        for tech in renewable_techs_opt
    }
    charge_l = [pulp.LpVariable(f"charge_{h}", lowBound=0) for h in hours]
    discharge_l = [pulp.LpVariable(f"discharge_{h}", lowBound=0) for h in hours]
    soc_l = [pulp.LpVariable(f"soc_{h}", lowBound=0) for h in hours]

    m += capacity_l["Biomass"] <= max_biomass_capa

    for tech in commitable_techs_opt:
        for h in hours:
            m += gen_l[tech][h] <= capacity_l[tech]

    for tech in renewable_techs_opt:
        for h in hours:
            m += gen_res_l[tech][h] <= capacity_l[tech] * cf[tech][h]

    for h in hours:
        m += charge_l[h] <= capacity_battery_l
        m += discharge_l[h] <= capacity_battery_l
        m += soc_l[h] <= capacity_battery_l * battery_duration_h

    for h in hours:
        prev = soc_l[h - 1] if h > 0 else soc_l[hours[-1]]
        m += soc_l[h] == prev + charge_l[h] * bat_eff - discharge_l[h] / bat_eff

    for h in hours:
        supply = (
            pulp.lpSum(gen_l[tech][h] for tech in commitable_techs_opt)
            + pulp.lpSum(gen_res_l[tech][h] for tech in renewable_techs_opt)
            + discharge_l[h]
            - charge_l[h]
        )
        m += supply == demand_values[h]

    m += (
        pulp.lpSum(
            co2_intensity_by_tech[tech] * pulp.lpSum(gen_l[tech][h] for h in hours)
            for tech in commitable_techs_opt
        )
        <= emissions_cap_local,
        "CO2_cap"
    )

    fixed_term_l = pulp.lpSum(
        fixed_costs_opt[tech] * capacity_l[tech] for tech in commitable_techs_opt + renewable_techs_opt
    ) + battery_fixed_cost_opt * capacity_battery_l

    variable_term_l = pulp.lpSum(
        var_costs_opt[tech] * pulp.lpSum(gen_l[tech][h] for h in hours)
        for tech in commitable_techs_opt
    ) + pulp.lpSum(
        var_costs_opt[tech] * pulp.lpSum(gen_res_l[tech][h] for h in hours)
        for tech in renewable_techs_opt
    )

    m += fixed_term_l + variable_term_l

    solver_l = pulp.PULP_CBC_CMD(msg=verbose)
    m.solve(solver_l)

    status = pulp.LpStatus[m.status]
    if status != "Optimal":
        return {"status": status, "shadow_price": None, "total_cost": None}

    shadow_price = m.constraints["CO2_cap"].pi
    shadow_price = abs(shadow_price) if shadow_price is not None else None

    return {
        "status": status,
        "shadow_price": shadow_price,
        "total_cost": pulp.value(m.objective),
    }

In [ ]:
# 1. Loop over CO2 reduction targets, collect shadow prices
# ============================================================

co2_reduction_targets = np.arange(0.0, 0.91, 0.15)  # 0% to 90%, step 15%

results_sensitivity = []

for target in co2_reduction_targets:
    print(f"Solving for CO2 reduction target = {target*100:.0f}% ...")
    res = solve_co2_constrained_model(target, verbose=False)
    results_sensitivity.append({
        "CO2_reduction_target": target,
        "Status": res["status"],
        "Shadow_price_EUR_per_tCO2": res["shadow_price"],
        "Total_cost_MEUR": res["total_cost"] / 1e6 if res["total_cost"] is not None else None,
    })

sensitivity_df = pd.DataFrame(results_sensitivity)
display(sensitivity_df.round(2))

In [ ]:
# 2. Plot: shadow price vs CO2 reduction target
# ============================================================

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=sensitivity_df["CO2_reduction_target"] * 100,
        y=sensitivity_df["Shadow_price_EUR_per_tCO2"],
        mode="lines+markers",
        name="Shadow price"
    )
)

fig.update_layout(
    title="Marginal Cost of CO2 Abatement — Shadow Price vs Reduction Target",
    xaxis_title="CO2 Reduction Target (%, vs unconstrained baseline)",
    yaxis_title="Shadow Price (€/tCO2)",
    template="plotly_white",
    width=1000,
    height=550
)

fig.show()

### 6.2.3 - Political choices

As you have seen the power mix are for now technology neutral. If for political reason you do not want gas, oil or nuclear for instance, here is the place to add these constraints

Note1: Similarly to what we did before with carbon emission redution targets, you will be able to get the shadow price of your political decisions

Note2: We keep the carbon emmission target if you also want to use it jointly with the policial choices. By default we set it at 0. (the carbon reduction target uses 6.2.1 as a reference)

Note3: The contraints on capapcities must be political, not for logistics or industrial reasons for now.

In [ ]:
# ============================================================
# 0. Capacity bounds per technology (edit these to reflect yur political choices)
# ============================================================
# We use 0 / a very large number as defaults when no real bound applies.
# Keys must match commitable_techs_opt + renewable_techs_opt + "Battery".

DEFAULT_MIN = 0
DEFAULT_MAX = 1e9

min_capacity = {tech: DEFAULT_MIN for tech in commitable_techs_opt + renewable_techs_opt}
min_capacity["Battery"] = DEFAULT_MIN

max_capacity = {tech: DEFAULT_MAX for tech in commitable_techs_opt + renewable_techs_opt}  # effectively unbounded
max_capacity["Battery"] = DEFAULT_MAX

# Example overrides — edit as needed:
max_capacity["Biomass"] = max_biomass_capa
min_capacity["Nuclear"] = 0    # e.g. force at least 30 GW of nuclear (MW)
max_capacity["Nuclear"] = 30000    # e.g. cap nuclear at 63 GW (MW)
min_capacity["Wind Offshore"] =0  # e.g. minimum committed offshore wind (MW)

co2_reduction = 0  # same convention as model1: vs `total_emissions` baseline
emissions_cap = (1 - co2_reduction) * total_emissions



In [ ]:
# ---- Print Scenario Summary ----
print("=== Scenario summary (model3) ===\n")

print(f"CO2 reduction target : {co2_reduction*100:.0f}% (vs 'model' baseline: {total_emissions:,.0f} tCO2/year)")
print(f"  -> Emissions cap    : {emissions_cap:,.0f} tCO2/year\n")

print("Capacity bounds changed from default (0 GW min, unbounded max):\n")

any_changed = False
for tech in list(min_capacity.keys()):
    lo = min_capacity[tech]
    hi = max_capacity[tech]

    lo_changed = lo != DEFAULT_MIN
    hi_changed = hi != DEFAULT_MAX

    if lo_changed or hi_changed:
        any_changed = True
        lo_str = f"{lo/1e3:.1f} GW" if lo_changed else "default (0 GW)"
        hi_str = f"{hi/1e3:.1f} GW" if hi_changed else "default (unbounded)"
        print(f"  {tech:<15} : min = {lo_str:<20} max = {hi_str}")

if not any_changed:
    print("  (none — all technologies use default bounds)")

In [ ]:
# 1. Build model3: CO2 cap + per-technology min/max capacity bounds
# ============================================================

model3 = pulp.LpProblem("Capacity_Expansion_CO2_and_capacity_bounds", pulp.LpMinimize)

# --- Capacity decision variables (MW), with min/max bounds baked in ---
capacity3 = {
    tech: pulp.LpVariable(
        f"capacity3_{tech}",
        lowBound=min_capacity[tech],
        upBound=max_capacity[tech]
    )
    for tech in commitable_techs_opt + renewable_techs_opt
}
capacity_battery3 = pulp.LpVariable(
    "capacity_battery3",
    lowBound=min_capacity["Battery"],
    upBound=max_capacity["Battery"]
)

# --- Hourly dispatch variables (MW) ---
gen3 = {
    tech: [pulp.LpVariable(f"gen3_{tech}_{h}", lowBound=0) for h in hours]
    for tech in commitable_techs_opt
}
gen_res3 = {
    tech: [pulp.LpVariable(f"gen_res3_{tech}_{h}", lowBound=0) for h in hours]
    for tech in renewable_techs_opt
}
charge3 = [pulp.LpVariable(f"charge3_{h}", lowBound=0) for h in hours]
discharge3 = [pulp.LpVariable(f"discharge3_{h}", lowBound=0) for h in hours]
soc3 = [pulp.LpVariable(f"soc3_{h}", lowBound=0) for h in hours]

# --- Constraint: commitable dispatch capped by installed capacity ---
for tech in commitable_techs_opt:
    for h in hours:
        model3 += gen3[tech][h] <= capacity3[tech]

# --- Constraint: renewable dispatch capped by capacity * capacity factor (curtailable) ---
for tech in renewable_techs_opt:
    for h in hours:
        model3 += gen_res3[tech][h] <= capacity3[tech] * cf[tech][h]

# --- Constraint: battery power and energy limits ---
for h in hours:
    model3 += charge3[h] <= capacity_battery3
    model3 += discharge3[h] <= capacity_battery3
    model3 += soc3[h] <= capacity_battery3 * battery_duration_h

# --- Constraint: battery state-of-charge balance (cyclic) ---
for h in hours:
    prev = soc3[h - 1] if h > 0 else soc3[hours[-1]]
    model3 += soc3[h] == prev + charge3[h] * bat_eff - discharge3[h] / bat_eff

# --- Constraint: hourly supply-demand balance ---
for h in hours:
    supply = (
        pulp.lpSum(gen3[tech][h] for tech in commitable_techs_opt)
        + pulp.lpSum(gen_res3[tech][h] for tech in renewable_techs_opt)
        + discharge3[h]
        - charge3[h]
    )
    model3 += supply == demand_values[h]

# --- Constraint: CO2 emissions cap ---
model3 += (
    pulp.lpSum(
        co2_intensity_by_tech[tech] * pulp.lpSum(gen3[tech][h] for h in hours)
        for tech in commitable_techs_opt
    )
    <= emissions_cap,
    "CO2_cap"
)

# --- Objective: minimize annualized fixed cost + variable (OPEX) cost ---
fixed_term3 = pulp.lpSum(
    fixed_costs_opt[tech] * capacity3[tech] for tech in commitable_techs_opt + renewable_techs_opt
) + battery_fixed_cost_opt * capacity_battery3

variable_term3 = pulp.lpSum(
    var_costs_opt[tech] * pulp.lpSum(gen3[tech][h] for h in hours)
    for tech in commitable_techs_opt
) + pulp.lpSum(
    var_costs_opt[tech] * pulp.lpSum(gen_res3[tech][h] for h in hours)
    for tech in renewable_techs_opt
)

model3 += fixed_term3 + variable_term3

In [ ]:
# 2. Solve model3
# ============================================================
solver3 = pulp.PULP_CBC_CMD(msg=True)
model3.solve(solver3)

print(f"\nSolver status: {pulp.LpStatus[model3.status]}")
print(f"Total system cost (model3): {pulp.value(model3.objective)/1e6:,.1f} M€/year")

if pulp.LpStatus[model3.status] != "Optimal":
    print("⚠️ Model3 did not find an optimal solution — check for conflicting "
          "constraints (e.g. min capacity bounds incompatible with the CO2 target).")

In [ ]:
# 3. Full description of the optimal system — model3
# ============================================================

results_summary3 = []

for tech in commitable_techs_opt:
    cap = capacity3[tech].value()
    generation = sum(gen3[tech][h].value() for h in hours)
    fixed_cost = fixed_costs_opt[tech] * cap
    variable_cost = var_costs_opt[tech] * generation
    results_summary3.append({
        "Technology": tech,
        "Capacity (GW)": cap/1e3,
        "Generation (GWh)": generation / 1e3,
        "Fixed cost (M€/yr)": fixed_cost / 1e6,
        "Variable cost (M€/yr)": variable_cost / 1e6,
        "Total cost (M€/yr)": (fixed_cost + variable_cost) / 1e6,
    })

for tech in renewable_techs_opt:
    cap = capacity3[tech].value()
    generation = sum(gen_res3[tech][h].value() for h in hours)
    fixed_cost = fixed_costs_opt[tech] * cap
    variable_cost = var_costs_opt[tech] * generation
    results_summary3.append({
        "Technology": tech,
        "Capacity (GW)": cap/1e3,
        "Generation (GWh)": generation / 1e3,
        "Fixed cost (M€/yr)": fixed_cost / 1e6,
        "Variable cost (M€/yr)": variable_cost / 1e6,
        "Total cost (M€/yr)": (fixed_cost + variable_cost) / 1e6,
    })

battery_cap3 = capacity_battery3.value()
battery_generation3 = sum(discharge3[h].value() * bat_eff for h in hours)
battery_fixed_cost3 = battery_fixed_cost_opt * battery_cap3
results_summary3.append({
    "Technology": "Battery",
    "Capacity (GW)": battery_cap3/1e3,
    "Generation (GWh)": battery_generation3 / 1e3,
    "Fixed cost (M€/yr)": battery_fixed_cost3 / 1e6,
    "Variable cost (M€/yr)": 0.0,
    "Total cost (M€/yr)": battery_fixed_cost3 / 1e6,
})

results_df3 = pd.DataFrame(results_summary3).set_index("Technology")

print("=== Optimal system with CO2 cap + capacity bounds (model3) — full breakdown ===\n")
display(results_df3.round(2))

total_fixed3 = results_df3["Fixed cost (M€/yr)"].sum()
total_variable3 = results_df3["Variable cost (M€/yr)"].sum()
total_cost3 = results_df3["Total cost (M€/yr)"].sum()

print(f"\nTotal annualized CAPEX (fixed costs) : {total_fixed3:,.1f} M€/year "
      f"({100*total_fixed3/total_cost3:.1f}%)")
print(f"Total OPEX (variable costs)          : {total_variable3:,.1f} M€/year "
      f"({100*total_variable3/total_cost3:.1f}%)")
print(f"TOTAL SYSTEM COST                     : {total_cost3:,.1f} M€/year")

# --- Verify emissions cap and capacity bounds are respected ---
total_emissions3 = sum(
    co2_intensity_by_tech[tech] * sum(gen3[tech][h].value() for h in hours)
    for tech in commitable_techs_opt
)
print(f"\nTOTAL SYSTEM EMISSIONS (model3) : {total_emissions3:,.0f} tCO2/year")
print(f"Emissions cap                    : {emissions_cap:,.0f} tCO2/year")

print("\nCapacity bounds check:")
for tech in commitable_techs_opt + renewable_techs_opt:
    print(f"  {tech:<15} : {min_capacity[tech]/1e3:.1f} GW <= "
          f"{capacity3[tech].value()/1e3:.2f} GW <= {max_capacity[tech]/1e3:.1f} GW")
print(f"  {'Battery':<15} : {min_capacity['Battery']/1e3:.1f} GW <= "
      f"{battery_cap3/1e3:.2f} GW <= {max_capacity['Battery']/1e3:.1f} GW")

### 6.2.4 - *(Optional) Additonal (LP) constraints on nuc*

Nuclear power modelling is a bit too optimistic, making it totally flexible. Dynamic constraints could be added, be would require to change the optimisation problem's nature (MILP), a thus increase complexity. Here we juste add rampu-up/down and min power level constraints to observe how it affect the mix

In [ ]:
# 1. Build model2: decision variables, constraints, objective
#    (adds nuclear minimum stable level + ramp constraints)
# ============================================================

demand = net_demand[target_year][target_ws]
hours = list(range(len(demand)))
demand_values = demand.values

cf = {
    "Solar PV": solar_cf_by_year[target_year][target_ws].values,
    "Wind Onshore": wind_cf_by_year[target_year][target_ws].values,
    "Wind Offshore": wind_offshore_cf_by_year[target_year][target_ws].values,
}

# --- Nuclear dynamic constraint parameters (illustrative, adjustable) ---
nuclear_min_stable_pct = 0.5       # minimum output as a fraction of installed capacity
nuclear_ramp_pct_per_hour = 0.05   # max hourly change as a fraction of installed capacity

model2 = pulp.LpProblem("Capacity_Expansion_v2", pulp.LpMinimize)

# --- Capacity decision variables (MW) ---
capacity2 = {
    tech: pulp.LpVariable(f"capacity2_{tech}", lowBound=0)
    for tech in commitable_techs_opt + renewable_techs_opt
}
capacity_battery2 = pulp.LpVariable("capacity_battery2", lowBound=0)

# --- Hourly dispatch variables (MW) ---
gen2 = {
    tech: [pulp.LpVariable(f"gen2_{tech}_{h}", lowBound=0) for h in hours]
    for tech in commitable_techs_opt
}
gen_res2 = {
    tech: [pulp.LpVariable(f"gen_res2_{tech}_{h}", lowBound=0) for h in hours]
    for tech in renewable_techs_opt
}
charge2 = [pulp.LpVariable(f"charge2_{h}", lowBound=0) for h in hours]
discharge2 = [pulp.LpVariable(f"discharge2_{h}", lowBound=0) for h in hours]
soc2 = [pulp.LpVariable(f"soc2_{h}", lowBound=0) for h in hours]

# --- Constraints: max biomass capacity ---
model2 += capacity2["Biomass"] <= max_biomass_capa

# --- Commitable dispatch capped by installed capacity ---
for tech in commitable_techs_opt:
    for h in hours:
        model2 += gen2[tech][h] <= capacity2[tech]

# --- Nuclear-specific: minimum stable level ---
for h in hours:
    model2 += gen2["Nuclear"][h] >= nuclear_min_stable_pct * capacity2["Nuclear"]

# --- Nuclear-specific: ramp constraints ---
for h in hours[1:]:
    model2 += gen2["Nuclear"][h] - gen2["Nuclear"][h - 1] <= nuclear_ramp_pct_per_hour * capacity2["Nuclear"]
    model2 += gen2["Nuclear"][h - 1] - gen2["Nuclear"][h] <= nuclear_ramp_pct_per_hour * capacity2["Nuclear"]

# --- Renewable dispatch capped by capacity * capacity factor (curtailable) ---
for tech in renewable_techs_opt:
    for h in hours:
        model2 += gen_res2[tech][h] <= capacity2[tech] * cf[tech][h]

# --- Battery power and energy limits ---
for h in hours:
    model2 += charge2[h] <= capacity_battery2
    model2 += discharge2[h] <= capacity_battery2
    model2 += soc2[h] <= capacity_battery2 * battery_duration_h

# --- Battery state-of-charge balance (cyclic) ---
for h in hours:
    prev = soc2[h - 1] if h > 0 else soc2[hours[-1]]
    model2 += soc2[h] == prev + charge2[h]*bat_eff - discharge2[h]/bat_eff

# --- Hourly supply-demand balance ---
for h in hours:
    supply = (
        pulp.lpSum(gen2[tech][h] for tech in commitable_techs_opt)
        + pulp.lpSum(gen_res2[tech][h] for tech in renewable_techs_opt)
        + discharge2[h] * bat_eff
        - charge2[h]
    )
    model2 += supply == demand_values[h]

# --- Objective: minimize annualized fixed cost + variable (OPEX) cost ---
fixed_term2 = pulp.lpSum(
    fixed_costs_opt[tech] * capacity2[tech] for tech in commitable_techs_opt + renewable_techs_opt
) + battery_fixed_cost_opt * capacity_battery2

variable_term2 = pulp.lpSum(
    var_costs_opt[tech] * pulp.lpSum(gen2[tech][h] for h in hours)
    for tech in commitable_techs_opt
) + pulp.lpSum(
    var_costs_opt[tech] * pulp.lpSum(gen_res2[tech][h] for h in hours)
    for tech in renewable_techs_opt
)

model2 += fixed_term2 + variable_term2

In [ ]:
# Solve model2
# ============================================================
solver2 = pulp.PULP_CBC_CMD(msg=True)
model2.solve(solver2)

print(f"\nSolver status: {pulp.LpStatus[model2.status]}")
print(f"Total system cost: {pulp.value(model2.objective)/1e6:,.1f} M€/year")

In [ ]:
# 3. Read results — model2
# ============================================================
results_summary2 = []

for tech in commitable_techs_opt:
    cap = capacity2[tech].value()
    generation = sum(gen2[tech][h].value() for h in hours)
    fixed_cost = fixed_costs_opt[tech] * cap
    variable_cost = var_costs_opt[tech] * generation
    results_summary2.append({
        "Technology": tech,
        "Capacity (MW)": cap,
        "Generation (GWh)": generation / 1e3,
        "Fixed cost (M€/yr)": fixed_cost / 1e6,
        "Variable cost (M€/yr)": variable_cost / 1e6,
        "Total cost (M€/yr)": (fixed_cost + variable_cost) / 1e6,
    })

for tech in renewable_techs_opt:
    cap = capacity2[tech].value()
    generation = sum(gen_res2[tech][h].value() for h in hours)
    fixed_cost = fixed_costs_opt[tech] * cap
    variable_cost = var_costs_opt[tech] * generation
    results_summary2.append({
        "Technology": tech,
        "Capacity (MW)": cap,
        "Generation (GWh)": generation / 1e3,
        "Fixed cost (M€/yr)": fixed_cost / 1e6,
        "Variable cost (M€/yr)": variable_cost / 1e6,
        "Total cost (M€/yr)": (fixed_cost + variable_cost) / 1e6,
    })

battery_cap2 = capacity_battery2.value()
battery_generation2 = sum(discharge2[h].value() * bat_eff for h in hours)
battery_fixed_cost2 = battery_fixed_cost_opt * battery_cap2
results_summary2.append({
    "Technology": "Battery",
    "Capacity (MW)": battery_cap2,
    "Generation (GWh)": battery_generation2 / 1e3,
    "Fixed cost (M€/yr)": battery_fixed_cost2 / 1e6,
    "Variable cost (M€/yr)": 0.0,
    "Total cost (M€/yr)": battery_fixed_cost2 / 1e6,
})

results_df2 = pd.DataFrame(results_summary2).set_index("Technology")

print("=== Optimal system (model2: nuclear ramp + min stable level) — full breakdown ===\n")
display(results_df2.round(2))

total_fixed2 = results_df2["Fixed cost (M€/yr)"].sum()
total_variable2 = results_df2["Variable cost (M€/yr)"].sum()
total_cost2 = results_df2["Total cost (M€/yr)"].sum()

print(f"\nTotal annualized CAPEX (fixed costs) : {total_fixed2:,.1f} M€/year "
      f"({100*total_fixed2/total_cost2:.1f}%)")
print(f"Total OPEX (variable costs)          : {total_variable2:,.1f} M€/year "
      f"({100*total_variable2/total_cost2:.1f}%)")
print(f"TOTAL SYSTEM COST                     : {total_cost2:,.1f} M€/year")

# ============================================================
# 3bis. Total power system emissions (direct/operational only) — model2
# ============================================================

emissions_by_tech2 = {}

for tech in commitable_techs_opt:
    fuel_type = tech_data[tech].get("Fuel type")
    efficiency = tech_data[tech].get("Efficiency")

    if fuel_type is None or efficiency is None or fuel_type not in fuel_data:
        # No fuel (e.g. Deficit) -> zero direct emissions
        emissions_by_tech2[tech] = 0.0
        continue

    co2_intensity = fuel_data[fuel_type]["co2_emissions"] / efficiency  # tCO2/MWh_e
    generation = sum(gen2[tech][h].value() for h in hours)  # MWh
    emissions_by_tech2[tech] = co2_intensity * generation  # tCO2

# Renewables and battery have zero direct operational emissions
for tech in renewable_techs_opt:
    emissions_by_tech2[tech] = 0.0
emissions_by_tech2["Battery"] = 0.0

total_emissions2 = sum(emissions_by_tech2.values())

print("=== Operational (direct) CO2 emissions by technology (model2) ===\n")
for tech, em in emissions_by_tech2.items():
    if em > 0:
        print(f"  {tech:<15} : {em:,.0f} tCO2/year")

print(f"\nTOTAL SYSTEM EMISSIONS (operational only): {total_emissions2:,.0f} tCO2/year")

## 6.3 - Choice of a mix considering various climatic years

For now, you built your mix for a single climatic year. If you set the variables you want to use in terms
- Min/Max capacity constraint for each technology (Policy except for Biomass -> physical & environmental)
- CO2 price
- Battery characteristics


Please fill them in below:

In [ ]:
# ============================================================
# 0. Scenario parameters: CO2 price, capacity bounds, battery characteristics
# ============================================================

target_year = 2025  # ERAA projection year (fixed across all climatic years)
co2_price = 80       # EUR/tCO2

# --- Capacity bounds per technology (MW) — edit as needed ---
DEFAULT_MIN = 0
DEFAULT_MAX = 1e9  # effectively unbounded

min_capacity = {tech: DEFAULT_MIN for tech in commitable_techs_opt + renewable_techs_opt}
min_capacity["Battery"] = DEFAULT_MIN

max_capacity = {tech: DEFAULT_MAX for tech in commitable_techs_opt + renewable_techs_opt}
max_capacity["Battery"] = DEFAULT_MAX

# Example overrides — edit as needed:
max_capacity["Biomass"] = max_biomass_capa
max_capacity["Deficit"] = 1e4 # 10 GW should be enough, and it help for the plots readability

# --- Battery characteristics (MW power decision variable; these fix
#     duration / efficiency / cost assumptions used in the optimization) ---
battery_duration_h = 2          # hours
bat_eff = 0.81                  # round-trip efficiency
battery_capex_eur_per_kwh = 300 # EUR/kWh
battery_lifetime = 15           # years
battery_wacc = 0.06
battery_fom_pct = 0.025         # fraction of CAPEX (EUR/kW) per year

In [ ]:
# 1. Recompute fixed & variable costs (technology + battery)
#       from the CURRENT tech_data / co2_price / battery parameters
# ============================================================

fixed_costs_opt = {}
var_costs_opt = {}

for tech, vals in tech_data.items():
    capex = vals.get("CAPEX")
    lifetime = vals.get("Lifetime")
    wacc = vals.get("WACC")
    fom = vals.get("FOM")
    if None in (capex, lifetime, wacc, fom):
        fixed_costs_opt[tech] = None
        continue
    annualized_investment = vpm(wacc, lifetime, capex)
    fixed_costs_opt[tech] = annualized_investment + fom

for tech, vals in tech_data.items():
    fuel_type = vals.get("Fuel type")
    efficiency = vals.get("Efficiency")
    vom = vals.get("VOM")
    if fuel_type is None or efficiency is None or fuel_type not in fuel_data:
        var_costs_opt[tech] = vom
        continue
    fuel_info = fuel_data[fuel_type]
    fuel_cost = fuel_info["cost_per_ton"] / fuel_info["energy_density_per_ton"] / efficiency
    co2_cost = co2_price * fuel_info["co2_emissions"] / efficiency
    var_costs_opt[tech] = fuel_cost + co2_cost + vom

commitable_techs_opt = [
    tech for tech, vals in tech_data.items()
    if vals.get("Commitable") and fixed_costs_opt.get(tech) is not None
]
renewable_techs_opt = ["Solar PV", "Wind Onshore", "Wind Offshore"]

co2_intensity_by_tech = {}
for tech in commitable_techs_opt:
    fuel_type = tech_data[tech].get("Fuel type")
    efficiency = tech_data[tech].get("Efficiency")
    if fuel_type is None or efficiency is None or fuel_type not in fuel_data:
        co2_intensity_by_tech[tech] = 0.0
    else:
        co2_intensity_by_tech[tech] = fuel_data[fuel_type]["co2_emissions"] / efficiency

battery_capex_eur_per_mw = battery_capex_eur_per_kwh * 1000 * battery_duration_h
battery_fixed_cost_opt = vpm(battery_wacc, battery_lifetime, battery_capex_eur_per_mw) \
                          + battery_fom_pct * battery_capex_eur_per_mw

print("Fixed & variable costs recomputed from current tech_data / co2_price / battery parameters.")

In [ ]:
def solve_optimal_mix(target_ws_local, verbose=False):
    demand_local = net_demand[target_year][target_ws_local]
    hours_local = list(range(len(demand_local)))
    demand_values_local = demand_local.values

    cf_local = {
        "Solar PV": solar_cf_by_year[target_year][target_ws_local].values,
        "Wind Onshore": wind_cf_by_year[target_year][target_ws_local].values,
        "Wind Offshore": wind_offshore_cf_by_year[target_year][target_ws_local].values,
    }

    m = pulp.LpProblem(f"Capacity_Expansion_{target_ws_local}", pulp.LpMinimize)

    capacity_l = {
        tech: pulp.LpVariable(f"capacity_{tech}", lowBound=min_capacity[tech], upBound=max_capacity[tech])
        for tech in commitable_techs_opt + renewable_techs_opt
    }
    capacity_battery_l = pulp.LpVariable(
        "capacity_battery", lowBound=min_capacity["Battery"], upBound=max_capacity["Battery"]
    )

    gen_l = {
        tech: [pulp.LpVariable(f"gen_{tech}_{h}", lowBound=0) for h in hours_local]
        for tech in commitable_techs_opt
    }
    gen_res_l = {
        tech: [pulp.LpVariable(f"gen_res_{tech}_{h}", lowBound=0) for h in hours_local]
        for tech in renewable_techs_opt
    }
    charge_l = [pulp.LpVariable(f"charge_{h}", lowBound=0) for h in hours_local]
    discharge_l = [pulp.LpVariable(f"discharge_{h}", lowBound=0) for h in hours_local]
    soc_l = [pulp.LpVariable(f"soc_{h}", lowBound=0) for h in hours_local]

    for tech in commitable_techs_opt:
        for h in hours_local:
            m += gen_l[tech][h] <= capacity_l[tech]

    for tech in renewable_techs_opt:
        for h in hours_local:
            m += gen_res_l[tech][h] <= capacity_l[tech] * cf_local[tech][h]

    for h in hours_local:
        m += charge_l[h] <= capacity_battery_l
        m += discharge_l[h] <= capacity_battery_l
        m += soc_l[h] <= capacity_battery_l * battery_duration_h

    for h in hours_local:
        prev = soc_l[h - 1] if h > 0 else soc_l[hours_local[-1]]
        m += soc_l[h] == prev + charge_l[h] * bat_eff - discharge_l[h] / bat_eff

    for h in hours_local:
        supply = (
            pulp.lpSum(gen_l[tech][h] for tech in commitable_techs_opt)
            + pulp.lpSum(gen_res_l[tech][h] for tech in renewable_techs_opt)
            + discharge_l[h]
            - charge_l[h]
        )
        m += supply == demand_values_local[h]

    fixed_term_l = pulp.lpSum(
        fixed_costs_opt[tech] * capacity_l[tech] for tech in commitable_techs_opt + renewable_techs_opt
    ) + battery_fixed_cost_opt * capacity_battery_l

    variable_term_l = pulp.lpSum(
        var_costs_opt[tech] * pulp.lpSum(gen_l[tech][h] for h in hours_local)
        for tech in commitable_techs_opt
    ) + pulp.lpSum(
        var_costs_opt[tech] * pulp.lpSum(gen_res_l[tech][h] for h in hours_local)
        for tech in renewable_techs_opt
    )

    m += fixed_term_l + variable_term_l

    solver_l = pulp.PULP_CBC_CMD(msg=verbose)
    m.solve(solver_l)

    status = pulp.LpStatus[m.status]
    if status != "Optimal":
        return {"status": status}

    # --- Extract results ---
    capacities = {tech: capacity_l[tech].value() for tech in commitable_techs_opt + renewable_techs_opt}
    capacities["Battery"] = capacity_battery_l.value()

    generation = {}
    for tech in commitable_techs_opt:
        generation[tech] = sum(gen_l[tech][h].value() for h in hours_local)
    for tech in renewable_techs_opt:
        generation[tech] = sum(gen_res_l[tech][h].value() for h in hours_local)
    generation["Battery"] = sum(discharge_l[h].value() * bat_eff for h in hours_local)

    cost_breakdown = []
    total_fixed = 0.0
    total_variable = 0.0
    for tech in commitable_techs_opt + renewable_techs_opt:
        fixed_cost = fixed_costs_opt[tech] * capacities[tech]
        variable_cost = var_costs_opt[tech] * generation[tech]
        total_fixed += fixed_cost
        total_variable += variable_cost
        cost_breakdown.append({
            "Technology": tech,
            "Capacity (MW)": capacities[tech],
            "Generation (GWh)": generation[tech] / 1e3,
            "Fixed cost (M€/yr)": fixed_cost / 1e6,
            "Variable cost (M€/yr)": variable_cost / 1e6,
        })
    battery_fixed_cost = battery_fixed_cost_opt * capacities["Battery"]
    total_fixed += battery_fixed_cost
    cost_breakdown.append({
        "Technology": "Battery",
        "Capacity (MW)": capacities["Battery"],
        "Generation (GWh)": generation["Battery"] / 1e3,
        "Fixed cost (M€/yr)": battery_fixed_cost / 1e6,
        "Variable cost (M€/yr)": 0.0,
    })

    total_cost = total_fixed + total_variable

    total_emissions_local = sum(
        co2_intensity_by_tech[tech] * generation[tech] for tech in commitable_techs_opt
    )

    return {
        "status": status,
        "capacities": capacities,
        "generation": generation,
        "cost_breakdown": pd.DataFrame(cost_breakdown).set_index("Technology"),
        "total_fixed_cost_MEUR": total_fixed / 1e6,
        "total_variable_cost_MEUR": total_variable / 1e6,
        "total_cost_MEUR": total_cost / 1e6,
        "total_emissions_tCO2": total_emissions_local,
    }

In [ ]:
# 2. Run the optimization for all climatic years, save all results
# ============================================================

climatic_years_available = [
    cy for cy in net_demand[target_year].keys()
    if cy in solar_cf_by_year[target_year].columns
    and cy in wind_cf_by_year[target_year].columns
    and cy in wind_offshore_cf_by_year[target_year].columns
]

results_by_cy = {}

for cy in climatic_years_available:
    print(f"Solving optimal mix for {cy} ...")
    result = solve_optimal_mix(cy, verbose=False)
    if result["status"] != "Optimal":
        print(f"  WARNING: solver status = {result['status']} for {cy} — skipped")
        continue
    results_by_cy[cy] = result

print(f"\nDone — {len(results_by_cy)} / {len(climatic_years_available)} climatic years solved optimally.")

In [ ]:
# ============================================================
# 3. Plot 1 — Installed capacities: technology (x) vs capacity (y),
#    one point per (technology, climatic year), colored by climatic year
# ============================================================

all_techs = commitable_techs_opt + renewable_techs_opt + ["Battery"]
palette = pc.qualitative.Plotly
cy_color_map = {cy: palette[i % len(palette)] for i, cy in enumerate(results_by_cy.keys())}

fig = go.Figure()

for cy, result in results_by_cy.items():
    y_vals = [result["capacities"][tech] / 1e3 for tech in all_techs]  # GW
    fig.add_trace(
        go.Scatter(
            x=all_techs,
            y=y_vals,
            mode="markers",
            name=cy,
            marker=dict(color=cy_color_map[cy], size=9)
        )
    )

fig.update_layout(
    title=f"Optimal Installed Capacities by Technology — All Climatic Years ({target_year})",
    xaxis_title="Technology",
    yaxis_title="Installed Capacity (GW)",
    template="plotly_white",
    legend_title="Climatic year",
    width=1100,
    height=600
)
fig.show()

In [ ]:
# ============================================================
# 4. Plot 2 — Bar chart of total system cost per climatic year
# ============================================================

cy_list = list(results_by_cy.keys())
costs_meur = [results_by_cy[cy]["total_cost_MEUR"] for cy in cy_list]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=cy_list,
    y=costs_meur,
    marker_color="royalblue"
))
fig.update_layout(
    title=f"Optimal Total System Cost by Climatic Year ({target_year})",
    xaxis_title="Climatic year",
    yaxis_title="Total system cost (M€/year)",
    template="plotly_white",
    width=900,
    height=500
)
fig.show()

In [ ]:
# ============================================================
# 5. Plot 3 — Bar chart of CO2 emissions per climatic year
# ============================================================

emissions_tco2 = [results_by_cy[cy]["total_emissions_tCO2"] for cy in cy_list]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=cy_list,
    y=emissions_tco2,
    marker_color="darkorange"
))
fig.update_layout(
    title=f"Optimal System CO2 Emissions by Climatic Year ({target_year})",
    xaxis_title="Climatic year",
    yaxis_title="Total CO2 emissions (tCO2/year)",
    template="plotly_white",
    width=900,
    height=500
)
fig.show()

Based on the results, can you chose what you would argue could be a good mix for your cas study country in 2025:

***(Input the capacities below, in GW)***
- Nuclear: 
- Biomass:
- Oil: 
- CCGT:
- OCGT:
- Hard Coal:
- Lignite:
- Solar PV: 
- Wind Onshore: 
- Wind Offshore:

NB: this is the mix ecluding all hydro capacities (and other ones that could exist in real life)

***Write briefly what motivated you decisions here***:

*...*
 

# 7 - Choice a power mix

Based on an economic rationale, political and environmental decisions, you ended up with a power mix that would be optimal for you in your study case country in 2025. Let's compare it to the real data provided by the TSO to ERAA

## 7.1 - Comparison with ERAA power mix

First of all, the let's compare your desired mix with the mix the country actually has (for 2025)

Note: This is note exactly the real mix, as it is 2025 projected from 2022-2023 by the TSOs. Nonetheless, it provides a pretty accurate overview of the power mix.

In [ ]:
# Load the generation capacities data for 2025 from the specified file
generation_capas_path = f"data/ERAA_2023-2/generation_capas/generation-capa_2025_{country_choice}.csv "
generation_capas = pd.read_csv(generation_capas_path,delimiter=";")

generation_capas

Questions:
- How different is your power mix from the real one?
- How can you explain it? (Hints: climatic years, risks, generation technologies available, economic variables overviewed, polictical decisions, time varying parameters...)
- Can you still assess if the base/peak capacity is similar to your?
- What about the flexibility potential (daily/seasonal)?
- Based on these observations, what would be your recommendations? What do you want to change to be aligned with your economic and environmental objectives?

## 7.3 - Capacity Expansion : start from current power mix

Now, we starts the core of this week's objectve: modelling a future power EU power system.

Previously, we tried to size the system starting from zero. But actually, as you witnessed, the power system actors already did some investment decisions, and some capacities are already installed. When sizing a future power mix, you will need to consider these existing technologies, as they are free of CAPEX. The decisionn as to be made on the decommissioning and the investment in new power plants!

In this section, we are going to consider that the decommissioning is free, and the system cost will only be the one from new installed capacity. This will enable to optimize on the potential new capacities and the decommissioning of existing ones. However, the result will not really represent the cost of the power system, but only the cost of new capacities + cost of operation

POINT D'ATTENTION ICI!!!!

Pour partir des données ERAA, il y a des technos que l'on a pas modélisé: hydro STEP, hydro reservoirs, DSR, "other renewables", et "other non renewables".

On essaie de le smodéliser simplement (si tant est que c'est possible?)

Une suggestion si on juge qu'il faut rajouter ça ici:
- Hydro reservoirs: Batterie anuelle, mais dont l'apport est fixé (somme des apports donnés dans le data folder.) Elle se décharge sur les heures le plus tendues de l'année
- Hydro STEP: Comme une batterie, avec un cycle annuel de 1, qui charge aux heures avec le moins de conso de l'année, et qui se déchafrge sur les heures avec le plus de conso de l'année. On considère qu'on ne peut plus invetsir dedans
- DSR: comme des batteries, mais avec un CAPEX de zéro (mais dans lequel on ne peux pas investir, ou très peu), et se cpmporte comme une batterie, mais avec un coût d'activation proche de la littérature (mais cout fictif à ne pas garder à la fin pour le décompte, juste pour l'optim pour prendre en compte la WTA)
- other renewables: considérer un facteur d'émission de 0 et dire que c'est un OCGT pour ce qui est du fonctionnement (ou CCGT?)
- other non renewables: pareil que pour renewbales, mais avec un facteur d'émission à choisir

## 7.4 - Choice of power mix for PyPSA

Based on the current power mix of your country, your economic and environmental objectives, and your overall preferences, what would be the power mix you suggest for 2033? *(including the capapcities not modelled here in this notebook)*

**Capacities**:
- ...
- ...

**Prepare a short presentation** of your country, its demand profile, the power mix you would have advised (starting from zero), the actual current power mix (2025), and the power mix evolution up to 2033. *(Justify your choices)*